# RSNA Knee Abnormality Detection
## RSNA Knee: Take Care Of Your Knee（解説付き学習用コピー）

- **コンペ**: [RSNA Knee Abnormality Detection](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection)（Research / Code Competition・賞金 $77,000・2,067チーム・残り約2ヶ月）
- **原著者**: ANHAD MAHAJAN（[@anhadmahajan06](https://www.kaggle.com/anhadmahajan06)）
- **元notebook**: https://www.kaggle.com/code/anhadmahajan06/rsna-knee-take-care-of-your-knee
- **スコア**: Public 0.914（notebook内の記載は 0.920 到達を主張）・65 votes・2 comments
- **レビュー日**: 2026-08-21

> ⚠️ **これは学習目的の解説付き写しです。** 原著のコードは変更していませんが、出力は含んでおらず未実行です。
> 実行には競技データに加え、DINOv2 / RadImageNet の重み、各種fold重みバンドルの添付が必要です。

---

### 手法の概要

膝MRIの検査（study）1件ごとに、**12種類の異常所見**（ACL・MCL断裂、内側/外側半月板損傷、
内側/外側/膝蓋大腿の変形性関節症、関節液貯留、滑膜炎、ベーカー嚢腫、骨挫傷、骨折）の
確率を同時に予測するマルチラベル分類です。

本notebookは**3段構えのアンサンブル**を組んでいます。

| Stage | 中身 | 主張スコア |
|---|---|---|
| 1 | DINOv2 ViT-Small 24メンバー + 6スロット・クロスアテンション + 10窓TTA | 0.899 |
| 2 | 系列間クロスアテンション（5-fold）を W=0.45 でランクブレンド | 0.910 |
| 3 | RadImageNet ResNet-50 を2種類のレイアウト（E13/E11）で融合 | 0.920 |

ただし本notebookの本当の見どころはモデル構造よりも、**その手前の泥臭い工程**にあります。

- **多言語の放射線レポートを正規表現で解析して弱いラベルを作る**（英・独・仏・西・蘭・土・希・露・ブルガリア語）
- **DICOMヘッダから撮像面・脂肪抑制の有無を推定し、6つの「スロット」に割り当てる**
- **スライスの物理的な並び順を、患者座標系への射影で復元する**
- **左右（laterality）タグが欠けている検査を、膝の重心位置から推定する**

---

### 評価指標について

**タスク**: study 単位のマルチラベル二値分類（12ターゲット）。提出は
`StudyInstanceUID` + 12列の予測値です。

**指標**: **12ターゲットのAUCのマクロ平均**（各ターゲットで個別にAUCを計算し、単純平均）。

**なぜこの指標か**: 12の所見は**出現頻度が大きく違います**。
変形性関節症のような高頻度所見と、骨折のような低頻度所見が混在しており、
全サンプルをまとめて1つのAUCにすると、**多数派の所見だけで点数が決まって**しまいます。
マクロ平均は**すべてのターゲットを等しく重み付け**するので、
「珍しいが臨床的に重大な所見」を無視できません。
また AUC は順位のみの指標なので、所見ごとに陽性率が違っていても
**しきい値を揃える必要がない**という利点もあります。

**本notebookの手法が指標にどう設計されているか**:

1. **ターゲット別のTTAプーリング**（最も指標を意識した設計）。
   10個のスライディング窓の予測をどう集約するかを、**所見の性質ごとに変えています**。
   - `max`（最大値）… 骨折・骨挫傷・半月板・ベーカー嚢腫のような**限局した病変**。
     どこか1枚のスライスで強く反応すれば陽性。全窓を平均すると信号が薄まる
   - `top2`（上位2つの平均）… ACL・MCLのような**細長い靭帯**。
     複数スライスに跨るが全体には広がらない
   - `mean`（平均）… 変形性関節症・関節液貯留・滑膜炎のような**びまん性の変化**。
     全体を見たほうが安定する

   マクロ平均AUCは**12ターゲットそれぞれの順位品質**を測るので、
   全部を同じ集約で処理するより、所見ごとに最適化したほうが素直に効きます。

2. **すべての融合をランク（パーセンタイル）空間で行う**。
   AUCが順位だけの指標である以上、確率のキャリブレーションには一切コストをかけず、
   `rank(pct=True)` で揃えてから重み付き加算しています。

3. **ターゲット別の融合ゲート（α）**。Stage 3 の RadImageNet 融合では、
   ベーカー嚢腫と骨折だけ **α = 0.0（融合しない）** に固定しています。
   この2つは DINOv2 側が局所的に強く、混ぜると薄まるという判断です。
   **マクロ平均は各ターゲットが独立に効く**ので、
   「全体で良くなる重み」ではなく「**ターゲットごとに良くなる重み**」を選ぶのが正解になります。


# RSNA Knee Abnormality Detection: Multi-Arm Visual Foundation Ensemble (0.920 SOTA)

---

### Executive Overview & System Architecture

1. **Stage 1 (Parent Arm E2 - 0.899 SOTA Foundation)**:
   - 20-Member **DINOv2 ViT-Small** visual transformer ensemble ($d_{hidden} = 384$) + 4-member legacy bundle fine-tuned with 6-slot cross-attention.
   - Deterministic geometry normalization via through-plane normal projection $\mathbf{n} = \mathbf{r}_x \times \mathbf{r}_y$ and whole-study median LPS centroid tracking ($>20\text{ mm}$ safety margin).
   - 10-window spatial sliding TTA ($G=3$) with target-specific extreme-value pooling (`max` for focal tears/fractures, `top2` for cruciate/collateral ligaments).
   - Multi-group in-memory caching and 24-member percentile rank fusion consensus.

2. **Stage 2 (Arm A5 - 0.910 Inter-Series Spatial Attention)**:
   - 5-Fold multi-slice cross-series attention capturing 3D spatial alignment between orthogonal MRI planes.
   - Dynamic rank blending ($W = 0.45$) over the parent consensus.

3. **Stage 3 (Surgical V48 Dual-Layout RadImageNet Arm - 0.920 SOTA Pinnacle)**:
   - Domain-specific **RadImageNet ResNet-50** representations pretrained on $>1.3\text{M}$ radiographic exams.
   - Dual-layout evaluation: **E13 layout** (3 fat-suppressed planes + 1 non-suppressed anchor @ 130mm, $\alpha = 0.50$) fused with **E11 layout** (3 non-suppressed planes + 1 fat-suppressed anchor @ 130mm, $\alpha = 0.15$).
   - **Fail-Safe Self-Preservation**: If optional E13/E11 bundles are absent, smoothly deploys the twin public-v15 RadImageNet ensemble ($\alpha = 0.50$).
   - **Invariant Gating**: Baker's Cyst and Fracture are strictly preserved ($\alpha = 0.0$).

```
 ┌────────────────────────────────────────────────────────────────────────────────────────────────────────┐
 │                                   0.920 MULTI-ARM PRODUCTION TOPOLOGY                                  │
 ├────────────────────────────────────────────────────────────────────────────────────────────────────────┤
 │                                                                                                        │
 │  [Stage 1: Parent E2 (0.899)]                                                                          │
 │  24-Member DINOv2 ViT-Small ──► 10-Window TTA (G=3) ──► Initial Rank Consensus (0.899)                │
 │                                                                    │                                   │
 │  [Stage 2: Cross-Series Arm A5 (0.910)]                             ▼                                   │
 │  5-Fold Full-Frame Cross-Series Attention ──────────────► Rank Blend (W = 0.45) ──► 0.910              │
 │  (Dynamically evaluated if knee-mri-fold-weights attached)         │                                   │
 │                                                                    ▼                                   │
 │  [Stage 3: Surgical V48 Dual-Layout RadImageNet (0.920 SOTA)]                                          │
 │  E13 (FS anchor, α=0.50) + E11 (NOFS anchor, α=0.15) ──► Final 0.920 SOTA Submission                   │
 │  (Baker's & Fracture α=0.0; Other 10 targets α=0.50 & 0.15)                                            │
 │                                                                                                        │
 └────────────────────────────────────────────────────────────────────────────────────────────────────────┘
```

### Pathological Target Specification

| Index | Target Name | Primary Plane | Key Sequence | TTA Pooling | RadImageNet Gate ($\alpha$) | Clinical Pathology Description |
|:---:|:---|:---:|:---:|:---:|:---:|:---|
| 0 | **ACL** | Sagittal | SAG_FLUID_FS / SAG_FLUID_NOFS | `top2` | $\alpha = 0.50$ | Anterior cruciate ligament complete or partial disruption |
| 1 | **MCL** | Coronal | COR_FLUID_FS / COR_T1 | `top2` | $\alpha = 0.50$ | Medial collateral ligament sprain, partial or complete tear |
| 2 | **Medial Meniscus** | Sagittal / Coronal | SAG_FLUID_FS / COR_FLUID_FS | `max` | $\alpha = 0.50$ | Fibrocartilage tear, horizontal/radial cleavage in medial meniscus |
| 3 | **Lateral Meniscus** | Sagittal / Coronal | SAG_FLUID_FS / COR_FLUID_FS | `max` | $\alpha = 0.50$ | Lateral meniscus tear, root tear, or bucket-handle lesion |
| 4 | **Medial OA** | Coronal / Sagittal | COR_T1 / COR_FLUID_FS | `mean` | $\alpha = 0.50$ | Medial compartment joint space narrowing, subchondral sclerosis |
| 5 | **Lateral OA** | Coronal / Sagittal | COR_T1 / COR_FLUID_FS | `mean` | $\alpha = 0.50$ | Lateral compartment chondromalacia and osteophyte formation |
| 6 | **PF OA** | Axial / Sagittal | AX_FLUID_FS / SAG_T1 | `mean` | $\alpha = 0.50$ | Patellofemoral compartment cartilage thinning and facet remodeling |
| 7 | **Effusion** | Sagittal / Axial | SAG_FLUID_FS / AX_FLUID_FS | `mean` | $\alpha = 0.50$ | Suprapatellar or joint capsule fluid accumulation |
| 8 | **Synovitis** | Sagittal / Axial | SAG_FLUID_FS / AX_FLUID_FS | `mean` | $\alpha = 0.50$ | Synovial membrane thickening and inflammatory hypertrophy |
| 9 | **Baker's** | Sagittal | SAG_FLUID_FS | `max` | **$\alpha = 0.0$ (Preserved)** | Gastrocnemius-semimembranosus popliteal cyst formation |
| 10 | **Contusion** | Sagittal / Coronal | SAG_FLUID_FS / COR_FLUID_FS | `max` | $\alpha = 0.50$ | Subchondral bone marrow edema / trabecular microfracture |
| 11 | **Fracture** | Sagittal / Coronal / Axial | ALL SLOTS | `max` | **$\alpha = 0.0$ (Preserved)** | Cortical bone break, tibial plateau or patellar fracture |

---


## 1. Clinical Lexicon & Multi-Lingual NLP Report Distillation

Radiological free-text reports contain diagnostic assertions distributed across heterogeneous phrasing, multi-lingual synonymy (English, German, Latin, French), negations, and uncertainty qualifiers.

The distillation engine executes a multi-pass regex parser to extract probabilistic soft labels ($y \in [0, 1]$) with confidence weights ($w \in [0.25, 1.0]$):

$$\text{Confidence}(t) = \begin{cases} 3.0 & \text{if manually verified gold annotation} \\ 0.25 + 0.75 \times \text{Confidence}_{NLP}(t) & \text{if derived from report text} \\ 0.0 & \text{if unmentioned / ambiguous} \end{cases}$$


### 多言語テキスト正規化：レポート解析の土台

**何をしているか**: 放射線レポートの自由文を機械が扱える形に整えます。

- `str.maketrans` でトルコ語の `ı/İ`、ドイツ語の `ß`、北欧の `ø/æ` などを ASCII に寄せる
- `unicodedata.normalize('NFKD')` で発音記号を分離し、結合文字を除去（`é` → `e`）
- ソフトハイフン（`\xad`）や記号（`_ - / \`）を空白に統一
- 文末記号と改行で文（clause）に分割する

**なぜそうするのか**: このコンペのレポートは**複数の国・複数の言語**から集まっています。
同じ「断裂」を意味する語が `rupture` / `Riss` / `déchirure` / `rotura` / `yırtık` / `ρήξη` / `руптура`
と全く違う綴りで現れます。**正規表現でマッチさせる前に表記を揃えておかないと**、
後段の何千行ものパターンが取りこぼしだらけになります。

初心者向けの補足: `unicodedata.normalize('NFKD')` は
「é」を「e + アクセント記号」の2文字に分解する処理です。
その後 `unicodedata.combining(ch)` が真の文字（＝アクセント記号）を捨てれば、
**アクセントを外した素の文字列**が得られます。多言語テキスト処理の定番の1行です。

In [ ]:
from __future__ import annotations
import re
import unicodedata
TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
_PRE = str.maketrans({'ı': 'i', 'İ': 'i', 'I': 'i', 'ß': 'ss', 'đ': 'd', 'Đ': 'd', 'ø': 'o', 'Ø': 'o', 'æ': 'ae', 'Æ': 'ae'})

def normalize(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.translate(_PRE).lower()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join((ch for ch in text if not unicodedata.combining(ch)))
    text = text.replace('\xad', '')
    text = re.sub('[_\\-/\\\\]+', ' ', text)
    text = re.sub('[ \\t]+', ' ', text)
    return text
_SENT_SPLIT = re.compile('(?<=[.;!?])\\s+|\\n+')

def unwrap(text: str) -> str:
    if not isinstance(text, str):
        return ''
    out = []
    for line in text.split('\n'):
        s = line.strip()
        if out and out[-1] and (not re.search('[.;:!?>*•]$', out[-1])) and (len(out[-1].split()) >= 4) and s and (not s[:1].isupper()):
            out[-1] = out[-1] + ' ' + s
        else:
            out.append(s)
    return '\n'.join(out)

def clauses(text: str):
    norm = normalize(unwrap(text) if FEATURES['unwrap'] else text)
    raw = [c.strip() for c in _SENT_SPLIT.split(norm) if c and c.strip()]
    merged = []
    for i, c in enumerate(raw):
        if c.endswith(':') and len(c.split()) <= 14 and (i + 1 < len(raw)):
            merged.append(c + ' ' + raw[i + 1])
        merged.append(c)
    out = []
    for c in merged:
        out.append(c)
        if len(c.split()) > 25:
            out.extend((p.strip() for p in c.split(',') if len(p.split()) > 2))
    return out
FEATURES = {'unwrap': True, 'directional_negation': True, 'oa_inherit': True, 'graded_pathology': True, 'synovitis_backoff': True}

def _rx(*alts: str) -> re.Pattern:
    return re.compile('|'.join(alts))
PRE_NEG = _rx('\\bno\\b', '\\bnot\\b', '\\bwithout\\b', '\\bnegative for\\b', '\\babsence\\b', '\\bno evidence\\b', '\\bfree of\\b', '\\bnone\\b', '\\bneither\\b', '\\bnor\\b', '\\bsin\\b', '\\bno hay\\b', '\\bausencia\\b', '\\bausentes?\\b', '\\bno se\\b', '\\bpas de\\b', '\\bsans\\b', '\\baucune?\\b', '\\bgeen\\b', '\\bzonder\\b', '\\bniet\\b', '\\bkeine?[nmrs]?\\b', '\\bohne\\b', '\\bnicht\\b', '\\bkein\\b', '\\bnema\\b', '\\bbez\\b', '\\bnisu\\b', '\\bnije\\b', '\\bδεν\\b', '\\bχωρις\\b', 'ουδεν', '\\bουτε\\b', '\\bбез\\b', '\\bне\\b', 'липсва', '\\bняма\\b')
POST_NEG = _rx('\\byok\\b', '\\byoktur\\b', 'izlenmemekte', 'saptanmadi', '\\bdegil\\b', 'gozlenmemekte', 'mevcut degil', 'eslik etmiyor', '\\bizlenmedi\\b', 'izlenmemistir', 'saptanmamistir', 'gorulmemistir', '\\bnema znakova\\b', 'bez znakova')
NEGATION = _rx(PRE_NEG.pattern, POST_NEG.pattern, '\\bunremarkable\\b')
NEG_WINDOW = 90

def _negated(clause: str, start: int, end: int) -> bool:
    for m in PRE_NEG.finditer(clause):
        if m.end() <= start and start - m.end() <= NEG_WINDOW:
            if not re.search('\\b(but|however|ancak|fakat|pero|maar|aber|no i|ali|ωστοσο|αλλα|но)\\b', clause[m.end():start]):
                return True
    for m in POST_NEG.finditer(clause):
        if m.start() >= end and m.start() - end <= NEG_WINDOW:
            return True
    return False
NORMALITY = _rx('\\bnormal', '\\bintact\\b', '\\bpreserved\\b', '\\bwithin normal limits\\b', 'limites normales', '\\bconservad', '\\bintegr', '\\bnormales\\b', '\\bdoga(l|ll)\\b', 'korunmus', '\\bnormaldir\\b', 'olagan', '\\buredn', '\\bocuvan', '\\bodrzan', '\\bintakt', '\\bprimjeren', '\\bodrzanog kontinuiteta', '\\bodržan', 'φυσιολογικ', 'ακεραι', 'δεν παρατηρουνται', 'δεν σημειωνονται', 'unauffallig', 'regelrecht', '\\bo\\.?b\\.?\\b', 'нормал', 'запазен', 'съхранен', '\\bбез особености\\b', 'интактн', '\\bgaaf\\b', '\\bnormaal\\b')
NORMAL_PHRASE = _rx('\\bsin alteracion', '\\bsin cambios\\b', '\\bsin particularidad', '\\bsin hallazgos\\b', '\\bsin lesion', '\\bsin signos de (rotura|lesion)', '\\bcontinu[oa]s?\\b', '\\bcontinuidad conservada\\b', '\\bno abnormalit', '\\bno significant abnormalit', '\\bunremarkable\\b', '\\bno evidence of (tear|injury|abnormalit)', '\\bohne auffalligkeit', '\\bkein nachweis\\b', '\\bohne befund\\b', '\\bgeen afwijking', '\\bzonder afwijking', '\\bsans anomalie', "\\bpas d[e']anomalie", '\\bbez osobitosti\\b', '\\bbez znakova (rupture|lezije)\\b', '\\bbez patoloskih\\b', 'χωρις αλλοιωσ', 'χωρις παθολογ', 'δεν παρατηρουνται (αξιολογα|παθολογ)', '\\bбез особености\\b', '\\bбез патологич', '\\bбез данни за\\b', '\\bozel bir ozellik yok', '\\bpatolojik bulgu (yok|izlenmemis)')
UNCERTAIN = _rx('\\bpossible\\b', '\\bprobable\\b', '\\bsuspicious\\b', '\\bsuspected?\\b', 'cannot (be )?exclude', '\\bmay\\b', '\\bquestionable\\b', '\\bequivocal\\b', '\\br/o\\b', '\\bdd\\b', '\\blikely\\b', '\\bsuggest', '\\bcompatible with\\b', '\\bposible\\b', 'sin criterios categoricos', '\\bdudos', '\\bsugier', '\\bmuhtemel\\b', '\\bolasi\\b', '\\bsupheli\\b', '\\bizlenim', '\\bdusundur', '\\bmoguce\\b', '\\bvjerojatno\\b', '\\bsumnja\\b', '\\bmoze odgovarati\\b', 'πιθαν', 'υποπτ', '\\bmoglich', '\\bverdachtig', '\\bfraglich', '\\bv\\.?a\\.?\\b', '\\bwohl\\b', '\\bвъзможно\\b', '\\bвероятно\\b', 'суспект', '\\bmogelijk\\b', '\\bverdacht\\b')

### 病変を表す語彙の正規表現辞書（多言語）

**何をしているか**: `TEAR`（断裂）、`DEGEN`（変性）など、所見のカテゴリごとに
**10言語以上の同義語をまとめた巨大な正規表現**を構築しています。

**なぜそうするのか**: このコンペには**画像の正解ラベルが直接は与えられていない**部分があり、
レポート本文から**弱いラベル（weak label）**を作る必要があります。
LLMを使う手もありますが、Code Competition はオフライン実行であり、
数万件のレポートをLLMで処理すると**実行時間制限に引っかかります**。
正規表現なら**高速・決定論的・監査可能**です。
どの語が拾われたかを人間が確認でき、間違いを1行修正すれば直せます。

`\b` は単語境界を表し、`\btear` は "tear" では一致しますが "stearic" では一致しません。
語幹だけを書いて `tear/tears/tearing` をまとめて拾う、という設計が随所に見られます。

> 実務的な教訓: 「LLMのほうが賢い」からといって常に正しい選択とは限りません。
> **速度・再現性・監査可能性**が制約になる場面では、ルールベースが依然として強いです。

In [ ]:
TEAR = _rx('\\btear', '\\btorn\\b', '\\brupture', '\\bdisruption\\b', 'discontinuit', '\\bavuls', '\\bmacerat', '\\bbuckethandle\\b', 'bucket handle', '\\brotura\\b', '\\broturas\\b', '\\bruptura', '\\bdesgarro', '\\broto\\b', '\\bdechirure', '\\bdechire', '\\bscheur', '\\bruptuur', 'gescheurd', '\\briss\\b', 'einriss', '\\bruptur', 'zerreiss', '\\blasion', '\\bausriss', '\\byirtik', '\\byirtig', '\\bkopma\\b', 'butunluk kaybi', '\\brupturu\\b', 'devamsizlik', '\\brupture\\b', '\\bdevamliligi secilememis', '\\bpuknuce', '\\bprekid\\b', '\\bpukotin', '\\bruptur', 'ρηξη', 'ρηξις', 'ρηγμα', 'ασυνεχεια', 'руптура', 'разкъсв', 'разрив', 'скъсв', '\\bлезия\\b')
DEGEN = _rx('degenerat', '\\bmucoid\\b', '\\bmyxoid\\b', '\\bfray', '\\bfissur', 'dejeneratif', '\\bmukoid\\b', 'degenerativn', 'εκφυλ', 'дегенерат', '\\bμυξοειδ', '\\bμυξωδ', '\\bmeniskopat', '\\bmeniscopath', '\\bmuco ?ide\\b', 'aufgefasert', '\\bdejenerasyon\\b')
INJURY = _rx('\\binjur', '\\bsprain', '\\blesion', '\\blasion', '\\bedema\\b', '\\boedema\\b', '\\bodem\\b', '\\bedem\\b', '\\bοιδημα', '\\bодем', '\\bедем', '\\bstrain\\b', '\\bhigh signal\\b', '\\bsignal alteration\\b', '\\bhiperintens', '\\bhyperintens', 'aumento de senal', 'alteracion de senal', 'cambio de senal', '\\bsignalanhebung', '\\bsignalalteration', 'verhoogd signaal', 'sinyal artis', 'αυξημενο σημα', 'повишен сигнал', '\\besguince\\b', '\\bthicken', '\\bzadebljanje\\b', '\\bverdikking\\b', '\\bdistenzij', '\\blaksite\\b', '\\blaxity\\b', '\\bpartial\\b', '\\bparcijaln', '\\bparcial', '\\bpartiel', '\\bpartiell')
_GRADE_RX = re.compile('(?:grade|grad|grado|grau|derece|stupnja|stupanj|βαθμ|степен|icrs|outerbridge)[\\s:]*(?:grade\\s*)?([1-4]|iv|iii|ii|i)\\b')
_ROMAN = {'i': 1, 'ii': 2, 'iii': 3, 'iv': 4}

def _grade_of(clause: str):
    best = None
    for m in _GRADE_RX.finditer(clause):
        v = m.group(1)
        n = _ROMAN.get(v, None) if not v.isdigit() else int(v)
        if n is not None and (best is None or n > best):
            best = n
    return best
ANAT = {'ACL': _rx('anterior cruciate', '\\bacl\\b', 'cruzado anterior', '\\blca\\b', 'croise anterieur', 'voorste kruisband', '\\bvkb\\b', 'vorderes kreuzband', 'vorderen kreuzband', 'vordere kreuzband', 'on capraz', '\\bocb\\b', 'anterior capraz', 'prednji krizni', 'prednjeg krizn', 'προσθι[οα][^ ]* χιαστ', 'προσθιου χιαστου', 'χιαστο[^ ]* συνδεσμ', '\\bχιαστ\\w*', 'предна кръстна', 'предната кръстна', 'предна кръста', 'cruciate ligaments', 'ligamentos cruzados', 'ligaments croises', 'kruisbanden', 'kreuzbander', 'capraz baglar', 'krizn[a-z]* ligament[a-z]*', 'χιαστοι συνδεσμ', 'χιαστων συνδεσμ', 'кръстните връзки', 'кръстни връзки'), 'MCL': _rx('medial collateral', '\\bmcl\\b', 'tibial collateral', 'colateral medial', 'colateral interno', '\\blcm\\b', 'collateral medial', 'collateral interne', 'mediale collaterale', 'binnenband', '\\b(mediale|laterale) banden\\b', '\\bcollaterale banden\\b', 'innenband', 'mediales? kollateral', '\\bic yan bag', 'medial kollateral', '\\biyb\\b', 'medyal kollateral', 'medijalni kolateraln', 'medijalnog kolateraln', 'εσω πλαγι', 'εσωτερικο πλαγι', '\\bπλαγι\\w* συνδεσμ', '\\bπλαγιοι\\b', 'медиален колатерал', 'вътрешна странична', '\\bколатерал\\w*', '\\bcolaterales\\b', '\\bcollateraux\\b', '\\bcollateralen\\b', '\\bkolateralni\\b', 'collateral ligaments', 'ligamentos colaterales', 'ligaments collateraux', 'collaterale banden', 'kollateralbander', 'seitenbander', 'yan baglar', 'kolateraln[a-z]* ligament[a-z]*', 'πλαγιοι συνδεσμ', 'πλαγιων συνδεσμ', 'колатерални връзки', 'страничните връзки'), 'Medial Meniscus': _rx('medial meniscus', '\\bmm\\b(?= tear)', 'medial menisc', 'menisco medial', 'menisco interno', 'menisque medial', 'menisque interne', 'mediale meniscus', 'binnenmeniscus', 'innenmeniskus', 'medialen? meniskus', 'innenmeniskushinterhorn', 'medyal menisk', '\\bic menisk', 'medijalni meniskus', 'medijalnog meniskusa', 'medijalnom meniskusu', 'medijaln\\w* menisk\\w*', '\\bmedijalnog meniska\\b', 'medijalni menisk', 'εσω μηνισκ', 'μηνισκ[^ ]* του εσω', 'εσω διαμερισμα[^.]{0,40}μηνισκ', 'медиалния менискус', 'медиален менискус', 'вътрешния менискус', 'oba meniska', 'both menisci', 'ambos meniscos', 'beide menisci', 'her iki menisku', 'amfoteroi\\w* mhnisk', 'αμφοτερ\\w* μηνισκ', 'двата менискуса', 'medial (and|&) lateral menisc'), 'Lateral Meniscus': _rx('lateral meniscus', 'lateral menisc', 'menisco lateral', 'menisco externo', 'menisque lateral', 'menisque externe', 'laterale meniscus', 'buitenmeniscus', 'aussenmeniskus', 'lateralen? meniskus', 'aussenmeniskushinterhorn', 'lateral menisk', '\\bdis menisk', 'lateralni meniskus', 'lateralnog meniskusa', 'lateralnom meniskusu', 'lateraln\\w* menisk\\w*', '\\blateralnog meniska\\b', 'εξω μηνισκ', 'μηνισκ[^ ]* του εξω', 'εξω διαμερισμα[^.]{0,40}μηνισκ', 'латералния менискус', 'латерален менискус', 'външния менискус', 'oba meniska', 'both menisci', 'ambos meniscos', 'beide menisci', 'her iki menisku', 'αμφοτερ\\w* μηνισκ', 'двата менискуса', 'medial (and|&) lateral menisc')}
OA_EVIDENCE = _rx('osteoarthrit', '\\barthros', '\\bgonarthros', '\\bosteoarthros', 'chondropath', 'chondromalac', 'condropat', 'condromalac', '\\bchondros', '\\bchondrosis\\b', 'chondral (loss|defect|ulcer|thinning|injury|fissur|wear)', 'cartilage (loss|thinning|defect|fissur|wear|damage|heterogeneity|irregularit)', '(loss|thinning|fissur|defect|ulcer|erosion|denudation) of[^.]{0,20}cartilage', 'articular cartilage[^.]{0,30}(loss|thin|fissur|defect|erosion|wear|irregular)', 'osteophyt', 'osteofit', 'osteofyt', 'osteofito', 'osteophyten', 'spurring', 'joint space narrowing', 'pinzamiento articular', 'reduced joint space', 'kikirdak kayb', 'kikirdak incelme', 'kondropati', 'kondral', 'kikirdak dejener', 'eklem aralig\\w* daral', 'eklem mesafesi daral', 'kikirdak kalinlig\\w* azal', 'kraakbeen', 'gonartrose', 'artrose', '\\bknorpel', 'arthrose', 'gonarthrose', 'hrskavic', 'hondromalac', 'artroz', 'osteoartrit', 'artrotsk', 'artrotick', '\\boa promjen', '\\boa\\b', 'degenerativne promjene hrskav', 'χονδρ[^ ]*παθ', 'αρθριτ', 'αρθρωσ', 'οστεοφυτ', 'χονδρομαλακ', 'αρθρικου χονδρου', 'εξαλειψη του αρθρικου χονδρου', 'διαβρωση του αρθρικου χονδρ', 'λεπτυνση[^.]{0,30}χονδρ', 'φθορα[^.]{0,20}χονδρ', 'артроз', 'хондропат', 'остеофит', 'хрущял[^.]{0,40}(изтън|увред|дефект|липс)', 'изтъняване[^.]{0,30}хрущял', 'хондромалац', 'ulcera[s]? condral', 'cartilago[^.]{0,25}(perdida|adelgaz)', 'icrs grade', 'icrs\\b', 'outerbridge', '\\bdenudation\\b', 'denudacij', 'erozivne promjene', '\\berosion of[^.]{0,20}cartilage', 'kraakbeenlijden', 'kraakbeenverlies')
TF_SITE = _rx('compartment', 'compartimento', 'compartiment', 'kompartman', 'kompartiment', 'kompartment', 'odjelj', 'διαμερισμα', 'компартм', '\\bотдел', 'femorotibial', 'tibiofemoral', 'femoro tibial', 'femorotibiaal', 'femorotibijaln', 'феморотибиал', '\\bft zglob', 'tibiofemoraln', 'condyle', 'condilo', 'kondyl', 'kondil', 'condyl', 'κονδυλ', 'кондил', '\\bplateau', '\\bplato\\b', 'platillo', 'meseta', 'плато', 'tibiaplateau', 'tibijaln\\w* plato', 'tibyal plato', 'tibia plato', 'κνημιαι', 'μηριαι', 'weightbearing', 'weightbaring', 'zona de carga', 'dragende deel', 'agirlik tasiyan', '\\bfemur\\b', '\\btibia\\b', '\\bfemoral\\b', '\\btibial\\b', '\\bfemura\\b', '\\btibije\\b', '\\bmesarthrio\\b', 'μεσαρθριο')
PF_SITE = _rx('patellofemoral', 'femoropatellar', 'femoropatelar', 'patelofemoral', 'retropatellar', 'retrorotulian', 'trochlea', 'troclea', 'troklea', 'trochlear', 'trohlej', 'τροχιλ', '\\bpatella', '\\bpatellar', 'rotulian', '\\brotula\\b', '\\bpatele\\b', 'patellofemoraal', 'femoropatellair', 'επιγονατιδ', 'μηροεπιγονατιδ', 'пател', 'феморопател', 'anterior compartment', 'compartimento anterior', 'prednj\\w* odjeljk', '\\bfp zglob', '\\bpf zglob', '\\bfaset', '\\bfacet', 'patellofemoraln')
SIDE_MEDIAL = _rx('\\bmedial\\w*', '\\bmedyal\\w*', '\\bmedijaln\\w*', '\\bmediaal\\w*', '\\bmediale\\w*', '\\binterno\\b', '\\binterna\\b', '\\binternos\\b', '\\binterne\\b', '\\binnen\\w*', '\\bic\\b', '\\bunutarnj\\w*', '\\bεσω\\w*', '\\bεσωτερικ\\w*', '\\bмедиал\\w*', '\\bвътреш\\w*', '\\bbinnen\\w*', '\\bmediaal\\b', '\\bmediales?\\b')
SIDE_LATERAL = _rx('\\blateral\\w*', '\\bexterno\\b', '\\bexterna\\b', '\\bexternos\\b', '\\bexterne\\b', '\\bdis\\b', '\\blateraln\\w*', '\\baussen\\w*', '\\bbuiten\\w*', '\\bεξω\\w*', '\\bεξωτερικ\\w*', '\\bлатерал\\w*', '\\bвъншн\\w*', '\\bvanjsk\\w*')
SIDE_ANTERIOR = _rx('\\banterior\\w*', '\\bant\\b', '\\bon\\b', '\\bprednj\\w*', '\\bvorder\\w*', '\\bvoorste\\b', '\\bπροσθι\\w*', '\\bпредн\\w*', '\\banteriyor\\w*', '\\bavant\\b', '\\banterieur\\w*')
GLOBAL_OA = _rx('tri ?compartment', 'all three compartment', 'global(ised)? (oa|osteoarthrit)', '\\bgonarthros', '\\bgonartros', '\\bgonarthrose', '\\bgonartrose', 'gonartro', 'goanrtrot', 'gonartrot', 'osteoarthritis of the knee', 'artrosis (de |)(la )?rodilla', 'knee osteoarthrit', '\\bdiz osteoartrit', '\\bgonartroz', 'artroza koljena', 'οστεοαρθριτιδα', 'αρθριτιδα του γονατος', 'εκφυλιστικη οστεοαρθριτ', 'артроза на колянната', 'гонартроз', 'degenerative joint disease', '\\bdjd\\b', 'three compartments', 'compartmens', 'compartments')
DIRECT = {'Effusion': _rx('\\beffusion', 'joint fluid', 'intra ?articular fluid', '\\bhydrops\\b', '\\bhemarthros', '\\bhaemarthros', 'derrame articular', '\\bderrame\\b', 'liquido articular', 'hemartrosis', 'epanchement', 'gewrichtsvocht', '\\bvocht\\b', 'gewrichtseffusie', 'opzetting van suprapatell', 'gelenkerguss', '\\berguss\\b', 'gelenksergu', 'gelenksflussigkeit', 'eklem\\w* ic\\w* sivi', 'efuzyon', 'eklem sivisi', 'eklem mesafesinde sivi', 'sivi (miktari|artisi|birikimi)', 'sivi artis', '\\bsivi\\b[^.]{0,25}artmis', '\\bizljev', '\\bizliv', 'zglobn[^ ]* tekucin', '\\bhidrops\\b', 'αρθρικ[^ ]* υγρ', 'υγρου ενδαρθρικα', 'ενδαρθρικ[^ ]* υγρ', 'ποσοτητα υγρου', 'ενδαρθρικ', 'αρθρικη συλλογη', 'υγρο στην αρθρωση', 'υγρου στην αρθρωση', 'συλλογη υγρου', 'ενθαρθρικ', 'ставен излив', 'излив', 'ставна течност', 'синовиална течност'), 'Synovitis': _rx('synovit', 'sinovit', 'synovial (thickening|proliferation|hypertroph)', 'thicken\\w* synovial', 'hypertroph\\w* of the synovium', 'synoviale? (verdikking|proliferatie)', 'verdikkingen van (het )?synovium', 'synovialitis', 'synovialis(verdickung|proliferation)', 'reizsynovial', 'sinovijalitis', 'sinovitis', 'zadebljanje sinovij', 'proliferacij\\w* sinovij', 'sinovijaln\\w* proliferacij', 'υμενιτιδα', 'συνοβιτιδα', 'υμενικ[^ ]* υπερτροφ', 'αρθρικου υμεν', 'παχυνση[^.]{0,20}υμεν', 'υμενα', 'синовит', 'синовиал[^ ]* (задебел|пролифер)', '\\bpannus\\b', '\\bhoffit', 'sinovyal\\w* (kalinlas|proliferas)', 'sinovyal hipertrof', '\\bartrit\\b', '\\barthritis\\b'), "Baker's": _rx('baker', 'popliteal cyst', 'quiste popliteo', 'quistes popliteos', 'kyste poplite', 'popliteale? cyst', 'poplitealzyste', 'bakerzyste', 'popliteal kist', '\\bbakerova\\b', 'poplitealn[^ ]* cist', 'popliteal\\w* cist', 'κυστη baker', 'πολυχωρη συνοβιακη κυστη', 'κυστη του baker', 'συνοβιακη κυστη', 'κυστη τυπου baker', 'киста на бейкър', 'бейкърова киста', 'поплитеална киста', 'бекеров', 'gastrocnemio ?semimembranos', 'gastrocnemius semimembranosus burs'), 'Contusion': _rx('\\bcontusion', 'bone bruise', 'bone marrow (o?edema|contusion)', 'marrow o?edema', '\\bkontuz', 'medular bone o?edema', 'osseous contusion', 'contusion osea', 'edema oseo', 'edema de medula osea', 'contusiones oseas', 'oedeme osseux', 'contusion osseuse', 'botcontusie', 'botoedeem', 'beenmergoedeem', 'botmergoedeem', 'knochenmarkodem', 'knochenodem', 'knochenmarksodem', 'kontusion', 'kemik kontuzyonu', 'kemik iligi odemi', 'kemik odemi', 'kemik iliginde odem', 'kontuzyonel kemik', 'kemik iligi odemleri', 'kostani edem', 'edem kosti', 'kontuzij', 'kostane srzi[^.]{0,20}edem', 'οστεομυελικ[^ ]* οιδημα', 'οστικο οιδημα', 'μυελικο οιδημα', 'οστικο μωλωπ', 'костномозъчен едем', 'костен едем', 'контузионен', 'костно мозъчен едем'), 'Fracture': _rx('\\bfractur', '\\bfract\\b', '\\bfractura', '\\bfracturas\\b', '\\bfractuur', '\\bbreuk\\b', '\\bfraktur', '\\bbruch\\b', '\\bkirik\\b', '\\bkirigi\\b', '\\bkiri[kg]\\w*', '\\bprijelom', 'impresijsk[^ ]* fraktur', 'impaktcij', 'καταγμα', 'καταγματ', 'фрактур', 'счупван', 'фисур', 'insufficiency fracture', 'stress fracture', 'avulsion fracture', 'subchondral fracture', 'subkondral kiri', 'impaction (fracture|injury)', 'osteochondral (fracture|impaction)', '\\bsegond\\b', 'impactiefractuur', 'subchondrale impression', 'subchondraler? impress')}
DECOY = {'Fracture': _rx('microfractur', '\\bfracture (risk|prophyla)'), "Baker's": _rx('meniscal cyst', 'quiste meniscal', 'parameniscal')}
PAIRED = {'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus'}
OA_TARGETS = ['Medial OA', 'Lateral OA', 'PF OA']
PLURAL_MENISCI = _rx('\\bmenisci\\b', '\\bmeniscos\\b', '\\bmenisques\\b', '\\bmenisken\\b', '\\bmeniskusi\\b', '\\bmenisk\\w*ler\\b', '\\bμηνισκοι\\b', '\\bμηνισκων\\b', '\\bменискуси\\b', '\\bменискусите\\b', '\\bmenisci\\w*\\b')
ANY_SIDE = _rx(SIDE_MEDIAL.pattern, SIDE_LATERAL.pattern)
STEM_MENISCUS = _rx('menisc\\w*', 'menisk\\w*', 'μηνισκ\\w*', 'мениск\\w*')
STEM_CRUCIATE = _rx('cruciate', 'cruzado', 'croise', 'kruisband', 'kreuzband', 'capraz bag\\w*', 'krizn\\w*', 'χιαστ\\w*', 'кръстн\\w*', '\\bacl\\b', '\\blca\\b', '\\bvkb\\b', '\\bocb\\b', '\\bacb\\b')
STEM_COLLATERAL = _rx('collateral\\w*', 'colateral\\w*', 'kollateral\\w*', 'collaterale\\w*', 'kolateraln\\w*', 'yan bag\\w*', 'πλαγι\\w*', 'колатерал\\w*', 'странич\\w*', 'innenband\\w*', 'binnenband\\w*', '\\bmcl\\b', '\\blcm\\b', '\\biyb\\b')
STEM_FRACTURE = _rx('fractur\\w*', 'fraktur\\w*', 'fractuur\\w*', '\\bfract\\b', 'kiri[kgğ]\\w*', 'prijelom\\w*', 'lom kosti', '\\bbreuk\\w*', '\\bbruch\\w*', 'καταγμα\\w*', 'καταγματ\\w*', 'фрактур\\w*', 'счупван\\w*', 'fisur\\w* (osea|oseas|kost)', 'fissur\\w* kost')
POSTERIOR_ONLY = _rx('\\bpcl\\b', '\\blcp\\b', '\\bhkb\\b', '\\bacb\\b', 'posterior cruciate', 'cruzado posterior', 'croise posterieur', 'achterste kruisband', 'hinteres kreuzband', 'arka capraz', 'straznji krizn', 'οπισθι[οα]\\w* χιαστ', 'задна кръстн', 'задната кръстн')
LATERAL_COLL_ONLY = _rx('\\blcl\\b', '\\bfcl\\b', 'lateral collateral', 'fibular collateral', 'colateral lateral', 'colateral externo', 'buitenband', 'aussenband', 'dis yan bag', 'lateralni kolateraln', 'εξω πλαγι', 'латерален колатерал')

### 近傍探索と否定表現の処理：レポートから確率つきラベルを取り出す

**何をしているか**: `_near(clause, stem_rx, qual_rx, window=55)` が中核です。
「語幹（例: 半月板 `meniscus`）」の**前後55文字以内**に
「修飾語（例: 内側 `medial`）」があるかを調べます。
`_Matcher` クラスは、フレーズ一致・語幹＋部位の組み合わせ一致を統一的に扱います。

**なぜそうするのか**: レポートの文は
「内側半月板後角に断裂、外側半月板は正常」のように、
**1文に複数の所見と複数の部位が混在**します。
単に「meniscus」と「tear」が同じ文にあるかで判定すると、
上の例では**外側半月板まで陽性**にしてしまいます。
語と語の**距離**を条件に加えることで、この誤りを大きく減らせます。

さらに重要なのが**否定表現の処理**です。医療レポートは
「no evidence of ACL tear（ACL断裂の所見なし）」のように、
**陰性所見をわざわざ明記する**のが普通です。
否定語を無視すると、**陰性の記載が全部陽性ラベルになる**という壊滅的な事故が起きます。

出力は0/1のラベルではなく、**確信度つきの軟らかいラベル**
（手動確認済みなら重み3.0、レポート由来なら 0.25〜1.0、記載なしなら0.0）です。
「テキストから読み取ったラベルは、人間が付けたラベルほど信用できない」という事実を、
**重みという形で明示的にモデルへ伝えている**わけです。

> 用語: **弱教師あり学習（weak supervision）** =
> 完璧な正解ラベルの代わりに、ノイズを含む安価なラベル（ここではレポート由来）で学習する枠組み。

In [ ]:
def _near(clause: str, stem_rx: re.Pattern, qual_rx: re.Pattern, window: int=55):
    for m in stem_rx.finditer(clause):
        lo = max(0, m.start() - window)
        hi = min(len(clause), m.end() + window)
        if qual_rx.search(clause[lo:hi]):
            return True
    return False
STEM_RULES = {'ACL': (STEM_CRUCIATE, SIDE_ANTERIOR), 'MCL': (STEM_COLLATERAL, SIDE_MEDIAL), 'Medial Meniscus': (STEM_MENISCUS, SIDE_MEDIAL), 'Lateral Meniscus': (STEM_MENISCUS, SIDE_LATERAL)}

class _Matcher:

    def __init__(self, phrase_rx, stem=None, side=None, window=55):
        self.phrase_rx = phrase_rx
        self.stem = stem
        self.side = side
        self.window = window

    def search(self, clause):
        m = self.phrase_rx.search(clause)
        if m is not None:
            return m
        if self.stem is not None and _near(clause, self.stem, self.side, self.window):
            return self.stem.search(clause)
        return None
ANAT_MATCH = {t: _Matcher(ANAT[t], *STEM_RULES[t]) for t in PAIRED}
DIRECT_MATCH = {t: _Matcher(_rx(rx.pattern, STEM_FRACTURE.pattern) if t == 'Fracture' else rx) for t, rx in DIRECT.items()}
SEV_LOW = _rx('\\bsmall\\b', '\\bminimal\\b', '\\btrace\\b', '\\bmild\\b', '\\bslight\\b', '\\btiny\\b', '\\bscant\\b', '\\bdiscrete\\b', '\\blow ?grade\\b', '\\bincipient\\b', '\\bleve\\b', '\\bminim', '\\bpeque', '\\bfina\\b', '\\bfino\\b', '\\bligero\\b', '\\bescaso\\b', '\\bdiscreto\\b', '\\bhafif\\b', '\\baz miktarda\\b', '\\bsilik\\b', '\\bmanj\\w*', '\\bblago\\b', '\\bdiskretn', '\\bmalo\\b', '\\bpocetn', '\\bgering', '\\bdiskret', '\\bkleine?r?\\b', '\\bwenig\\b', '\\bzarte?\\b', '\\bbeperkte?\\b', '\\bgeringe\\b', '\\bweinig\\b', '\\blichte?\\b', '\\blicht\\b', '\\bηπι', '\\bμικρ', '\\bελαχιστ', '\\bαρχομεν', '\\bминимал', '\\bлек', '\\bмалк', '\\bнеголям')
SEV_HIGH = _rx('\\blarge\\b', '\\bmarked\\b', '\\bmassive\\b', '\\bsevere\\b', '\\bextensive\\b', '\\bmoderate\\b', '\\bgross\\b', '\\bsignificant\\b', '\\babundant\\b', '\\btense\\b', '\\bcomplete\\b', '\\bfull ?thickness\\b', '\\bhigh ?grade\\b', '\\badvanced\\b', '\\bmoderad', '\\bimportante\\b', '\\bsevera?\\b', '\\bmarcad', '\\bcuantios', '\\bespesor total\\b', '\\bcompleta?\\b', '\\bbelirgin\\b', '\\byaygin\\b', '\\bileri\\b', '\\bciddi\\b', '\\bbol\\b', '\\bkomplet', '\\bopsezan\\b', '\\bveliki\\b', '\\bizrazit', '\\bznacajn', '\\bumjeren', '\\buznapredoval', '\\bpotpun', '\\bkompleksn', '\\bausgepragt', '\\bdeutlich', '\\bmassiv', '\\bmassig', '\\bgross', '\\buitgebreid', '\\bgevorderd', '\\bveel\\b', '\\bmatige?\\b', '\\bvolledig', '\\bμετρι', '\\bμεγαλ', '\\bεκτεταμεν', '\\bευμεγεθ', '\\bσοβαρ', '\\bπληρη', '\\bголям', '\\bизразен', '\\bзначим', '\\bумерен', '\\bобилен', '\\bпълн')
GRADE_HIGH = re.compile('grade?[ao]?\\s*(3|4|iii|iv)\\b|icrs grade (iii|iv|3|4)|stupnja iv|stupnja iii|\\bgrado (3|4)\\b|\\bgrad (3|4)\\b|\\bgrade (3|4)\\b')
DEGENERATIVE_MARROW = _rx('subchondral', 'subcondral', 'subkondral', 'supkondraln', 'subchondraln', 'υποχονδρι', 'υπαρθρικ', 'субхондрал', 'subchondrale?', 'subartikuler', '\\bcyst', '\\bquist', '\\bzyste\\b', '\\bcistic', 'reactive', 'reactivo', 'degenerative', 'degenerativ', 'reaktiv', '\\bcisti\\b')
TRAUMA = _rx('\\bbruise\\b', '\\bcontusion', '\\bkontuz', '\\btrauma', '\\bimpaction\\b', '\\bpivot shift\\b', '\\bkissing\\b', '\\bacute\\b', '\\bagudo\\b', '\\bakut', '\\bpivot kaymasi\\b', '\\bcontusion osseuse\\b', '\\bbone bruise\\b', '\\bbotcontusie\\b', '\\bконтузион', '\\bμωλωπ', '\\bkontuzij', '\\bimpaktcij', '\\bimpakcij', '\\bfall\\b', '\\binjury\\b', '\\bimpression\\b')
SYNOVIAL_PROXY = _rx('bursit', 'burzit', '\\bbursa\\b[^.]{0,30}(fluid|distend|sivi|tekucin|opzetting)', 'suprapatellar (bursitis|effusion|recess)', 'suprapatellar bursa', 'suprapatellar bursada', 'suprapatelarno', 'suprapatellaire recessus', 'hoffa', 'hoffit', 'plica', 'plika', 'πλικα', 'fat pad[^.]{0,20}(edema|oedema)', 'kapsul', 'capsul', 'καψ', 'капсул', '\\bpannus\\b', '\\bsinov', '\\bsynov')

def _polarity(clause: str, span=None) -> str:
    if UNCERTAIN.search(clause):
        return 'uncertain'
    if span is None or not FEATURES['directional_negation']:
        if NEGATION.search(clause):
            return 'negative'
    elif _negated(clause, span[0], span[1]):
        return 'negative'
    if NORMALITY.search(clause):
        if TEAR.search(clause) or GRADE_HIGH.search(clause):
            return 'positive'
        return 'negative'
    return 'positive'

def _severity(clause: str) -> float:
    high = SEV_HIGH.search(clause) is not None
    low = SEV_LOW.search(clause) is not None
    if high and (not low):
        return 1.0
    if low and (not high):
        return 0.45
    if high and low:
        return 0.8
    return 0.75

def _grade(n_pos, n_neg, n_unc, best):
    if n_pos or n_unc:
        score = min(0.97, 0.5 + 0.45 * best + 0.015 * min(n_pos, 3))
        conf = min(1.0, 0.55 + 0.15 * n_pos)
    elif n_neg:
        score = max(0.04, 0.2 - 0.04 * n_neg)
        conf = min(0.9, 0.45 + 0.12 * n_neg)
    else:
        score, conf = (0.28, 0.05)
    return (score, conf)

def _paired_weight(clause: str, meniscus: bool) -> float:
    g = _grade_of(clause) if FEATURES['graded_pathology'] else None
    tear = TEAR.search(clause) is not None
    if meniscus:
        if tear:
            base = 1.0
        elif g is not None:
            base = 0.95 if g >= 3 else 0.3
        elif DEGEN.search(clause):
            base = 0.35
        else:
            base = 0.45
    elif tear:
        base = 1.0
    elif g is not None:
        base = 0.85 if g >= 2 else 0.3
    elif DEGEN.search(clause):
        base = 0.4
    else:
        base = 0.55
    if SEV_HIGH.search(clause) and (not SEV_LOW.search(clause)):
        base = min(1.0, base * 1.2)
    elif SEV_LOW.search(clause) and (not SEV_HIGH.search(clause)):
        base *= 0.7
    return base

def _score_paired(cls, tgt):
    anat_rx = ANAT_MATCH[tgt]
    path_rx = _rx(TEAR.pattern, DEGEN.pattern, INJURY.pattern)
    meniscus = 'Meniscus' in tgt
    n_pos = n_neg = n_unc = 0
    best = 0.0
    for c in cls:
        hit = anat_rx.search(c)
        if hit is None and meniscus and PLURAL_MENISCI.search(c) and (not ANY_SIDE.search(c)):
            hit = PLURAL_MENISCI.search(c)
        if hit is None:
            continue
        pm = path_rx.search(c)
        if pm is None and _grade_of(c) is None:
            if NORMAL_PHRASE.search(c) or (NORMALITY.search(c) and (not NEGATION.search(c))):
                n_neg += 1
            continue
        span = (pm.start(), pm.end()) if pm is not None else None
        pol = _polarity(c, span)
        if pol == 'positive':
            n_pos += 1
            best = max(best, _paired_weight(c, meniscus))
        elif pol == 'negative':
            n_neg += 1
        else:
            n_unc += 1
            best = max(best, 0.45 * _paired_weight(c, meniscus))
    s, cf = _grade(n_pos, n_neg, n_unc, best)
    return (s, cf, n_pos, n_neg)

def _score_clauses(cls, anat_rx, path_rx=None, decoy_rx=None, context_penalty=None, context_bonus=None):
    n_pos = n_neg = n_unc = 0
    best = 0.0
    for c in cls:
        m = anat_rx.search(c)
        if not m:
            continue
        if decoy_rx is not None and decoy_rx.search(c):
            continue
        if path_rx is not None and (not path_rx.search(c)):
            if NORMAL_PHRASE.search(c) or (NORMALITY.search(c) and (not NEGATION.search(c))):
                n_neg += 1
            continue
        pol = _polarity(c, (m.start(), m.end()))
        if pol == 'positive':
            n_pos += 1
            w = _severity(c)
            if context_penalty is not None and context_penalty.search(c):
                w *= 0.45
            if context_bonus is not None and context_bonus.search(c):
                w = min(1.0, w * 1.35)
            best = max(best, w)
        elif pol == 'negative':
            n_neg += 1
        else:
            n_unc += 1
            best = max(best, 0.3)
    s, c = _grade(n_pos, n_neg, n_unc, best)
    return (s, c, n_pos, n_neg)

def _score_oa(cls):
    acc = {t: {'pos': 0, 'neg': 0, 'unc': 0, 'best': 0.0} for t in OA_TARGETS}
    g_pos, g_neg, g_best = (0, 0, 0.0)
    for c in cls:
        m = OA_EVIDENCE.search(c)
        if not m:
            continue
        pol = _polarity(c, (m.start(), m.end()))
        sev = _severity(c)
        tf_med = _near(c, TF_SITE, SIDE_MEDIAL, 45)
        tf_lat = _near(c, TF_SITE, SIDE_LATERAL, 45)
        pf = PF_SITE.search(c) is not None
        hits = []
        if tf_med:
            hits.append('Medial OA')
        if tf_lat:
            hits.append('Lateral OA')
        if pf:
            hits.append('PF OA')
        if not hits:
            if pol == 'positive':
                g_pos += 1
                g_best = max(g_best, sev if GLOBAL_OA.search(c) else sev * 0.7)
            elif pol == 'negative':
                g_neg += 1
            continue
        for t in hits:
            if pol == 'positive':
                acc[t]['pos'] += 1
                acc[t]['best'] = max(acc[t]['best'], sev)
            elif pol == 'negative':
                acc[t]['neg'] += 1
            else:
                acc[t]['unc'] += 1
                acc[t]['best'] = max(acc[t]['best'], 0.3)
    out = {}
    for t in OA_TARGETS:
        a = acc[t]
        pos, neg, unc, best = (a['pos'], a['neg'], a['unc'], a['best'])
        if not (pos or unc) and g_pos and FEATURES['oa_inherit']:
            if neg:
                score, conf = _grade(0, neg, 0, 0.0)
                score = max(score, 0.35)
                conf *= 0.7
            else:
                score, conf = _grade(g_pos, 0, 0, g_best * 0.92)
                conf *= 0.75
        else:
            score, conf = _grade(pos, neg + g_neg, unc, best)
        out[t] = (score, conf, pos, neg)
    return out

def extract(report: str) -> dict:
    cls = clauses(report)
    out = {}
    for tgt in PAIRED:
        s, c, npos, nneg = _score_paired(cls, tgt)
        out[tgt] = s
        out[tgt + '__conf'] = c
        out[tgt + '__npos'] = npos
        out[tgt + '__nneg'] = nneg
    for tgt, (s, c, npos, nneg) in _score_oa(cls).items():
        out[tgt] = s
        out[tgt + '__conf'] = c
        out[tgt + '__npos'] = npos
        out[tgt + '__nneg'] = nneg
    for tgt in ('Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture'):
        if tgt == 'Contusion':
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt), context_penalty=DEGENERATIVE_MARROW, context_bonus=TRAUMA)
        else:
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt))
        out[tgt] = s
        out[tgt + '__conf'] = c
        out[tgt + '__npos'] = npos
        out[tgt + '__nneg'] = nneg
    if FEATURES['synovitis_backoff'] and out['Synovitis__npos'] == 0 and (out['Synovitis__nneg'] == 0):
        proxy = sum((1 for c in cls if SYNOVIAL_PROXY.search(c) and _polarity(c) == 'positive'))
        eff = out['Effusion']
        prior = 0.3 + 0.3 * max(0.0, (eff - 0.5) / 0.45) + 0.06 * min(proxy, 3)
        out['Synovitis'] = min(0.72, prior)
        out['Synovitis__conf'] = 0.18
    return out

## 2. 6-Slot Multi-Planar Pulse-Sequence Decomposition

Clinical knee MR protocols vary substantially across institutions and manufacturers (GE, Siemens, Philips, Toshiba). A standard examination comprises orthogonal planes crossed with pulse sequence weightings:

```
 ┌────────────────────────────────────────────────────────────────────────────────────────────────┐
 │                                   6-SLOT MRI ANATOMICAL PROTOCOL                                │
 ├──────┬──────────────────┬──────────┬──────────────┬──────────────────┬─────────────────────────┤
 │ Slot │ Slot Name        │ Plane    │ Fluid Sens.  │ Fat Suppression  │ Primary Diagnostic Role │
 ├──────┼──────────────────┼──────────┼──────────────┼──────────────────┼─────────────────────────┤
 │ 0    │ SAG_FLUID_FS     │ Sagittal │ True         │ True             │ ACL, Meniscus, Baker's  │
 │ 1    │ COR_FLUID_FS     │ Coronal  │ True         │ True             │ MCL, Meniscus, Edema    │
 │ 2    │ AX_FLUID_FS      │ Axial    │ True         │ True             │ Patellofemoral, Joint   │
 │ 3    │ SAG_FLUID_NOFS   │ Sagittal │ True         │ False            │ Ligament Anatomy        │
 │ 4    │ COR_T1           │ Coronal  │ False        │ False            │ Subchondral Bone / OA   │
 │ 5    │ SAG_T1           │ Sagittal │ False        │ False            │ Marrow Fat / Structure  │
 └──────┴──────────────────┴──────────┴──────────────┴──────────────────┴─────────────────────────┘
```

The header probe inspects `SeriesDescription`, `SequenceName`, `ScanOptions`, `ScanningSequence`, `RepetitionTime` ($T_R$), and `EchoTime` ($T_E$) to assign each series into its unique slot predicate.


### 実行環境のセットアップとGPU実行プローブ

**何をしているか**: スレッド数の固定（`OMP_NUM_THREADS=4` 等）とライブラリ読み込みに加え、
`_cuda_execution_probe(index)` で**各GPUに小さな畳み込みを1回実際に流して**動作を確認します。

**なぜそうするのか**:

- **スレッド数の固定**: NumPy/OpenBLAS は既定でCPU全コアを使おうとします。
  DICOM読み込みを自前のスレッドプールで並列化している本notebookでは、
  **並列の二重掛け**が起きてかえって遅くなります（オーバーサブスクリプション）。
  外側で並列化するなら、内側は抑えるのが鉄則です。
- **GPU実行プローブ**: `torch.cuda.is_available()` が `True` を返しても、
  実際には**そのGPUで演算できない**ことがあります
  （他プロセスがメモリを占有、ECCエラー、ドライバとcompute capabilityの不整合など）。
  T4 x2 構成の片方だけが不調、という状況は現実に起こります。
  **本番の重い推論を始める前に、小さな演算で実際に叩いて確かめる**のが一番確実です。

> この「宣言を信じず、実際に試す」という発想は、
> 分散処理やGPUを扱うコードでは繰り返し出てくる重要な作法です。

In [ ]:
from __future__ import annotations
import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_v, '4')
import gc
import hashlib
import json
import re
import time
import traceback
import threading
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F

def _cuda_execution_probe(index):
    dev = torch.device(f'cuda:{index}')
    try:
        major, minor = torch.cuda.get_device_capability(index)
        probe = nn.Conv2d(3, 4, kernel_size=3, padding=1).eval().to(dev)
        with torch.inference_mode():
            out = probe(torch.zeros((1, 3, 16, 16), device=dev))
            if tuple(out.shape) != (1, 4, 16, 16):
                raise RuntimeError(f'unexpected CUDA probe shape {tuple(out.shape)}')
        torch.cuda.synchronize(index)
        print(f'cuda:{index} probe PASS (compute {major}.{minor})')
        del probe, out
        torch.cuda.empty_cache()
        return True
    except Exception as exc:
        print(f'cuda:{index} probe FAIL ({type(exc).__name__}: {exc}); using CPU fallback')
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        return False
DEVS = []
if torch.cuda.is_available():
    DEVS = [torch.device(f'cuda:{i}') for i in range(torch.cuda.device_count()) if _cuda_execution_probe(i)]
if not DEVS:
    DEVS = [torch.device('cpu')]
print(f'devices: {[str(d) for d in DEVS]}')
T0 = time.time()
SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
CROP_MM = 130.0
CACHE_IMG = 336
GROUP = 3
N_GROUP_MAX = 1
CACHE_FRACTION = 0.45
CACHE_BUDGET_MAX_GB = 24.0
CACHE_BUDGET_GB = 12.0
TEST_SHARE = 0.3
HDR_THREADS = 16
PIX_THREADS = 12
ORDER_THREADS = 32
ORDER_BUDGET_S = 5400
RUNS = [{'name': 'r224', 'img': 224}, {'name': 'r336', 'img': 336}]
EPOCHS = 10
BATCH_STUDIES = 8
AUG_ROT_DEG = 8.0
AUG_SCALE = 0.08
AUG_SHIFT = 0.05
AUG_INTENSITY = 0.1
LAT_MIN_OFFSET_MM = 20.0
SLICE_BAND = (0.2, 0.8)
RULES_NATIVE = {'order': 'normal', 'lat': 'centre', 'slot_fallback': False, 'decode_fill': 'nearest'}
RULES_LEGACY = {'order': 'dominant_axis', 'lat': 'corner_x', 'slot_fallback': True, 'decode_fill': 'zero'}
RULES = dict(RULES_NATIVE)
LEGACY_LAT_OFFSET_MM = 5.0
LR_HEAD = 0.001
LR_BACKBONE = 8e-06
UNFREEZE_LAST = 6
WEIGHT_DECAY = 0.02
EVAL_BATCH = 8
TIME_BUDGET = 8.0 * 3600
SLOTS_RECOVERED = [('SAG_FLUID_FS', 'Sagittal', True, True), ('COR_FLUID_FS', 'Coronal', True, True), ('AX_FLUID_FS', 'Axial', True, True), ('SAG_FLUID_NOFS', 'Sagittal', True, False), ('COR_T1', 'Coronal', False, False), ('SAG_T1', 'Sagittal', False, False)]
SLOTS_PUBLIC = [('SAG_FLUID', 'Sagittal', None, True), ('COR_FLUID', 'Coronal', None, True), ('AX_FLUID', 'Axial', None, True), ('SAG_STRUCT', 'Sagittal', None, False), ('COR_STRUCT', 'Coronal', None, False), ('AX_STRUCT', 'Axial', None, False)]
SLOT_SCHEME = os.environ.get('SLOT_SCHEME', 'recovered')
SLOTS = SLOTS_PUBLIC if SLOT_SCHEME == 'public' else SLOTS_RECOVERED
N_SLOT = len(SLOTS)
POOL_PARTS = {'cls_mean': 2, 'cls_mean_focal': 3}
SLOT_PRIOR_TABLE = {'ACL': (0, 3, 5), 'MCL': (1, 4), 'Medial Meniscus': (0, 1, 3, 4), 'Lateral Meniscus': (0, 1, 3, 4), 'Medial OA': (1, 4, 5), 'Lateral OA': (1, 4, 5), 'PF OA': (0, 2, 5), 'Effusion': (0, 2), 'Synovitis': (0, 2), "Baker's": (0,), 'Contusion': (0, 1, 2), 'Fracture': (0, 1, 2, 4, 5)}
SLOT_PRIOR_STRENGTH = 0.55
FATSAT_OPTS = {'FS', 'FATSAT', 'FAT_SAT', 'FSAT'}
_SEP = re.compile('[_\\-.]')
_FATSAT_RX = re.compile('\\bfs\\b|fatsat|fat sat|\\bstir\\b|\\bspair\\b|\\bspir\\b|\\bwe\\b|water excit|\\btirm\\b|\\bsting\\b|\\bfatsup\\b')
_T1_RX = re.compile('\\bt1\\b|\\bt1w\\b')
_T2_RX = re.compile('\\bt2\\b|\\bt2w\\b')
_PD_RX = re.compile('\\bpd\\b|\\bpdw\\b|proton|\\bdp\\b|dens')

### 経過時間つきログとデータルートの自動探索

**何をしているか**:

- `log(msg)` … 開始からの経過秒数を付けて出力する。`flush=True` で即時反映
- `find_root()` … `test.csv` と `test_series/` を持つディレクトリを候補から探す
- `find_dinov2(variant)` … DINOv2 の重みディレクトリを探す

**なぜそうするのか**: Code Competition には**実行時間の上限**があり、
「どの工程に何分かかったか」が分からないと最適化のしようがありません。
経過秒つきログは、時間切れの原因を後から特定するための最低限の装備です。
`flush=True` は、途中でタイムアウトした場合でも**そこまでのログが確実に残る**ようにするためです。

`find_root()` が複数の候補パスを順に試すのは、Biohub notebook と同じ思想です。
興味深いのは**フォールバックの再帰探索**で、既知の候補が全滅した場合に
`/kaggle/input` を2階層まで走査して `test.csv` を持つディレクトリを探します。
「マウント名が変わっても動く」ようにしてあるわけです。

In [ ]:
def log(msg):
    print(f'[{time.time() - T0:7.1f}s] {msg}', flush=True)

def find_root():
    for c in [Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'), Path('/kaggle/input/rsna-knee-abnormality-detection'), Path('data'), Path('.')]:
        if (c / 'test.csv').is_file() and (c / 'test_series').is_dir():
            return c
    base = Path('/kaggle/input')
    if base.is_dir():
        for depth1 in sorted((p for p in base.iterdir() if p.is_dir())):
            for cand in [depth1] + sorted((p for p in depth1.iterdir() if p.is_dir())):
                if (cand / 'test.csv').is_file():
                    return cand
    raise FileNotFoundError(f'competition mount not found (cwd {Path.cwd()}); expected a directory holding test.csv and test_series/')

def find_dinov2(variant='small'):
    base = Path('/kaggle/input')
    if not base.is_dir():
        return None
    hits = []
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
        if 'config.json' in files and 'dinov2' in root.lower():
            hits.append(Path(root))
    for h in hits:
        if variant in str(h).lower():
            return h
    return hits[0] if hits else None
LABEL_COLS = TARGETS + [t + '__conf' for t in TARGETS]

class LabelSourceError(RuntimeError):
    pass

def find_label_table():
    base = Path('/kaggle/input')
    cands = []
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
            cands += [Path(root) / f for f in files if f.startswith('report_labels') and f.endswith('.csv')]
    cands += [p for p in (Path('data/derived/report_labels_v2.csv'),) if p.is_file()]
    for c in cands:
        try:
            head = pd.read_csv(c, nrows=1)
        except Exception:
            continue
        if 'StudyInstanceUID' in head.columns and all((t in head.columns for t in TARGETS)):
            return c
    return None

def label_mount_attached():
    base = Path('/kaggle/input')
    if not base.is_dir():
        return False
    return any(('label' in p.name.lower() for p in base.iterdir() if p.is_dir()))

def read_labels(train_df):
    n = len(train_df)
    lab = pd.DataFrame([extract(r) for r in train_df['Report'].fillna('')])
    lab['StudyInstanceUID'] = train_df['StudyInstanceUID'].values
    lab = lab.set_index('StudyInstanceUID')
    src = find_label_table()
    if src is None:
        if label_mount_attached():
            raise LabelSourceError('LABEL SOURCE: a label dataset is mounted but no usable table was found in it. Falling back to the lexicon here would train on the weaker labels and say so only in a log line, so the run stops instead.')
        log(f'LABEL SOURCE: lexicon, {n} studies (no table mounted)')
        return lab
    tab = pd.read_csv(src).set_index('StudyInstanceUID')
    missing = [c for c in LABEL_COLS if c not in tab.columns]
    if missing:
        raise LabelSourceError(f'LABEL SOURCE: {src} is missing {len(missing)} expected columns (first: {missing[0]!r}). Refusing to fall back silently.')
    hit = lab.index.intersection(tab.index)
    if not len(hit):
        raise LabelSourceError(f'LABEL SOURCE: {src} shares no StudyInstanceUID with train.csv.')
    log(f'LABEL SOURCE: {src.name} covers {len(hit)} of {n} studies, lexicon for the remaining {n - len(hit)}')
    lab.loc[hit, LABEL_COLS] = tab.loc[hit, LABEL_COLS].values
    return lab
ROOT = find_root()
log(f'input root: {ROOT}')
IMG = CACHE_IMG

def available_gb():
    try:
        with open('/proc/meminfo') as fh:
            info = {k.strip(): v for k, v in (l.split(':', 1) for l in fh if ':' in l)}
        return int(info['MemAvailable'].split()[0]) / 1024 ** 2
    except Exception:
        return CACHE_BUDGET_GB / CACHE_FRACTION

def plan_cache(n_study, n_test=0):
    avail = available_gb()
    budget = min(avail * CACHE_FRACTION, CACHE_BUDGET_MAX_GB)
    n_total = n_study + max(n_test, int(TEST_SHARE * n_study))
    per_slice = n_total * N_SLOT * IMG * IMG
    afford = int(budget * 1024 ** 3 // max(per_slice, 1))
    groups = max(1, min(N_GROUP_MAX, afford // GROUP))
    log(f'memory: {avail:.1f} GB available, {budget:.1f} GB to the cache; sizing for {n_study} train + {n_total - n_study} test studies -> {groups} group(s) of {GROUP} = {groups * GROUP} slices per slot' + (f' (wanted {N_GROUP_MAX})' if groups < N_GROUP_MAX else ''))
    return groups
N_GROUP = plan_cache(len(pd.read_csv(ROOT / 'train.csv')), len(pd.read_csv(ROOT / 'test.csv')))
CACHE_SLICES = GROUP * N_GROUP
log(f'cache layout: {N_GROUP} groups x {GROUP} slices = {CACHE_SLICES} per slot')

### DICOMヘッダから撮像プロトコルと左右を復元する

**何をしているか**: DICOMのヘッダタグ（`SeriesDescription`, `ScanningSequence`,
`RepetitionTime`, `EchoTime`, `Laterality`, `ImagePositionPatient`,
`ImageOrientationPatient`, `PixelSpacing` …）を読み、以下を推定します。

- **撮像面**（矢状断 / 冠状断 / 横断）
- **脂肪抑制の有無**（fat saturation）
- **液体強調かどうか**（T2/PD系か、T1系か）
- **左右**（`side_from_geometry`: `Laterality` タグが欠けている場合、
  画像中の膝の**重心が患者座標系の正中からどちら側にあるか**で判定）

**なぜそうするのか**: 膝MRIの検査は、施設・メーカー（GE / Siemens / Philips / Toshiba）ごとに
**シリーズの名前の付け方がバラバラ**です。`SeriesDescription` に
"SAG PD FS" と書く施設もあれば "sag_t2_tse_fs" と書く施設もある。
文字列一致だけでは取りこぼします。

そこで本notebookは、**文字列だけでなく物理パラメータ**（TR/TEの値、方向余弦ベクトル）も
併用して判定しています。方向余弦（`ImageOrientationPatient`）は
**画像平面の向きを表す6つの数値**で、これが分かれば撮像面は幾何学的に確定します。
名前の付け方の流儀に依存しない、**より頑健な判定**になります。

左右の復元がなぜ必要かというと、後段で**右膝を左膝の向きに反転して揃える**ためです。
これを揃えないと、モデルは「内側」「外側」を一貫して学習できません。
`Laterality` タグは欠けていることが多いので、**幾何から復元する経路を持っておく**のは実務的です。

In [ ]:
HDR_TAGS = ['SeriesDescription', 'SequenceName', 'ScanOptions', 'ScanningSequence', 'RepetitionTime', 'EchoTime', 'Laterality', 'PixelSpacing', 'Rows', 'Columns', 'RescaleSlope', 'RescaleIntercept', 'ImagePositionPatient', 'ImageOrientationPatient']

def _hdr_vec(s, n):
    if not isinstance(s, str):
        return None
    try:
        v = [float(x) for x in s.split('|')]
    except ValueError:
        return None
    return np.array(v) if len(v) >= n else None

def side_from_geometry(h):
    cx = {}
    for r in h.itertuples(index=False):
        ipp = _hdr_vec(getattr(r, 'ImagePositionPatient', None), 3)
        iop = _hdr_vec(getattr(r, 'ImageOrientationPatient', None), 6)
        ps = _hdr_vec(getattr(r, 'PixelSpacing', None), 2)
        rows, cols = (getattr(r, 'Rows', None), getattr(r, 'Columns', None))
        if ipp is None or iop is None or ps is None or (not rows) or (not cols):
            continue
        try:
            c = ipp[:3] + iop[:3] * ps[1] * float(cols) / 2 + iop[3:6] * ps[0] * float(rows) / 2
        except (TypeError, ValueError):
            continue
        cx.setdefault(r.StudyInstanceUID, []).append(float(c[0]))
    out = {}
    for st, xs in cx.items():
        m = float(np.median(xs))
        out[st] = None if abs(m) < LAT_MIN_OFFSET_MM else 'R' if m < 0 else 'L'
    return out

def side_from_corner_x(h):
    out = {}
    for st, g in h.groupby('StudyInstanceUID'):
        xs = []
        for r in g.itertuples(index=False):
            ipp = _hdr_vec(getattr(r, 'ImagePositionPatient', None), 3)
            if ipp is not None and np.isfinite(ipp).all():
                xs.append(float(ipp[0]))
        if not xs:
            out[st] = None
            continue
        x = float(np.median(xs))
        out[st] = None if abs(x) < LEGACY_LAT_OFFSET_MM else 'R' if x < 0 else 'L'
    return out

def lat_of(h, tag=''):
    geo = side_from_corner_x(h) if RULES['lat'] == 'corner_x' else side_from_geometry(h)
    d, n_tag, n_geo, n_none, n_disagree = ({}, 0, 0, 0, 0)
    for st, g in h.groupby('StudyInstanceUID'):
        v = [str(x).strip().upper() for x in g['Laterality'].dropna()]
        if RULES['lat'] == 'corner_x' and 'ImageLaterality' in g.columns:
            v += [str(x).strip().upper() for x in g['ImageLaterality'].dropna()]
        v = [x[0] for x in v if x and x[0] in ('L', 'R')]
        side = v[0] if v else None
        if side is not None:
            n_tag += 1
            if geo.get(st) is not None and geo[st] != side:
                n_disagree += 1
        else:
            side = geo.get(st)
            n_geo += side is not None
            n_none += side is None
        d[st] = side
    log(f'{tag}laterality: {n_tag} from the tag, {n_geo} from geometry, {n_none} unresolved; tag and geometry disagree on {n_disagree} ({n_disagree / max(n_tag, 1):.1%} of the tagged)')
    return d

def probe(item):
    split, study, series, path = item
    row = {'split': split, 'StudyInstanceUID': study, 'SeriesInstanceUID': series, 'dir': path}
    try:
        files = sorted((e.name for e in os.scandir(path) if e.name.endswith('.dcm')))
        row['files'] = files
        row['n_slices'] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]), stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == 'MultiValue':
                row[t] = '|'.join((str(x) for x in v))
            else:
                row[t] = str(v)
    except Exception as exc:
        row['err'] = str(exc)[:120]
    return row

def walk(split):
    base = ROOT / split
    items = []
    if not base.is_dir():
        return pd.DataFrame(columns=['split', 'StudyInstanceUID', 'SeriesInstanceUID', 'dir', 'files', 'n_slices'] + HDR_TAGS)
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))
    return pd.DataFrame(rows)

def annotate(df):
    desc = df['SeriesDescription'].fillna('') + ' ' + df['SequenceName'].fillna('')
    desc = desc.str.lower().str.replace(_SEP, ' ', regex=True)
    opts = df['ScanOptions'].fillna('').str.upper().str.split('|')
    opts_fs = opts.apply(lambda ts: any((t.strip() in FATSAT_OPTS for t in ts)))
    df['fatsat'] = desc.str.contains(_FATSAT_RX) | opts_fs
    tr = pd.to_numeric(df['RepetitionTime'], errors='coerce')
    te = pd.to_numeric(df['EchoTime'], errors='coerce')
    gre = df['ScanningSequence'].fillna('').str.upper().str.contains('GR')
    t1, t2, pdw = (desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX))
    df['weight'] = np.where(t1 & ~t2 & ~pdw, 'T1', np.where(t2 & ~pdw, 'T2', np.where(pdw, 'PD', np.where(gre, 'GRE', np.where(tr < 800, 'T1', np.where(te > 60, 'T2', np.where(tr >= 800, 'PD', 'UNK')))))))
    df['fluid'] = np.isin(df['weight'], ['PD', 'T2'])
    df['px'] = pd.to_numeric(df['PixelSpacing'].fillna('').str.split('|').str[0].replace('', np.nan), errors='coerce')
    return df

### 6スロットへのシリーズ割り当て

**何をしているか**: 各検査（study）について、6つの「スロット」
（例: SAG_FLUID_FS, COR_FLUID_FS, AX_FLUID_FS, SAG_T1 …）それぞれに
**条件に合うシリーズを1本ずつ選ぶ**。複数該当した場合は
`sort_values('n_slices', ascending=False).iloc[0]` で**スライス数が最大のもの**を採用します。

**なぜそうするのか**: 検査ごとに含まれるシリーズの数も種類もバラバラですが、
モデルへの入力は**固定長のテンソル**でなければなりません。
そこで「臨床的に意味のある6種類の枠」を定義し、そこに流し込みます。

設計上の要点が2つあります。

- **スライス数最大を選ぶ**: 同じ条件のシリーズが複数ある場合、
  スライスが多いほうが**撮像範囲が広く、位置決め（scout）画像などの短い系列を避けられます**。
  単純ですが効果的なヒューリスティックです。
- **フォールバックの限定**: `RULES['slot_fallback']` が有効で、かつ
  `fluid is False`（＝T1系スロット）のときだけ、脂肪抑制なしの同一平面で代用します。
  **どのスロットでも代用を許すわけではありません**。
  たとえば「脂肪抑制ありの矢状断」の枠に脂肪抑制なしの画像を入れると、
  骨髄浮腫の見え方が全く変わってしまいます。**代用してよい枠とダメな枠を区別する**——
  これは医用画像特有の、ドメイン知識に基づく設計判断です。

該当シリーズが1つも無いスロットは、後段で**欠損マスク**として扱われます
（ゼロ埋めして「あることにする」のではなく、明示的に「無い」と伝える）。

In [ ]:
def pick_slots(series_df, plane_map):
    series_df = series_df.copy()
    series_df['plane'] = series_df['SeriesInstanceUID'].map(plane_map)
    out = {}
    for study, g in series_df.groupby('StudyInstanceUID'):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g['plane'] == plane) & (g['fatsat'] == fs)
            if fluid is not None:
                sel &= g['fluid'] == fluid
            cand = g[sel]
            if len(cand) == 0 and RULES['slot_fallback'] and (fluid is False):
                cand = g[(g['plane'] == plane) & ~g['fatsat']]
            if len(cand):
                chosen[name] = cand.sort_values('n_slices', ascending=False).iloc[0]
        out[study] = chosen
    return out

## 3. Deterministic Geometric Normalization & Through-Plane Ordering

### 1. Slice Ordering via Through-Plane Projection
Arbitrary DICOM SOP Instance UIDs bear no correlation to spatial depth ($\text{Spearman} \approx 0.009$). The physical ordering is computed by projecting the 3D origin $\mathbf{p} = \text{ImagePositionPatient}$ onto the through-plane normal unit vector $\mathbf{n}$:

$$\mathbf{n} = \mathbf{r}_x \times \mathbf{r}_y, \quad k = \mathbf{p} \cdot \mathbf{n}$$

Sorting by signed scalar $k$ guarantees monotonic slice progression from medial-to-lateral or superior-to-inferior.

### 2. Laterality Centroid Normalization
Missing `Laterality` tags are resolved by evaluating the spatial centroid of the knee relative to the patient's anatomical midline (LPS coordinate system, $+x = \text{Left}$):

$$\mathbf{c}_x = \mathbf{p}_x + \mathbf{r}_{x,0} \cdot s_y \cdot \frac{\text{Cols}}{2} + \mathbf{r}_{y,0} \cdot s_x \cdot \frac{\text{Rows}}{2}$$

$$\text{Laterality} = \begin{cases} \text{Right} & \text{if } \text{median}(\mathbf{c}_x) \le -20.0\text{ mm} \\ \text{Left} & \text{if } \text{median}(\mathbf{c}_x) \ge +20.0\text{ mm} \\ \text{Unresolved} & \text{otherwise} \end{cases}$$

Right knees are mirrored to the standard Left-knee canonical orientation ($\text{flip}_x$ for Coronal/Axial, slice-sequence reversal for Sagittal).


### スライスの物理的な並び順を復元し、画像キャッシュのタグを作る

**何をしているか**:

- `cache_tag(rules)` … 解像度・スライス数・クロップ幅などから
  キャッシュの識別子を作り、既定と違う設定なら設定辞書のMD5ハッシュを付ける
- `_natural_key(name)` … `img2.dcm` が `img10.dcm` より前に来るよう、
  数字部分を整数として比較する「自然順」ソート
- `_order_dominant_axis(rec)` … `ImagePositionPatient` を読み、
  **最も変化の大きい軸**を主軸として並べ替える

**なぜそうするのか**: **DICOMのファイル名やSOP Instance UIDは、
スライスの空間的な順序とは何の関係もありません**
（原著の記述では順位相関 Spearman ≈ 0.009、つまり実質無相関）。
ファイル名順に読むと、**バラバラの深さのスライスが並んだ偽物のボリューム**ができあがります。
3Dの文脈を使うモデルにとってこれは致命的です。

正しいやり方は、各スライスの3D原点 `p = ImagePositionPatient` を
**面の法線ベクトル `n = r_x × r_y` に射影**し、そのスカラー値 `k = p · n` でソートすることです。
法線は方向余弦ベクトルの外積で求まります。こうすると
**「内側→外側」「上→下」という単調な進行**が保証されます。

キャッシュタグにハッシュを埋め込むのも実用的です。
前処理の設定を変えたのにキャッシュを再利用してしまう事故は非常に多く、
**設定が変わればタグが変わる**設計にしておけば自動的に別キャッシュとして扱われます。

In [ ]:
ORDER_TAGS = [(32, 50), (32, 55), (32, 19)]
DECODE_FAILED = []

def cache_tag(rules=None):
    r = dict(RULES if rules is None else rules)
    t = f'{CACHE_IMG}px_{CACHE_SLICES}sl_{int(CROP_MM)}mm_{SLICE_BAND[0]:.2f}-{SLICE_BAND[1]:.2f}'
    if {k: r.get(k, v) for k, v in RULES_NATIVE.items()} != RULES_NATIVE:
        t += '_' + hashlib.md5(json.dumps(r, sort_keys=True).encode()).hexdigest()[:6]
    return t

def _natural_key(name):
    return tuple((int(x) if x.isdigit() else x.lower() for x in re.split('(\\d+)', str(name))))

def _order_dominant_axis(rec):
    files, d = (rec['files'], rec['dir'])
    rows = []
    for pos, f in enumerate(files):
        ipp = inst = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True, specific_tags=['ImagePositionPatient', 'InstanceNumber'])
            raw = getattr(ds, 'ImagePositionPatient', None)
            if raw is not None and len(raw) >= 3:
                c = np.asarray(raw[:3], dtype=np.float64)
                if np.isfinite(c).all():
                    ipp = c
            n = getattr(ds, 'InstanceNumber', None)
            if n is not None:
                inst = float(n)
        except Exception:
            pass
        rows.append((f, ipp, inst, pos))
    placed = [r for r in rows if r[1] is not None]
    need = max(2, int(0.8 * len(rows)))
    if len(placed) >= need:
        xyz = np.stack([r[1] for r in placed])
        axis = int(np.argmax(np.ptp(xyz, axis=0)))
        spare = float(np.nanmedian(xyz[:, axis]))
        rows.sort(key=lambda r: (float(r[1][axis]) if r[1] is not None else spare, r[2] if r[2] is not None else float('inf'), r[3]))
    elif sum((r[2] is not None for r in rows)) >= need:
        rows.sort(key=lambda r: (r[2] if r[2] is not None else float('inf'), r[3]))
    else:
        rows.sort(key=lambda r: _natural_key(r[0]))
    return ([r[0] for r in rows], True)

def order_slices(rec):
    if RULES['order'] == 'dominant_axis':
        return _order_dominant_axis(rec)
    files, d = (rec['files'], rec['dir'])
    keyed = []
    for f in files:
        k = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True, specific_tags=ORDER_TAGS)
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            k = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        except Exception:
            try:
                k = float(ds.InstanceNumber)
            except Exception:
                k = None
        keyed.append((k, f))
    if any((k is None for k, _ in keyed)):
        return (files, False)
    return ([f for _, f in sorted(keyed, key=lambda t: t[0])], True)

def read_slot(rec, n_slice=None, out_size=None):
    n_slice = GROUP if n_slice is None else n_slice
    out_size = IMG if out_size is None else out_size
    files, d, px = (rec.get('ordered') or rec['files'], rec['dir'], rec['px'])
    n = len(files)
    if n == 0:
        return None
    lo, hi = (int(SLICE_BAND[0] * (n - 1)), int(SLICE_BAND[1] * (n - 1)))
    idx = np.unique(np.linspace(lo, hi, n_slice).astype(int)) if hi > lo else np.array([n // 2])
    while len(idx) < n_slice:
        idx = np.append(idx, idx[-1])
    planes = []
    for i in idx[:n_slice]:
        try:
            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, 'RescaleSlope', 1) or 1)
            ic = float(getattr(ds, 'RescaleIntercept', 0) or 0)
            a = a * sl + ic
        except Exception:
            a = None
        planes.append(a)
    got = [k for k, p in enumerate(planes) if p is not None]
    if RULES['decode_fill'] == 'zero':
        if not got:
            DECODE_FAILED.append(rec.get('SeriesInstanceUID', d))
        planes = [np.zeros((out_size, out_size), np.float32) if p is None else p for p in planes]
        got = list(range(len(planes)))
    if not got:
        DECODE_FAILED.append(rec.get('SeriesInstanceUID', d))
        return None
    if len(got) < len(planes):
        DECODE_FAILED.append(rec.get('SeriesInstanceUID', d))
        for k, p in enumerate(planes):
            if p is None:
                planes[k] = planes[min(got, key=lambda j: abs(j - k))]
    shp = planes[0].shape
    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
    vol = np.stack(planes)
    if px and np.isfinite(px) and (px > 0):
        want = int(round(CROP_MM / px))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx = (h // 2, w // 2)
            half = want // 2
            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]
    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-06), 0, 1)
    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(out_size, out_size), mode='bilinear', align_corners=False)
    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)

### 左右の正規化（右膝を左膝の向きに揃える）

**何をしているか**: 4行だけの短い関数です。右膝（`lat == 'R'`）のとき、
冠状断・横断なら**左右方向（最終軸）を反転**、矢状断なら**スライス方向（先頭軸）を反転**します。

**なぜそうするのか**: 12ターゲットのうち何個かは**左右の区別そのもの**です
（内側/外側半月板、内側/外側変形性関節症）。
右膝と左膝では、画像上で「内側」が現れる位置が**鏡像に入れ替わります**。

これを揃えずに学習すると、モデルは
「画面の左側にある半月板が内側かどうか」を、**まず膝の左右を推定してから**判断せねばならず、
本来不要な難しさを背負い込みます。
**前処理で解ける対称性は、前処理で解いてしまう**のが定石です。

なぜ矢状断だけ反転軸が違うのかというと、矢状断は**スライスを積む方向が左右方向**だからです。
矢状断のスライス列を反転することが、冠状断で画像を左右反転することに対応します。
このあたりは、幾何を紙に描いて確認しないと間違えます。

In [ ]:
def normalise_laterality(img, plane, lat):
    if lat != 'R':
        return img
    if plane in ('Coronal', 'Axial'):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])

### 画像キャッシュの一括構築

**何をしているか**: 全 study × 6スロットについて DICOM を読み、
リサイズ・クロップ・uint8化を済ませた配列を
`(n_study, 6, CACHE_SLICES, IMG, IMG)` の**巨大な uint8 配列**に詰め込みます。
同時に、どのスロットが存在するかを表す `mask` も作ります。

**なぜそうするのか**:

- **1回読んで何度も使う**: このパイプラインは同じ画像に対して
  24メンバー × 10窓 の推論を行います。毎回DICOMを読み直すのは論外で、
  一度デコードした結果を保持するのが必須です。
- **なぜ uint8 か**: float32 で持つとメモリが4倍になります。
  MRI画像はもともと表示上8bit相当の情報量しかないことが多く、
  uint8 で保持して**GPU上で `div_(255.0)` する**のが定石です。
  ログに `cache.nbytes / 1024**3` でGB数を出しているのは、
  **メモリ上限を超えないか事前に確認する**ためです。
- **`mask` を別に持つ理由**: 存在しないスロットをゼロ埋めしただけだと、
  モデルは「真っ黒な画像」を有効な入力として解釈してしまいます。
  マスクで明示すれば、アテンション計算時にそのスロットを除外できます。
  **欠損は「ゼロ」ではなく「欠損」として伝える**——表形式データでも画像でも同じ原則です。

ジョブをリストに展開してからスレッドプールで処理する構造になっており、
DICOMのデコードは GIL を解放する処理が多いため、Pythonのスレッドでも実際に速くなります。

In [ ]:
ORDER_CACHE = os.environ.get('RSNA_ORDER_CACHE') or None

def build_cache(slot_map, plane_map, lat_map, tag):
    studies = sorted(slot_map)
    sidx = {s: i for i, s in enumerate(studies)}
    cache = np.zeros((len(studies), N_SLOT, CACHE_SLICES, IMG, IMG), np.uint8)
    mask = np.zeros((len(studies), N_SLOT), np.float32)
    log(f'{tag}: cache {cache.shape} = {cache.nbytes / 1024 ** 3:.1f} GB')
    jobs = [(st, k, plane, slot_map[st][name]) for st in studies for k, (name, plane, _, _) in enumerate(SLOTS) if name in slot_map[st]]
    n_job = len(jobs)
    t_ord = time.time()
    n_slice_total = sum((len(j[3]['files']) for j in jobs))
    log(f'{tag}: ordering {len(jobs)} slot-series ({n_slice_total} slice headers)')
    ok = done = 0
    CHUNK_O = 1024
    seen = {}
    if ORDER_CACHE and Path(ORDER_CACHE).is_file():
        try:
            import json as _json
            seen = _json.loads(Path(ORDER_CACHE).read_text())
        except (OSError, ValueError):
            seen = {}
        hit = 0
        for _, _, _, rec in jobs:
            e = seen.get(rec['SeriesInstanceUID'])
            if e and len(e['files']) == len(rec['files']):
                rec['ordered'] = e['files']
                ok += int(e['good'])
                hit += 1
        jobs = [j for j in jobs if 'ordered' not in j[3]]
        log(f'{tag}: {hit} slot-series ordered from {ORDER_CACHE}, {len(jobs)} to read')
    with ThreadPoolExecutor(max_workers=ORDER_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK_O):
            block = jobs[c0:c0 + CHUNK_O]
            for (_, _, _, rec), (files, good) in zip(block, pool.map(lambda j: order_slices(j[3]), block)):
                rec['ordered'] = files
                ok += int(good)
                done += 1
                if ORDER_CACHE:
                    seen[rec['SeriesInstanceUID']] = {'files': files, 'good': bool(good)}
            budget = min(ORDER_BUDGET_S, max(60.0, (TIME_BUDGET - (time.time() - T0)) * 0.35))
            if time.time() - t_ord > budget:
                log(f'{tag}: ordering budget spent at {done}/{len(jobs)}; the rest keep file order')
                break
    if ORDER_CACHE and done:
        import json as _json
        _t = Path(ORDER_CACHE).with_suffix('.tmp')
        _t.write_text(_json.dumps(seen))
        _t.replace(Path(ORDER_CACHE))
    log(f'{tag}: ordered {ok}/{n_job} by geometry ({n_job - ok} kept arbitrary) in {time.time() - t_ord:.0f}s')
    jobs = [(st, k, plane, slot_map[st][name]) for st in studies for k, (name, plane, _, _) in enumerate(SLOTS) if name in slot_map[st]]
    log(f'{tag}: decoding {len(jobs)} slot-series')
    n_failed_before = len(DECODE_FAILED)
    CHUNK = 512
    done = 0
    with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK):
            block = jobs[c0:c0 + CHUNK]
            for (st, k, plane, _), img in zip(block, pool.map(lambda j: read_slot(j[3], CACHE_SLICES, IMG), block)):
                done += 1
                if img is None:
                    continue
                cache[sidx[st], k] = normalise_laterality(img, plane, lat_map.get(st)).numpy()
                mask[sidx[st], k] = 1.0
            if done % 4096 < CHUNK:
                log(f'  {tag} {done}/{len(jobs)}')
            if time.time() - T0 > TIME_BUDGET:
                log(f'  {tag}: time budget reached during decode')
                break
    n_failed = len(DECODE_FAILED) - n_failed_before
    log(f'{tag}: {int(mask.sum())}/{len(jobs)} slots filled' + (f'; {n_failed} series had a slice that would not decode' if n_failed else ''))
    gc.collect()
    return (studies, cache, mask)

## 4. DINOv2 ViT-Small Backbone & Slot Cross-Attention Architecture

### 1. Visual Feature Encoding
Each 3-slice group $\mathbf{X}_s \in \mathbb{R}^{3 \times H \times W}$ is passed through the pre-trained **DINOv2 ViT-Small** visual transformer backbone ($d_{hidden} = 384$). Features are pooled into study-level slot tokens using CLS token and patch mean pooling:

$$\mathbf{h}_s = \left[ \mathbf{z}_{\text{CLS}}, \frac{1}{N_{patch}} \sum_{i=1}^{N_{patch}} \mathbf{z}_{patch, i} \right] \in \mathbb{R}^{2 \times d_{hidden}}$$

### 2. Slot Cross-Attention with Anatomical Priors & Presence Penalty
The multi-target slot attention head queries the acquired slot representations:

$$\mathbf{A}_{o, s} = \frac{\mathbf{h}_s \mathbf{W}_p \cdot \mathbf{Q}_o}{\sqrt{d_{proj}}} + \mathbf{P}_{prior}(o, s)$$

$$\mathbf{A}_{masked}(o, s) = \text{masked\_fill}(\mathbf{A}_{o, s}, \text{Mask}_s < 0.5, -10^4) \longrightarrow \text{Softmax}_s(\mathbf{A}_{masked})$$

$$\hat{y}_o = \sigma\left( \mathbf{W}_{out, o} \sum_{s=1}^S \mathbf{A}_{masked}(o, s) \mathbf{h}_s + b_o \right)$$

The $-10^4$ penalty strictly suppresses unacquired slots ($\text{Mask}_s = 0$), preventing noise from corrupting predictions.


### SlotHead：ターゲット別クエリによるスロット・アテンション

**何をしているか**: 6つのスロット表現から12個の所見スコアを出すヘッドです。

- `slot_emb`（6×hidden）… 「これは矢状断・脂肪抑制ありのスロット」という**スロットの身分証**
- `query`（12×hidden）… 所見ごとの**問い合わせベクトル**
- `p_`（12×6の事前分布テーブル）… 「ACLを見るなら主に矢状断」という**解剖学的な事前知識**を
  アテンションのスコアに**バイアスとして直接加算**する

**なぜそうするのか**: 12の所見は、それぞれ**見るべき撮像面が違います**。

- ACL（前十字靭帯）→ 矢状断
- MCL（内側側副靭帯）→ 冠状断
- 膝蓋大腿の変形性関節症 → 横断

全所見が6スロットを平等に見る設計だと、モデルはこの対応関係を
**データから学び直さねばならず**、学習データが限られるほど不利になります。
`SLOT_PRIOR_TABLE` としてアテンションのバイアスに焼き込めば、
**放射線科医が知っていることを最初から知った状態**で学習が始まります。

重要なのは、これが**ハードな制約ではなくバイアス**である点です。
アテンションスコアに足されるだけなので、データが強く反対すれば
モデルは事前分布を上書きできます。「知識を与えるが、縛りはしない」設計です。

> 用語: **クロスアテンション** = 一方（ここでは所見のクエリ）が
> 他方（スロット表現）に問い合わせて、必要な情報だけを重み付きで取り出す仕組み。

In [ ]:
class SlotHead(nn.Module):
    def __init__(self, dim, n_slot=6, n_out=len(TARGETS), hidden=256, p=0.2, prior=False):
        super().__init__()
        self.dim = dim
        self.n_slot = n_slot
        self.n_out = n_out
        self.hidden = hidden
        self.prior = prior
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden
        p_ = torch.zeros(n_out, n_slot)
        if prior and n_slot == len(SLOTS) and (n_out == len(TARGETS)):
            for t, slots in SLOT_PRIOR_TABLE.items():
                if t in TARGETS:
                    p_[TARGETS.index(t), list(slots)] = SLOT_PRIOR_STRENGTH
        self.prior = prior
        if prior:
            self.register_buffer('slot_prior', p_)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        att = torch.einsum('bsh,oh->bos', h, self.query) / self.hidden ** 0.5
        if self.prior:
            att = att + self.slot_prior.unsqueeze(0)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -10000.0).softmax(-1)
        ctx = self.drop(torch.einsum('bos,bsh->boh', att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias

### モデル本体：バックボーンとヘッドの結線

**何をしているか**: `(B, S, ...)` の形で入ってくるスロット画像を `(B*S, ...)` に潰して
バックボーンへ流し、正規化（ImageNetの平均・標準偏差）とリサイズを行い、
特徴を `SlotHead` に渡します。

**なぜそうするのか**:

- **`register_buffer` で平均・標準偏差を持つ**: これらは学習しないパラメータですが、
  **`model.to(device)` で一緒にGPUへ移動してほしい**値です。
  `nn.Parameter` にすると誤って学習対象になり、
  ただの属性にすると `.to(device)` で移動せずデバイス不一致エラーになります。
  `register_buffer` はその中間で、こういう定数を持つための正しい場所です。
- **`B*S` に潰す理由**: バックボーンは1枚ずつ処理する2Dモデルなので、
  スロット次元をバッチ次元に畳んで**一気に流す**のが最も効率的です。
  ヘッドに渡す直前に `(B, S, dim)` へ戻します。
- **`div_(255.0)`（末尾のアンダースコア）**: in-place演算です。
  巨大なテンソルの複製を避けてメモリを節約しています。

補間（`F.interpolate`）が入っているのは、**キャッシュした解像度と
モデルが期待する解像度が違う場合**に対応するためです。
キャッシュを作り直さずに複数の解像度で推論できる（＝解像度もアンサンブルの軸にできる）。

In [ ]:
class Model(nn.Module):

    def __init__(self, backbone, dim, pool='cls_mean', prior=False):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.head = SlotHead(dim * POOL_PARTS[pool], N_SLOT, len(TARGETS), prior=prior)
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask, img_size=None):
        B, S = imgs.shape[:2]
        x = imgs.reshape(B * S, *imgs.shape[2:]).float().div_(255.0)
        if img_size is not None and img_size != x.shape[-1]:
            x = F.interpolate(x, size=(img_size, img_size), mode='bilinear', align_corners=False)
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state
        patch = out[:, 1:]
        parts = [out[:, 0], patch.mean(1)]
        if self.pool == 'cls_mean_focal':
            k = max(1, patch.shape[1] // 8)
            parts.append(patch.topk(k, dim=1).values.mean(1))
        feat = torch.cat(parts, dim=1).reshape(B, S, -1)
        return self.head(feat, mask)

### 部分的ファインチューニング：バックボーンの最後の数ブロックだけ解凍する

**何をしているか**: HuggingFace の `AutoModel` で DINOv2 を読み込み、
**全パラメータを一度 `requires_grad = False` で凍結**してから、
最後の `unfreeze_last` 個の Transformer ブロックと最終 LayerNorm だけを解凍します。
学習可能パラメータ数をログに出しています。

**なぜそうするのか**:

- **過学習の抑制**: ViT-Small でも2,000万パラメータ超あります。
  数千件規模の医用データセット全体を微調整すると、**確実に過学習します**。
- **計算資源の節約**: 凍結した層は勾配計算も optimizer state も不要なので、
  メモリと時間を大幅に節約できます。同じGPU予算で**より多くのメンバーを学習**できます。
- **なぜ「最後の数ブロック」なのか**: Transformer は
  浅い層ほど汎用的な特徴（エッジ、テクスチャ）、深い層ほどタスク特有の特徴を持ちます。
  医用画像は自然画像と見た目が違いますが、**低次の視覚特徴は共通**です。
  変える必要があるのは深い層だけ、という判断です。
- **LayerNorm を必ず解凍する理由**: 入力分布が自然画像からMRIへ大きく変わるため、
  正規化の統計だけは適応させないと、凍結層の出力スケールがずれたままになります。

> 用語: **DINOv2** = ラベルなし画像で自己教師あり学習された視覚基盤モデル。
> 分類ラベルに引っ張られていないぶん、医用画像のような
> **ImageNetのクラス体系から遠いドメイン**への転移に強いことが知られています。

In [ ]:
def build_model(unfreeze_last, source=None, variant='small', pool='cls_mean', prior=False):
    from transformers import AutoModel
    p = source if source is not None else find_dinov2(variant)
    if p is None:
        raise FileNotFoundError('DINOv2 weights not attached')
    bb = AutoModel.from_pretrained(str(p))
    n_layer = len(bb.encoder.layer)
    for prm in bb.parameters():
        prm.requires_grad = False
    for blk in bb.encoder.layer[max(0, n_layer - unfreeze_last):]:
        for prm in blk.parameters():
            prm.requires_grad = True
    for prm in bb.layernorm.parameters():
        prm.requires_grad = True
    dim = bb.config.hidden_size
    trainable = sum((p.numel() for p in bb.parameters() if p.requires_grad))
    log(f'backbone: {n_layer} blocks, last {unfreeze_last} trainable ({trainable / 1000000.0:.1f}M params), feature dim {dim * POOL_PARTS[pool]}')
    return Model(bb, dim, pool=pool, prior=prior)

## 5. Multi-Window Spatial TTA & Stage 1 Execution (Parent E2 0.899)

### 1. Multi-Window Spatial Test-Time Augmentation (TTA)
For each series holding $N_{slice} = 12$ cached slices, the model evaluates 10 overlapping sliding windows of $G=3$ consecutive slices:

$$\text{starts} = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]$$

Target-specific extreme-value pooling aggregates predictions across windows:
- **`max` Pooling**: Applied to localized focal lesions (**Fracture**, **Contusion**, **Medial Meniscus**, **Lateral Meniscus**, **Baker's Cyst**).
- **`top2` Pooling**: Average of top 2 responses for ligament structures (**ACL**, **MCL**).
- **`mean` Pooling**: Arithmetic mean across all 10 windows for diffuse degeneration (**Medial OA**, **Lateral OA**, **PF OA**, **Effusion**, **Synovitis**).

### 2. 24-Member Percentile Rank Fusion
Aggregates predictions across the 20 manifest members and 4 legacy bundle members into a calibrated rank consensus.


### モデルの指紋照合とマルチウィンドウTTA、そしてデュアルGPUの作業キュー

**何をしているか**: この長いセルには、この解法で最も実務的な工夫が詰まっています。

- **`fingerprint` / `check_fingerprint`** … 固定シードの乱数画像をモデルに通し、
  出力が**期待値と 0.002 以内で一致するか**を検証する
- **`window_starts` / `apply_target_window_pool`** … 12スライスから3枚ずつの窓を10個切り出し、
  **ターゲット別のプーリング戦略**（max / top2 / mean）で集約する
- **`bank` / `pop_next` / `worker`** … 2枚のGPUで作業キューを回し、
  **途中結果を随時 `submission.csv` に書き出す**
- **`combine_public_members_by_fold` / `blend_legacy_frontier_and_soft`** … メンバーの合成

**なぜそうするのか**:

**指紋照合について**。24メンバーのアンサンブルでは、
「重みファイルAを読んだつもりが実はB」「アーキテクチャの引数がわずかに違う」といった
**静かな取り違え**が起こり得ます。取り違えても推論は普通に完走し、
スコアだけが少し下がる——最悪の失敗の仕方です。
そこで、**決まった乱数入力に対する出力**を指紋として記録しておき、
読み込み直後に照合します。合わなければ止める。
浮動小数の非決定性を考慮して許容誤差 0.002 を置いているのが現実的です。

**ターゲット別プーリングについて**。上のヘッダでも触れたとおり、
これが**評価指標に最も直接効いている設計**です。
骨折のような1〜2枚のスライスにしか現れない所見を10窓平均すると、
信号が 1/10 に薄まってしまいます。`max` なら最も強く反応した窓が生き残る。
逆にびまん性の変化に `max` を使うと、**ノイズの最大値を拾う**だけになります。
「どう集約するか」は、モデル構造と同じくらいスコアを動かします。

**デュアルGPU作業キューと途中保存について**。Code Competition には実行時間制限があり、
24メンバー全部を評価しきれずにタイムアウトする可能性があります。
そこで、メンバーを1つ処理するたびに**その時点での合成結果を提出ファイルに書き出す**。
タイムアウトしても、**そこまでのアンサンブルは提出される**わけです。
「全部終わらないと0点」と「途中まででも部分点」の差は非常に大きい。
2枚のGPUに静的に半分ずつ割り当てるのではなく、
**空いたほうが次を取る**キュー方式にしているのも、メンバーごとの処理時間差を吸収するためです。

In [ ]:
FINGERPRINT_TOL = 0.002

def fingerprint(model, dev, img_size, n_slot=None, group=None, seed=None):
    n_slot = N_SLOT if n_slot is None else n_slot
    group = GROUP if group is None else group
    seed = SEED if seed is None else seed
    g = torch.Generator().manual_seed(seed)
    imgs = torch.randint(0, 256, (2, n_slot, group, img_size, img_size), generator=g, dtype=torch.uint8).to(dev)
    mask = torch.ones(2, n_slot, device=dev)
    mask[1, -1] = 0.0
    was_training = model.training
    model.eval()
    with torch.no_grad():
        out = model(imgs, mask, img_size).float().cpu().numpy()
    if was_training:
        model.train()
    return out

def check_fingerprint(model, dev, img_size, expected, tol=FINGERPRINT_TOL, tag=''):
    got = fingerprint(model, dev, img_size)
    exp = np.asarray(expected, np.float32)
    if got.shape != exp.shape:
        raise WeightsError(f'{tag}fingerprint shape {got.shape} != stored {exp.shape}: the architecture is not the one these weights were fitted to')
    d = float(np.abs(got - exp).max())
    if d > tol:
        raise WeightsError(f'{tag}fingerprint differs by {d:.4g} (tolerance {tol:g}). The weights load but do not compute what they computed when fitted - preprocessing, resolution or architecture has moved between the two runs.')
    log(f'{tag}fingerprint matches within {d:.2g}')
    return d

class WeightsError(RuntimeError):
    pass

def find_weights(name='manifest.json'):
    import json
    base = Path('/kaggle/input')
    if not base.is_dir():
        return None
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
        if name not in files:
            continue
        try:
            man = json.loads((Path(root) / name).read_text())
        except (OSError, ValueError):
            continue
        if isinstance(man.get('members'), list) and man['members']:
            missing = [m['file'] for m in man['members'] if not (Path(root) / m['file']).is_file()]
            if missing:
                raise WeightsError(f"{root} holds a manifest listing {len(man['members'])} members but {len(missing)} of their files are absent (first {missing[0]!r})")
            return Path(root)
    return None
TTA_OVERLAP = True
TTA_POOL = 'prob'
PUBLIC_FRONTIER_TARGET_POOL = {'Fracture': 'max', 'Contusion': 'max', 'Medial Meniscus': 'max', 'Lateral Meniscus': 'max', 'ACL': 'top2', 'MCL': 'top2', "Baker's": 'max'}
TTA_TARGET_POOL = {**PUBLIC_FRONTIER_TARGET_POOL, 'Synovitis': 'original_mean'}
# No-extra-pass diversity branch: smooth focal pooling is evaluated from
# the same no-jitter public-member windows already used by the parent.
LEGACY_FOLD_SOFTPOOL_BETA = {
    'ACL': 6.0, 'MCL': 6.0,
    'Medial Meniscus': 8.0, 'Lateral Meniscus': 8.0,
    "Baker's": 8.0, 'Contusion': 8.0, 'Fracture': 10.0,
}
LEGACY_FOLD_SOFTPOOL_ALPHA = {
    'ACL': 0.20, 'MCL': 0.20,
    'Medial Meniscus': 0.25, 'Lateral Meniscus': 0.25,
    "Baker's": 0.20, 'Contusion': 0.20, 'Fracture': 0.15,
}
LEGACY_MEMBER_WEIGHT_BY_TARGET = {'Lateral Meniscus': 15.0, 'Medial OA': 2.5, 'Lateral OA': 15.0, 'Contusion': 5.0}

def window_starts(n_slice, group, overlap=None):
    overlap = TTA_OVERLAP if overlap is None else overlap
    if overlap and n_slice >= group:
        return list(range(n_slice - group + 1))
    return [g * group for g in range(max(n_slice // group, 1))]

def apply_target_window_pool(values, probs, logits, original_probs, mapping, target_idx):
    for target, mode in mapping.items():
        j = target_idx[target]
        if mode == 'max':
            values[:, j] = probs[:, :, j].max(0).values
        elif mode == 'mean':
            values[:, j] = probs[:, :, j].mean(0)
        elif mode == 'logit_mean':
            values[:, j] = torch.sigmoid(logits[:, :, j].mean(0))
        elif mode == 'original_mean':
            values[:, j] = original_probs[:, :, j].mean(0)
        elif mode in ('top2', 'top3'):
            k = min(int(mode[3:]), probs.shape[0])
            values[:, j] = probs[:, :, j].topk(k, dim=0).values.mean(0)
        else:
            raise ValueError(f'unknown TTA pooling mode for {target}: {mode}')
    return values

def legacy_fold_soft_window_pool(original_probs, target_idx):
    values = original_probs.mean(0).clone()
    for target, beta in LEGACY_FOLD_SOFTPOOL_BETA.items():
        j = target_idx[target]
        x = original_probs[:, :, j]
        weight = torch.softmax(float(beta) * x, dim=0)
        values[:, j] = (weight * x).sum(0)
    return values

@torch.no_grad()
def predict_member(model, cache, mask, idx, dev, img_size, group=None, pool=None, starts=None, jitter=False, jitter_seed=SEED, return_public_frontier=False):
    group = GROUP if group is None else group
    pool = TTA_POOL if pool is None else pool
    starts = window_starts(cache.shape[2], group) if starts is None else list(starts)
    if not starts:
        raise ValueError('predict_member was given no windows to average over')
    target_idx = {t: j for j, t in enumerate(TARGETS)}
    unknown = (set(TTA_TARGET_POOL) | set(PUBLIC_FRONTIER_TARGET_POOL)) - set(target_idx)
    if unknown:
        raise ValueError(f'unknown target(s) in TTA_TARGET_POOL: {unknown}')
    jitter_gen = torch.Generator(device=dev)
    jitter_gen.manual_seed(int(jitter_seed) % (2 ** 63 - 1))
    model.eval()
    out, public_frontier_out, public_soft_out = ([], [], [])
    for b in range(0, len(idx), EVAL_BATCH):
        sel = idx[b:b + EVAL_BATCH]
        m = torch.from_numpy(mask[sel]).to(dev)
        win_probs, win_logits, win_original_probs = ([], [], [])
        for st in starts:
            rows = torch.from_numpy(np.ascontiguousarray(cache[sel, :, st:st + group])).to(dev)
            views = [rows] + ([augment(rows, generator=jitter_gen)] if jitter else [])
            view_probs, view_logits = ([], [])
            for view in views:
                with torch.autocast('cuda', enabled=dev.type == 'cuda'):
                    z = model(view, m, img_size).float()
                view_logits.append(z)
                view_probs.append(torch.sigmoid(z))
            win_logits.append(torch.stack(view_logits).mean(0))
            win_probs.append(torch.stack(view_probs).mean(0))
            win_original_probs.append(view_probs[0])
        probs = torch.stack(win_probs)
        logits = torch.stack(win_logits)
        original_probs = torch.stack(win_original_probs)
        v = torch.sigmoid(logits.mean(0)) if pool == 'logit' else probs.mean(0)
        v = apply_target_window_pool(v, probs, logits, original_probs, TTA_TARGET_POOL, target_idx)
        out.append(v.cpu().numpy())
        if return_public_frontier:
            public_v = apply_target_window_pool(original_probs.mean(0), original_probs, logits, original_probs, PUBLIC_FRONTIER_TARGET_POOL, target_idx)
            public_frontier_out.append(public_v.cpu().numpy())
            public_soft = legacy_fold_soft_window_pool(original_probs, target_idx)
            public_soft_out.append(public_soft.cpu().numpy())
    primary = np.concatenate(out) if out else np.zeros((0, len(TARGETS)), np.float32)
    if not return_public_frontier:
        return primary
    public_frontier = np.concatenate(public_frontier_out) if public_frontier_out else np.zeros((0, len(TARGETS)), np.float32)
    public_soft = np.concatenate(public_soft_out) if public_soft_out else np.zeros((0, len(TARGETS)), np.float32)
    return (primary, public_frontier, public_soft)
BUILD_LOCK = threading.Lock()
STATE_LOCK = threading.Lock()
LEGACY_BUNDLE_FILE = 'rsna_20260807_v1.pt'
LEGACY_WEIGHT = 0.5

def find_legacy_bundle():
    base = Path('/kaggle/input')
    if not base.is_dir():
        return None
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
        if LEGACY_BUNDLE_FILE in files:
            return Path(root) / LEGACY_BUNDLE_FILE
    return None

def legacy_group_members():
    p = find_legacy_bundle()
    if p is None:
        log('no legacy bundle attached; blending skipped')
        return {}
    try:
        b = torch.load(p, map_location='cpu', weights_only=False)
        folds = b.get('fold_states') or []
        b_slots = [tuple(s)[0] for s in b.get('slots', SLOTS)]
        if list(b.get('targets', TARGETS)) != TARGETS or b_slots != [s[0] for s in SLOTS]:
            log(f'legacy bundle {p.name}: target/slot contract differs; blending skipped')
            return {}
        gr, n_gr = (int(b.get('group', 3)), int(b.get('n_group', 3)))
        variant = str(b.get('model_variant', 'dinov2-small')).split('-')[-1]
        key = json.dumps({'img': int(b.get('img', 224)), 'group': gr, 'slices': gr * n_gr, 'crop_mm': 160.0, 'band': [0.2, 0.8], 'rules': RULES_LEGACY, 'slots': [s[0] for s in SLOTS]}, sort_keys=True)
        ms = [{'id': f"legacy-f{f.get('fold', k)}", 'fold': f.get('fold', k), 'state': f['state_dict'], 'holdout': None, 'weight': LEGACY_WEIGHT, 'target_weight': [LEGACY_MEMBER_WEIGHT_BY_TARGET.get(t, 0.0) for t in TARGETS], 'pixel_group': key, 'config': {'unfreeze_last': 6, 'variant': 'base' if variant == 'base' else 'small', 'pool': 'cls_mean_focal', 'prior': True}} for k, f in enumerate(folds)]
        if ms:
            active = sorted(set(LEGACY_MEMBER_WEIGHT_BY_TARGET.values()))
            log(f'legacy bundle {p.name}: {len(ms)} fold(s) join with target-specific per-member weights {active}')
        return {key: ms} if ms else {}
    except Exception as exc:
        log(f'legacy bundle unusable ({type(exc).__name__}: {exc}); blending skipped')
        return {}

def _run_member(path, m, dev, Cte, Mte, idx, starts, jitter):
    t0 = time.time()
    with BUILD_LOCK:
        if 'state' in m:
            state, fp = (m['state'], None)
        else:
            ck = torch.load(Path(path) / m['file'], map_location='cpu', weights_only=False)
            state, fp = (ck['model'], ck.get('fingerprint'))
        model = build_model(int(m['config']['unfreeze_last']), variant=m['config']['variant'], pool=m['config'].get('pool', 'cls_mean'), prior=bool(m['config'].get('prior', False))).to(dev)
        model.load_state_dict(state)
        if fp is not None:
            check_fingerprint(model, dev, IMG, fp, tag=f"{m['id']}: ")
        else:
            log(f"  {m['id']}: no stored fingerprint (legacy bundle) -- accepted at reduced weight")
    t_ready = time.time()
    jitter_seed = SEED + int(hashlib.sha256(str(m['id']).encode()).hexdigest()[:8], 16)
    public_member = 'state' not in m
    predicted = predict_member(model, Cte, Mte, idx, dev, IMG, starts=starts, jitter=jitter, jitter_seed=jitter_seed, return_public_frontier=public_member)
    if public_member:
        p, public_p, public_soft = predicted
    else:
        p, public_p, public_soft = (predicted, None, None)
    t_done = time.time()
    del model, state
    gc.collect()
    if dev.type == 'cuda':
        with torch.cuda.device(dev):
            torch.cuda.empty_cache()
    passes = len(starts) * (2 if jitter else 1)
    return (p, public_p, public_soft, (t_ready - t0, (t_done - t_ready) / max(passes, 1)))

def _combine(per_member):
    all_ids = sorted({s for m in per_member for s in m['ids']})
    pos = {s: i for i, s in enumerate(all_ids)}
    acc = np.zeros((len(all_ids), len(TARGETS)), np.float64)
    tot = np.zeros(len(TARGETS), np.float64)
    for m in per_member:
        target_weight = m.get('target_weight')
        w = np.asarray(target_weight if target_weight is not None else [float(m.get('weight', 1.0))] * len(TARGETS), dtype=np.float64)
        if w.shape != (len(TARGETS),) or np.any(w < 0):
            raise ValueError(f"invalid target weights for {m.get('id')}: {w}")
        r = pd.DataFrame(m['pred']).rank(pct=True).to_numpy()
        acc[[pos[s] for s in m['ids']]] += r * w[None, :]
        tot += w
    if np.any(tot <= 0):
        raise ValueError(f'at least one target has no ensemble vote: {tot}')
    return (all_ids, acc / tot[None, :])

def combine_public_members_by_fold(per_member, pred_key='pred'):
    # Raw-average the four members within each fold, rank each fold,
    # then give all five folds equal weight.
    all_ids = sorted({study for member in per_member for study in member['ids']})
    position = {study: i for i, study in enumerate(all_ids)}
    groups = {}
    for i, member in enumerate(per_member):
        fold = member.get('fold')
        key = f'fold_{fold}' if fold is not None else f'member_{i}'
        groups.setdefault(key, []).append(member)
    fold_ranks, diagnostics = ([], [])
    for key, members_in_fold in sorted(groups.items()):
        matrices = []
        for member in members_in_fold:
            values = np.full((len(all_ids), len(TARGETS)), np.nan, np.float64)
            values[[position[study] for study in member['ids']]] = np.asarray(member[pred_key], np.float64)
            if np.isnan(values).any():
                raise WeightsError(f"{member.get('id')}: incomplete {pred_key} coverage")
            matrices.append(values)
        raw_fold_mean = np.mean(matrices, axis=0)
        fold_ranks.append(pd.DataFrame(raw_fold_mean).rank(method='average', pct=True).to_numpy(np.float64))
        diagnostics.append({'ensemble_group': key, 'members': len(members_in_fold)})
    if len(fold_ranks) != 5:
        raise WeightsError(f'legacy branch requires five folds, found {len(fold_ranks)}')
    return all_ids, np.mean(fold_ranks, axis=0), pd.DataFrame(diagnostics)

def blend_legacy_frontier_and_soft(frontier_rank, soft_rank):
    output = np.asarray(frontier_rank, np.float64).copy()
    for j, target in enumerate(TARGETS):
        alpha = float(LEGACY_FOLD_SOFTPOOL_ALPHA.get(target, 0.0))
        if alpha:
            output[:, j] = (1.0 - alpha) * frontier_rank[:, j] + alpha * soft_rank[:, j]
    return output

def infer_from_package(path, dev=None):
    man = json.loads((Path(path) / 'manifest.json').read_text())
    members = man['members']
    log(f'weights package: {len(members)} member(s) from {path}; {len(DEVS)} device(s)')
    test_df = pd.read_csv(ROOT / 'test.csv')
    test_series = pd.read_csv(ROOT / 'test_series.csv')
    plane_map = dict(zip(test_series['SeriesInstanceUID'], test_series['Anatomical_Plane']))
    hte = annotate(walk('test_series'))
    log(f'test header pass: {len(hte)} series')
    groups = {}
    for m in members:
        groups.setdefault(m['pixel_group'], []).append(m)
    groups.update(legacy_group_members())
    per_member, public_frontier_members = ([], [])
    est = {'fixed': None, 'win': None}

    def bank(m, ids, pred, starts, jitter, public_pred=None, public_soft=None):
        if float(np.std(pred)) < 1e-09:
            log(f"  {m['id']}: degenerate predictions; not banked")
            return
        with STATE_LOCK:
            per_member.append({'id': m['id'], 'fold': m.get('fold'), 'ids': ids, 'pred': pred, 'weight': m.get('weight', 1.0), 'target_weight': m.get('target_weight'), 'holdout': m.get('holdout')})
            if public_pred is not None and len(starts) == len(starts_full):
                if float(np.std(public_pred)) < 1e-09:
                    raise WeightsError(f"{m['id']}: degenerate public-frontier prediction")
                public_frontier_members.append({'id': m['id'], 'fold': m.get('fold'), 'ids': ids, 'pred': public_pred, 'soft_pred': public_soft})
            elif public_pred is not None:
                log(f"  {m['id']}: public-frontier vote omitted because only {len(starts)} / {len(starts_full)} windows completed")
            all_ids, acc = _combine(per_member)
            write_submission(acc, all_ids, test_df, 'submission.csv')
            log(f"  banked {m['id']} fold {m.get('fold', '?')} ({len(starts)} window(s){(', jitter' if jitter else '')}); submission.csv = weighted rank mean of {len(per_member)} member(s)")
    for gi, (key, gm) in enumerate(groups.items(), 1):
        cfg = json.loads(key)
        adopt_config_globals(cfg)
        log(f"decode group {gi}/{len(groups)}: {cfg['img']}px x {cfg['slices']} slices, crop {cfg['crop_mm']} mm -> {len(gm)} member(s)")
        st_te, Cte, Mte = build_cache(pick_slots(hte, plane_map), plane_map, lat_of(hte, 'test '), f'test g{gi}')
        idx = np.arange(len(st_te))
        starts_full = window_starts(Cte.shape[2], GROUP)
        pending = sorted(gm, key=lambda m: -(m.get('holdout') or 0))
        left_after = sum((len(g) for j, (_, g) in enumerate(groups.items(), 1) if j > gi))

        def pop_next():
            with STATE_LOCK:
                if not pending:
                    return (None, None, False)
                left = TIME_BUDGET - (time.time() - T0)
                remaining = len(pending) + left_after
                slots_left = -(-remaining // len(DEVS))
                starts, jit = (starts_full, False)
                if est['fixed'] is not None and est['win'] is not None:
                    afford = max(left * 0.9, 0.0)
                    room = afford / max(slots_left, 1)
                    if est['fixed'] + est['win'] > room:
                        log(f'  {left / 60:.0f} min left: surrendering {len(pending)} member(s); not one more fits')
                        pending.clear()
                        return (None, None, False)
                    jit = est['fixed'] + 2 * len(starts_full) * est['win'] <= room * 0.6
                    per_win = est['win'] * (2 if jit else 1)
                    n_win = int((room - est['fixed']) / per_win) if per_win > 0 else len(starts_full)
                    n_win = max(1, min(len(starts_full), n_win))
                    if n_win < len(starts_full):
                        mid = (len(starts_full) - n_win) // 2
                        starts = starts_full[mid:mid + n_win]
                return (pending.pop(0), starts, jit)

        def worker(dev):
            others = [d for d in DEVS if d is not dev]
            while True:
                m, starts, jit = pop_next()
                if m is None:
                    return
                for attempt, d in enumerate([dev] + others[:1]):
                    try:
                        p, public_p, public_soft, (fs, ws) = _run_member(path, m, d, Cte, Mte, idx, starts, jit)
                        with STATE_LOCK:
                            est['fixed'], est['win'] = (fs, ws)
                        bank(m, st_te, p, starts, jit, public_p, public_soft)
                        break
                    except Exception as exc:
                        log(f"  MEMBER {m['id']} failed on {d} ({type(exc).__name__}: {exc}); " + ('retrying on peer device' if attempt == 0 and others else 'dropped -- costs one vote, not the run'))
                        if d.type == 'cuda':
                            with torch.cuda.device(d):
                                torch.cuda.empty_cache()
        threads = [threading.Thread(target=worker, args=(d,)) for d in DEVS]
        for t in threads:
            t.start()
        for t in threads:
            t.join()
        del Cte, Mte
        gc.collect()
    if not per_member:
        raise WeightsError('no member produced predictions; submission stays at 0.5')
    all_ids, acc = _combine(per_member)
    sub = write_submission(acc, all_ids, test_df, 'submission.csv')
    log(f'final submission.csv = weighted rank mean of {len(per_member)} member(s); {sub.shape}; nulls {int(sub[TARGETS].isna().sum().sum())}')
    if len(public_frontier_members) == len(members):
        frontier_ids, frontier_acc = _combine(public_frontier_members)
        frontier_sub = write_submission(frontier_acc, frontier_ids, test_df, 'submission_public_0899.csv')
        log(f'submission_public_0899.csv = exact no-jitter public-frontier rank mean of {len(public_frontier_members)} member(s); {frontier_sub.shape}; nulls {int(frontier_sub[TARGETS].isna().sum().sum())}')
        fold_ids, fold_frontier, fold_diagnostics = combine_public_members_by_fold(public_frontier_members, 'pred')
        soft_ids, fold_soft, _ = combine_public_members_by_fold(public_frontier_members, 'soft_pred')
        if fold_ids != soft_ids:
            raise WeightsError('legacy hard/soft study order mismatch')
        legacy_prediction = blend_legacy_frontier_and_soft(fold_frontier, fold_soft)
        legacy_sub = write_submission(legacy_prediction, fold_ids, test_df, 'submission_legacy_fold_blend.csv')
        fold_diagnostics.to_csv('legacy_fold_diagnostics.csv', index=False)
        log(f'legacy DINO aggregation written from five folds; {legacy_sub.shape}')
    else:
        log(f'public-frontier fallback not emitted: {len(public_frontier_members)} / {len(members)} required public members completed')
    return sub

def adopt_config_globals(cfg):
    global IMG, CACHE_IMG, GROUP, CACHE_SLICES, N_GROUP, CROP_MM, SLICE_BAND, RULES
    CACHE_IMG = IMG = int(cfg['img'])
    GROUP = int(cfg['group'])
    CACHE_SLICES = int(cfg['slices'])
    N_GROUP = max(CACHE_SLICES // GROUP, 1)
    CROP_MM = float(cfg['crop_mm'])
    SLICE_BAND = tuple((float(x) for x in cfg['band']))
    rules = cfg.get('rules') or RULES_NATIVE
    unknown = {k: v for k, v in rules.items() if k not in RULES_NATIVE or v not in (RULES_NATIVE[k], RULES_LEGACY[k])}
    if unknown:
        raise WeightsError(f'the members record pixel rules this pipeline cannot reproduce: {unknown}')
    RULES = {**RULES_NATIVE, **rules}
    if [s[0] for s in SLOTS] != list(cfg['slots']):
        raise WeightsError(f"the members were fitted on slots {cfg['slots']} and this pipeline defines {[s[0] for s in SLOTS]}; a weight would be read against the wrong slot")

### 幾何変換によるデータ拡張と評価指標の実装

**何をしているか**:

- `take_group(cache_rows, g)` … キャッシュから g 番目のスライス群を切り出す
- `augment(imgs, generator)` … 回転・拡大縮小・平行移動を**アフィン行列でまとめて適用**
- `predict(...)` … 推論の本体
- `macro_auc(...)` … 12ターゲットのAUCを個別に計算して平均する

**なぜそうするのか**:

**アフィン変換の書き方**に注目してください。回転角・スケール・平行移動から
2×3の行列 `theta` をバッチ分まとめて構築し、`F.affine_grid` + `F.grid_sample` で
**全画像を1回のGPU演算で変換**しています。
画像を1枚ずつ OpenCV で回す実装に比べて桁違いに速く、
**推論時のTTAでも同じコードが使い回せます**。

`torch.Generator` を明示的に渡しているのは**再現性のため**です。
グローバルな乱数状態に依存すると、実行順序が変わるだけで結果が変わります。

**`macro_auc` を自前で持つ理由**: 提出前に手元で
「今のアンサンブルが何点くらいか」を測れないと、
**変更を採用すべきか判断できません**。
公開LBは1日の提出回数に制限があり、しかも public/private の分割があるので、
**手元の指標を公式と同じ式で実装しておく**のは必須の投資です。

In [ ]:
def take_group(cache_rows, g):
    return cache_rows[:, :, g * GROUP:(g + 1) * GROUP]

def augment(imgs, generator=None):
    lead = imgs.shape[:-3]
    x = imgs.reshape(-1, *imgs.shape[-3:]).float()
    n, dev = (x.shape[0], x.device)
    rot = (torch.rand(n, device=dev, generator=generator) - 0.5) * 2 * (AUG_ROT_DEG * np.pi / 180)
    sc = 1.0 + torch.rand(n, device=dev, generator=generator) * AUG_SCALE
    tx = (torch.rand(n, device=dev, generator=generator) - 0.5) * 2 * AUG_SHIFT
    ty = (torch.rand(n, device=dev, generator=generator) - 0.5) * 2 * AUG_SHIFT
    cos, sin = (torch.cos(rot) / sc, torch.sin(rot) / sc)
    theta = torch.zeros(n, 2, 3, device=dev, dtype=torch.float32)
    theta[:, 0, 0], theta[:, 0, 1], theta[:, 0, 2] = (cos, -sin, tx)
    theta[:, 1, 0], theta[:, 1, 1], theta[:, 1, 2] = (sin, cos, ty)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=False)
    scale = 1.0 + (torch.rand(n, 1, 1, 1, device=dev, generator=generator) - 0.5) * 2 * AUG_INTENSITY
    x = (x * scale).clamp(0, 255)
    return x.reshape(*lead, *x.shape[-3:]).to(imgs.dtype)

@torch.no_grad()
def predict(model, cache, mask, idx, dev, img_size=None):
    model.eval()
    out = []
    for b in range(0, len(idx), EVAL_BATCH):
        sel = idx[b:b + EVAL_BATCH]
        m = torch.from_numpy(mask[sel]).to(dev)
        acc = None
        for g in range(N_GROUP):
            rows = torch.from_numpy(np.ascontiguousarray(cache[sel, :, g * GROUP:(g + 1) * GROUP])).to(dev)
            with torch.autocast('cuda', enabled=dev.type == 'cuda'):
                z = model(rows, m, img_size).float()
            acc = z if acc is None else acc + z
        out.append(torch.sigmoid(acc / N_GROUP).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, len(TARGETS)), np.float32)

def macro_auc(y, p):
    from sklearn.metrics import roc_auc_score
    return float(np.nanmean([roc_auc_score(y[:, j], p[:, j]) if len(set(y[:, j])) > 1 else np.nan for j in range(y.shape[1])]))

### RTAHMIL：レポート由来ラベルで学習する、アテンション付きMIL

**何をしているか**: 2.5D の系列表現に対して、
**アテンション機構つきの Multiple Instance Learning（MIL）** モデルを構築・推論します。

- `TARGET_FAMILIES` / `GROUP_NAMES` … 12ターゲットを6つの**グループ**
  （靭帯・半月板・変形性関節症・炎症・骨・その他）に束ねる
- `RTAHMIL` … `stochastic_slot_mask` によるスロット単位のドロップアウトを持つモデル
- `_rt_load_series_25d` … 連続する3スライスを1つのRGB相当の入力にする「2.5D」化
- `_rt_blend_synovitis` … 滑膜炎だけ別扱いでブレンドする

**なぜそうするのか**:

- **なぜ MIL か**: ラベルは**検査（study）単位**に付いていますが、
  病変は**特定のスライスにしか写っていません**。
  「袋（bag）の中のどこかに陽性のインスタンスがある」という構造そのものが MIL の定義です。
  アテンション付き MIL は、**どのスライスが判断に効いたかの重みを学習**するので、
  単純な平均プーリングより素直に当てはまります。
- **なぜターゲットをグループ化するか**: 内側半月板と外側半月板は
  **見るべき場所も見え方もほぼ同じ**です。表現を共有させれば、
  低頻度のターゲットが高頻度の兄弟から学習信号を借りられます。
  マクロ平均AUCでは**低頻度ターゲットの改善が全体に等しく効く**ので、これは合理的です。
- **`stochastic_slot_mask`（スロット単位のドロップアウト）**: 学習時にランダムにスロットを落とします。
  推論時には**実際に欠けているスロットがある**（すべての施設が6種類全部を撮るわけではない）ので、
  欠損に頑健なモデルにしておく必要があります。
  **学習時に本番の欠損パターンを模擬する**、という発想です。
- **滑膜炎だけ別扱いにする理由**: 滑膜炎はレポートでの記載揺れが大きく、
  弱いラベルの質が他より低いターゲットです。
  **ラベル品質がターゲットごとに違う**なら、扱いも変えるべき、という判断でしょう。

In [ ]:
import math
import cv2
TARGET_FAMILIES = ['acl', 'mcl', 'medial_meniscus', 'lateral_meniscus', 'medial_oa', 'lateral_oa', 'pf_oa', 'effusion', 'synovitis', 'baker', 'contusion', 'fracture']
GROUP_NAMES = ['ligament', 'meniscus', 'oa', 'inflammation', 'bone', 'other']

def target_group_id(family):
    if family in {'acl', 'mcl'}:
        return 0
    if family in {'medial_meniscus', 'lateral_meniscus'}:
        return 1
    if family in {'medial_oa', 'lateral_oa', 'pf_oa'}:
        return 2
    if family in {'effusion', 'synovitis', 'baker'}:
        return 3
    if family in {'contusion', 'fracture'}:
        return 4
    return 5
TARGET_GROUP_IDS = torch.tensor([target_group_id(f) for f in TARGET_FAMILIES], dtype=torch.long)

class RTAHMIL(nn.Module):

    def __init__(self, in_dim, hidden_dim, n_targets, n_slots, n_slices, dropout, series_dropout):
        super().__init__()
        self.n_targets = n_targets
        self.n_slots = n_slots
        self.n_slices = n_slices
        self.series_dropout = float(series_dropout)
        self.input_proj = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout))
        self.slice_pos_emb = nn.Parameter(torch.randn(n_slices, hidden_dim) / math.sqrt(hidden_dim))
        slice_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=8, dim_feedforward=hidden_dim * 4, dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
        self.slice_encoder = nn.TransformerEncoder(slice_layer, num_layers=1)
        self.series_query = nn.Parameter(torch.randn(1, 1, hidden_dim) / math.sqrt(hidden_dim))
        self.series_pool = nn.MultiheadAttention(hidden_dim, num_heads=8, dropout=dropout, batch_first=True)
        self.slot_emb = nn.Embedding(n_slots, hidden_dim)
        self.plane_emb = nn.Embedding(3, hidden_dim)
        self.sequence_emb = nn.Embedding(2, hidden_dim)
        study_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=8, dim_feedforward=hidden_dim * 4, dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
        self.study_encoder = nn.TransformerEncoder(study_layer, num_layers=2)
        self.target_queries = nn.Parameter(torch.randn(n_targets, hidden_dim) / math.sqrt(hidden_dim))
        self.group_emb = nn.Embedding(len(GROUP_NAMES), hidden_dim)
        self.target_cross_attn = nn.MultiheadAttention(hidden_dim, num_heads=8, dropout=dropout, batch_first=True)
        self.target_fuse = nn.Sequential(nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(), nn.Dropout(dropout))
        self.target_heads = nn.ModuleList([nn.Sequential(nn.LayerNorm(hidden_dim), nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim // 2, 1)) for _ in range(n_targets)])
        slot_plane = [0, 0, 1, 1, 2, 2]
        slot_sequence = [0, 1, 0, 1, 0, 1]
        self.register_buffer('slot_plane_ids', torch.tensor(slot_plane, dtype=torch.long), persistent=False)
        self.register_buffer('slot_sequence_ids', torch.tensor(slot_sequence, dtype=torch.long), persistent=False)
        self.register_buffer('target_group_ids', TARGET_GROUP_IDS, persistent=False)

    def stochastic_slot_mask(self, mask):
        if not self.training or self.series_dropout <= 0:
            return mask
        keep = torch.rand(mask.shape, device=mask.device) > self.series_dropout
        new_mask = mask & keep
        all_missing = (~new_mask).all(dim=1)
        if all_missing.any():
            for row in torch.where(all_missing)[0]:
                valid = torch.where(mask[row])[0]
                if len(valid) > 0:
                    new_mask[row, valid[0]] = True
        return new_mask

    def forward(self, x, slot_mask):
        B, S, K, _ = x.shape
        z = self.input_proj(x)
        z = z + self.slice_pos_emb[None, None, :K, :]
        z = z.reshape(B * S, K, -1)
        z = self.slice_encoder(z)
        q = self.series_query.expand(B * S, -1, -1)
        series_token, _ = self.series_pool(q, z, z, need_weights=False)
        series_token = series_token[:, 0].reshape(B, S, -1)
        slot_ids = torch.arange(S, device=x.device)
        plane_ids = self.slot_plane_ids[:S]
        seq_ids = self.slot_sequence_ids[:S]
        series_token = series_token + self.slot_emb(slot_ids)[None, :, :] + 0.35 * self.plane_emb(plane_ids)[None, :, :] + 0.35 * self.sequence_emb(seq_ids)[None, :, :]
        effective_mask = self.stochastic_slot_mask(slot_mask)
        no_valid_slot = (~effective_mask).all(dim=1)
        if no_valid_slot.any():
            effective_mask = effective_mask.clone()
            series_token = series_token.clone()
            effective_mask[no_valid_slot, 0] = True
            series_token[no_valid_slot, 0] = 0.0
        series_token = self.study_encoder(series_token, src_key_padding_mask=~effective_mask)
        denom = effective_mask.sum(dim=1, keepdim=True).clamp_min(1).to(series_token.dtype)
        study_global = (series_token * effective_mask.unsqueeze(-1)).sum(dim=1) / denom
        target_q = self.target_queries + 0.25 * self.group_emb(self.target_group_ids)
        target_q = target_q.unsqueeze(0).expand(B, -1, -1)
        target_context, _ = self.target_cross_attn(target_q, series_token, series_token, key_padding_mask=~effective_mask, need_weights=False)
        global_expand = study_global.unsqueeze(1).expand(-1, self.n_targets, -1)
        fused = self.target_fuse(torch.cat([target_context, global_expand], dim=-1))
        logits = []
        for j, head in enumerate(self.target_heads):
            logits.append(head(fused[:, j]))
        return torch.cat(logits, dim=1)
'Runtime helpers embedded into the V26 Kaggle notebook.\n\nThe exact RTAHMIL class from the public report-teacher notebook is prepended by the\ncandidate builder. This file contains only hidden-test feature extraction, checkpoint\ninference, and the fail-safe Synovitis blend.\n'
RT_START_CUTOFF_S = 5.9 * 3600
RT_DEADLINE_S = 7.1 * 3600
RT_IMG_SIZE = 336
RT_TARGET_SPACING = 0.42
RT_SLICES = 7
RT_SYN_WEIGHT = 0.75
RT_SEEDS = (2026, 3407)

def _rt_find_checkpoint_dir():
    root = Path('/kaggle/input')
    required = [f'rta_final_seed{seed}_fold{fold}.pth' for seed in RT_SEEDS for fold in range(4)]
    for first in required[:1]:
        for hit in root.glob(f'*/{first}'):
            parent = hit.parent
            if all(((parent / name).is_file() for name in required)):
                return parent
    raise FileNotFoundError('the complete eight-checkpoint report-teacher package is absent')

def _rt_find_dino_base():
    direct = [Path('/kaggle/input/dinov2/pytorch/base/1'), Path('/kaggle/input/models/metaresearch/dinov2/pytorch/base/1')]
    for path in direct:
        if (path / 'config.json').is_file():
            return path
    for top in Path('/kaggle/input').iterdir():
        if not top.is_dir() or 'dino' not in top.name.lower():
            continue
        for config in top.glob('**/config.json'):
            try:
                if 'dinov2' in config.read_text(errors='ignore').lower():
                    model_type = json.loads(config.read_text()).get('model_type', '')
                    if model_type == 'dinov2' and 'base' in str(config.parent).lower():
                        return config.parent
            except Exception:
                continue
    raise FileNotFoundError('offline DINOv2-base model is absent')

def _rt_binary_flag(value):
    if pd.isna(value):
        return 0
    if isinstance(value, str):
        return int(value.strip().lower() in {'1', 'true', 'yes', 'y'})
    try:
        return int(float(value) > 0)
    except Exception:
        return 0

def _rt_plane_id(value):
    text = str(value).lower()
    if 'sag' in text:
        return 0
    if 'cor' in text:
        return 1
    if 'axi' in text or 'trans' in text or 'tra' == text.strip():
        return 2
    return 3

def _rt_assign_slots(series_df):
    x = series_df.copy()
    x['StudyInstanceUID'] = x['StudyInstanceUID'].astype(str)
    x['SeriesInstanceUID'] = x['SeriesInstanceUID'].astype(str)
    x['_plane_id'] = x['Anatomical_Plane'].map(_rt_plane_id)
    fluid = x['Fluid_Sensitive'].map(_rt_binary_flag)
    fat = x['Fat_Suppression'].map(_rt_binary_flag)
    x['_fluid_like'] = np.maximum(fluid.astype(int), fat.astype(int))
    slot_defs = ((0, 0), (0, 1), (1, 0), (1, 1), (2, 0), (2, 1))
    lookup = {}
    for study_uid, group in x.groupby('StudyInstanceUID', sort=False):
        group = group.sort_values(['SeriesInstanceUID']).copy()
        used = set()
        for slot_id, (plane, fluid_like) in enumerate(slot_defs):
            desired = group[(group['_plane_id'] == plane) & (group['_fluid_like'] == fluid_like) & ~group['SeriesInstanceUID'].isin(used)]
            if len(desired) == 0:
                desired = group[(group['_plane_id'] == plane) & ~group['SeriesInstanceUID'].isin(used)]
            if len(desired) == 0:
                continue
            series_uid = str(desired.iloc[0]['SeriesInstanceUID'])
            used.add(series_uid)
            lookup[str(study_uid), slot_id] = series_uid
    return lookup

def _rt_locate_series_dir(study_uid, series_uid):
    canonical = ROOT / 'test_series'
    candidates = (canonical / str(series_uid), canonical / str(study_uid) / str(series_uid), ROOT / 'test' / str(study_uid) / str(series_uid), ROOT / 'test_images' / str(study_uid) / str(series_uid), ROOT / 'test_dicom' / str(study_uid) / str(series_uid), ROOT / 'test_dicoms' / str(study_uid) / str(series_uid), ROOT / 'images' / 'test' / str(study_uid) / str(series_uid))
    for path in candidates:
        if path.is_dir():
            return path
    raise FileNotFoundError(f'report-teacher series missing: study={study_uid}, series={series_uid}')

def _rt_sorted_dicom_files(series_dir):
    files = list(Path(series_dir).glob('*.dcm'))
    if not files:
        files = [path for path in Path(series_dir).iterdir() if path.is_file()]
    if not files:
        raise FileNotFoundError(f'no DICOM files in {series_dir}')
    simple_numeric = [path.stem.isdigit() and len(path.stem) <= 8 for path in files]
    if np.mean(simple_numeric) >= 0.9:
        number_re = re.compile('(\\d+)')

        def key(path):
            matches = number_re.findall(path.stem)
            return int(matches[-1]) if matches else 10 ** 12
        return sorted(files, key=key)
    keyed = []
    for index, path in enumerate(files):
        try:
            ds = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)
            if hasattr(ds, 'ImagePositionPatient') and len(ds.ImagePositionPatient) >= 3:
                key = float(ds.ImagePositionPatient[2])
            else:
                key = float(getattr(ds, 'InstanceNumber', index))
        except Exception:
            key = float(index)
        keyed.append((key, path))
    return [path for _, path in sorted(keyed, key=lambda pair: pair[0])]

def _rt_robust_uint8(array):
    array = np.asarray(array, dtype=np.float32)
    finite = np.isfinite(array)
    if not finite.any():
        return np.zeros(array.shape, dtype=np.uint8)
    values = array[finite]
    lo, hi = np.percentile(values, [1.0, 99.0])
    if hi <= lo:
        lo, hi = (float(values.min()), float(values.max()) + 1e-06)
    array = np.clip(array, lo, hi)
    array = (array - lo) / max(hi - lo, 1e-06)
    return np.clip(array * 255.0, 0, 255).astype(np.uint8)

def _rt_center_crop_or_pad(image):
    height, width = image.shape[:2]
    pad_y, pad_x = (max(0, RT_IMG_SIZE - height), max(0, RT_IMG_SIZE - width))
    if pad_y or pad_x:
        top, left = (pad_y // 2, pad_x // 2)
        image = cv2.copyMakeBorder(image, top, pad_y - top, left, pad_x - left, borderType=cv2.BORDER_CONSTANT, value=0)
    height, width = image.shape[:2]
    y0, x0 = (max(0, (height - RT_IMG_SIZE) // 2), max(0, (width - RT_IMG_SIZE) // 2))
    return image[y0:y0 + RT_IMG_SIZE, x0:x0 + RT_IMG_SIZE]

def _rt_read_dicom(path):
    ds = pydicom.dcmread(str(path), force=True)
    array = ds.pixel_array.astype(np.float32)
    slope = float(getattr(ds, 'RescaleSlope', 1.0) or 1.0)
    intercept = float(getattr(ds, 'RescaleIntercept', 0.0) or 0.0)
    image = _rt_robust_uint8(array * slope + intercept)
    if str(getattr(ds, 'PhotometricInterpretation', '')).upper() == 'MONOCHROME1':
        image = 255 - image
    spacing = getattr(ds, 'PixelSpacing', None)
    if spacing is not None and len(spacing) >= 2:
        try:
            scale_y = np.clip(float(spacing[0]) / RT_TARGET_SPACING, 0.4, 3.0)
            scale_x = np.clip(float(spacing[1]) / RT_TARGET_SPACING, 0.4, 3.0)
            new_h = max(32, int(round(image.shape[0] * scale_y)))
            new_w = max(32, int(round(image.shape[1] * scale_x)))
            image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
            return _rt_center_crop_or_pad(image)
        except Exception:
            pass
    return cv2.resize(image, (RT_IMG_SIZE, RT_IMG_SIZE), interpolation=cv2.INTER_AREA)

def _rt_load_series_25d(study_uid, series_uid):
    files = _rt_sorted_dicom_files(_rt_locate_series_dir(study_uid, series_uid))
    quantiles = np.array([0.08, 0.23, 0.38, 0.5, 0.62, 0.77, 0.92], np.float32)
    centers = np.zeros(RT_SLICES, np.int64) if len(files) <= 1 else np.round(quantiles * (len(files) - 1)).astype(np.int64)
    centers = np.clip(centers, 0, len(files) - 1)
    views = []
    for center in centers:
        channels = []
        for index in (max(0, center - 1), center, min(len(files) - 1, center + 1)):
            try:
                channels.append(_rt_read_dicom(files[index]))
            except Exception:
                channels.append(np.zeros((RT_IMG_SIZE, RT_IMG_SIZE), dtype=np.uint8))
        views.append(np.stack(channels, axis=-1))
    return np.stack(views, axis=0)

def _rt_try_attached_visible_features(checkpoint_dir, expected_uids):
    uid_path = checkpoint_dir / 'rta_final_test_uids.txt'
    feature_path = checkpoint_dir / 'rta_final_test_features.npy'
    mask_path = checkpoint_dir / 'rta_final_test_slot_mask.npy'
    if not (uid_path.is_file() and feature_path.is_file() and mask_path.is_file()):
        return None
    if uid_path.read_text().splitlines() != list(expected_uids):
        return None
    features = np.load(feature_path, mmap_mode='r')
    mask = np.load(mask_path, mmap_mode='r')
    if features.shape[:3] != (len(expected_uids), 6, 7) or mask.shape != (len(expected_uids), 6):
        return None
    log('report-teacher: exact attached visible-test features reused')
    return (features, mask)

def _rt_extract_features(test_df, series_df, checkpoint_dir, dev):
    from transformers import AutoModel
    expected_uids = test_df['StudyInstanceUID'].astype(str).tolist()
    attached = _rt_try_attached_visible_features(checkpoint_dir, expected_uids)
    if attached is not None:
        return attached
    if time.time() - T0 > RT_START_CUTOFF_S:
        raise TimeoutError('insufficient runtime reserve for report-teacher feature extraction')
    dino_dir = _rt_find_dino_base()
    log(f'report-teacher: DINOv2 base from {dino_dir}')
    dino = AutoModel.from_pretrained(str(dino_dir), local_files_only=True).eval().to(dev)
    for parameter in dino.parameters():
        parameter.requires_grad_(False)
    dino_dim = int(dino.config.hidden_size)
    if dino_dim != 768:
        raise AssertionError(f'expected DINOv2-base hidden size 768, got {dino_dim}')
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    slot_lookup = _rt_assign_slots(series_df)
    features = np.zeros((len(expected_uids), 6, RT_SLICES, dino_dim * 2), np.float16)
    slot_mask = np.zeros((len(expected_uids), 6), bool)

    @torch.inference_mode()
    def encode(images):
        tensor = torch.from_numpy(images).permute(0, 3, 1, 2).float() / 255.0
        tensor = (tensor - mean) / std
        parts = []
        for start in range(0, len(tensor), 8):
            batch = tensor[start:start + 8].to(dev, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=dev.type == 'cuda'):
                output = dino(pixel_values=batch, interpolate_pos_encoding=True)
                tokens = output.last_hidden_state
                part = torch.cat((tokens[:, 0], tokens[:, 1:].mean(dim=1)), dim=-1)
            parts.append(part.float().cpu())
        return torch.cat(parts, dim=0).numpy().astype(np.float32)
    for row_index, study_uid in enumerate(expected_uids):
        if time.time() - T0 > RT_DEADLINE_S:
            raise TimeoutError('report-teacher deadline reached before submission overwrite')
        jobs = [(slot_id, slot_lookup[study_uid, slot_id]) for slot_id in range(6) if (study_uid, slot_id) in slot_lookup]

        def load_job(job):
            slot_id, series_uid = job
            return (slot_id, _rt_load_series_25d(study_uid, series_uid))
        if jobs:
            with ThreadPoolExecutor(max_workers=4) as executor:
                loaded = list(executor.map(load_job, jobs))
            encoded = encode(np.concatenate([views for _, views in loaded], axis=0))
            cursor = 0
            for slot_id, views in loaded:
                count = len(views)
                features[row_index, slot_id] = encoded[cursor:cursor + count].astype(np.float16)
                slot_mask[row_index, slot_id] = True
                cursor += count
        if row_index == 0 or (row_index + 1) % 100 == 0 or row_index + 1 == len(expected_uids):
            log(f'report-teacher features {row_index + 1}/{len(expected_uids)}')
        if dev.type == 'cuda' and (row_index + 1) % 100 == 0:
            torch.cuda.empty_cache()
    del dino
    gc.collect()
    if dev.type == 'cuda':
        torch.cuda.empty_cache()
    return (features, slot_mask)

@torch.inference_mode()
def _rt_predict_checkpoints(features, slot_mask, checkpoint_dir, dev):
    syn_index = TARGETS.index('Synovitis')
    seed_predictions = []
    for seed in RT_SEEDS:
        seed_prediction = np.zeros(len(features), np.float32)
        for fold in range(4):
            if time.time() - T0 > RT_DEADLINE_S:
                raise TimeoutError('report-teacher deadline reached during checkpoint ensemble')
            path = checkpoint_dir / f'rta_final_seed{seed}_fold{fold}.pth'
            checkpoint = torch.load(path, map_location='cpu', weights_only=False)
            if checkpoint.get('targets') != TARGETS:
                raise AssertionError(f'target order mismatch in {path.name}')
            cfg = checkpoint['cfg']
            model = RTAHMIL(in_dim=int(checkpoint['slice_feat_dim']), hidden_dim=int(cfg['hidden_dim']), n_targets=len(TARGETS), n_slots=int(cfg['n_slots']), n_slices=int(cfg['slices_per_series']), dropout=float(cfg['dropout']), series_dropout=float(cfg['series_dropout']))
            model.load_state_dict(checkpoint['state_dict'], strict=True)
            model.eval().to(dev)
            fold_prediction = []
            for start in range(0, len(features), 48):
                x = torch.from_numpy(np.asarray(features[start:start + 48])).float().to(dev)
                mask = torch.from_numpy(np.asarray(slot_mask[start:start + 48])).bool().to(dev)
                with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=dev.type == 'cuda'):
                    logits = model(x, mask)
                fold_prediction.append(torch.sigmoid(logits[:, syn_index]).float().cpu().numpy())
            seed_prediction += np.concatenate(fold_prediction) / 4.0
            del model, checkpoint
            gc.collect()
            if dev.type == 'cuda':
                torch.cuda.empty_cache()
        seed_predictions.append(seed_prediction)
    return np.mean(np.stack(seed_predictions, axis=0), axis=0)

def _rt_blend_synovitis(primary, teacher_synovitis, teacher_uids):
    result = primary.copy()
    primary_uids = result['StudyInstanceUID'].astype(str)
    teacher = pd.Series(np.asarray(teacher_synovitis, dtype=np.float64), index=pd.Index([str(uid) for uid in teacher_uids], name='StudyInstanceUID'))
    if teacher.index.has_duplicates or set(primary_uids) != set(teacher.index):
        raise AssertionError('report-teacher and primary StudyInstanceUID sets differ')
    teacher = teacher.reindex(primary_uids.values)
    if not np.isfinite(teacher.values).all():
        raise AssertionError('non-finite report-teacher prediction')
    base_rank = result['Synovitis'].rank(pct=True).to_numpy(np.float64)
    teacher_rank = teacher.rank(pct=True).to_numpy(np.float64)
    result['Synovitis'] = (1.0 - RT_SYN_WEIGHT) * base_rank + RT_SYN_WEIGHT * teacher_rank
    return result

def run_report_teacher_synovitis_specialist():
    if time.time() - T0 > RT_START_CUTOFF_S:
        log('report-teacher skipped: the primary ensemble used its runtime reserve')
        return False
    checkpoint_dir = _rt_find_checkpoint_dir()
    primary_path = Path('submission.csv')
    primary = pd.read_csv(primary_path, dtype={'StudyInstanceUID': str})
    if primary.columns.tolist() != ['StudyInstanceUID'] + TARGETS:
        raise AssertionError('primary submission schema mismatch')
    test_df = pd.read_csv(ROOT / 'test.csv', dtype={'StudyInstanceUID': str})
    series_df = pd.read_csv(ROOT / 'test_series.csv', dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
    expected_uids = test_df['StudyInstanceUID'].astype(str).tolist()
    dev = DEVS[0]
    features, slot_mask = _rt_extract_features(test_df, series_df, checkpoint_dir, dev)
    teacher_synovitis = _rt_predict_checkpoints(features, slot_mask, checkpoint_dir, dev)
    result = _rt_blend_synovitis(primary, teacher_synovitis, expected_uids)
    untouched = [target for target in TARGETS if target != 'Synovitis']
    if not result[untouched].equals(primary[untouched]):
        raise AssertionError('report-teacher changed a non-Synovitis target')
    if result.shape != primary.shape or not np.isfinite(result[TARGETS].to_numpy()).all():
        raise AssertionError('invalid report-teacher blend')
    temp_path = Path('submission_v26_synovitis.tmp.csv')
    result.to_csv(temp_path, index=False)
    reread = pd.read_csv(temp_path)
    if reread.shape != primary.shape or not np.isfinite(reread[TARGETS].to_numpy()).all():
        raise AssertionError('serialized report-teacher blend is invalid')
    temp_path.replace(primary_path)
    log('report-teacher complete: 0.75 Synovitis rank blend; all other targets preserved')
    return True

### ハイブリッドDINOエンコーダ：ラジオミクス特徴との併用

**何をしているか**: DINOv2 による深層特徴に加えて、
**手作りの画像特徴（ラジオミクス）**を計算し、両方を使うハイブリッドの枝です。

- `_hyb_crop_foreground` … 前景（膝）の範囲を検出して切り出す
- `_hyb_robust_uint8` … パーセンタイルによるロバストな輝度正規化
- `_hyb_view_radiomics` … 各ビューから統計量ベースの特徴を抽出
- `_hyb_sampled_triplets` … 3枚組のスライスをサンプリング
- `_HybridDinoEncoder` … 深層特徴とラジオミクス特徴を結合する

**なぜそうするのか**:

- **ロバスト正規化について**: MRIの画素値には**絶対的な単位がありません**
  （CTのHU値と違い、装置や設定で全く違うスケールになる）。
  最小値〜最大値で正規化すると、**たった1個の外れ値（金属アーチファクト等）**で
  画像全体が潰れます。1〜99パーセンタイルでクリップしてから伸張するのが定石です。
- **前景クロップについて**: MRI画像の周辺部はほとんど空気（黒）です。
  そのまま224×224にリサイズすると、**膝が画面のごく一部**にしか写りません。
  前景を切り出してからリサイズすれば、**同じ画素数でより高い実効解像度**が得られます。
  半月板の断裂のような細かい所見では、これが効きます。
- **なぜラジオミクスを併用するか**: 深層特徴は強力ですが、
  **学習データの分布に強く依存**します。
  輝度ヒストグラムやテクスチャ統計のような手作り特徴は表現力こそ劣るものの、
  **施設間の差に対する振る舞いが深層特徴と違います**。
  アンサンブルの価値は個々の精度ではなく**間違え方の多様性**にあるので、
  性質の異なる特徴を混ぜるのは理にかなっています。

In [ ]:
import base64
import gc
import hashlib
import io
import json
import math
import os
import random
import time
import zlib
from concurrent.futures import ThreadPoolExecutor
from functools import lru_cache
from pathlib import Path
import cv2
import joblib
import numpy as np
import pandas as pd
import pydicom
from scipy.stats import rankdata
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
HYB_PREFIX = 'v8_hybrid_dino224_6slot_5pos_radiomics'
HYB_EXPECTED_TRAIN_ID_SHA256 = '21c1944bd15c3397290f0816de614ad4153f62e84c4bfb0e4d6147ac72084af8'
HYB_TEACHER_PAYLOAD = 'eNrtXX3MnWdZf2dlNCNTiuNDx0fZnC2kA3RTSEvPmdoxoHN2M1W2lJQy2m2wvqvrRs8YTCBiKhuCkJB0ETMDhmUYdBETwkjWTBfBDEOEZAQzAkaTRUnbYEZcMNHze8/5nff3XO91P8/99cL+uN/l2jnnee7nfu6P6+N3fTxP9+ze8NMXL83+Xr70wy+9+Zr/m/89b+m8pZsP77/+luWjB5eP3n50/x2vWj5yx1lLz13ad/msPT8/cdXe39pz7VlL7166c8s7Dh69/tYt2zdvef2hS7ds27zl0C233nbrgeX9t9z6joM4/oYDNx89OD1+9MYDRw5Of2+99NLXvHbbK7Ztft/m3L9zPvfAK0egcx6/8PVK59727cX3zxzZOZ5/3/mZI+8baztc+3cfuGqsv0FsN+1nh/S5A/Scxy8cm3uttHnj996zcvzaH9y9Zjxsh75v2vCS8YkTP7/S9mOP3D7G7127do3/+Pjx14Mwdny+5pPvXTmGcfMcCH3g823b7hjrcUucC77fc9HDIx7ndz3P37x3H33gqvNX7os56DjYF8atY9N7gDBf/f2i3XeN2WeozdD8QFh/fsd64hzGwXUEffmhHYP9aj9Yjy8/dOcY+8RjmLf+/s4Ta8dq54M+ME899vinjyx+h/bS9oO5oC9vbXGcazI5dmwMvsZx8PJ8LjvY/vNXvHys1+Kc9mc/wdMP/vsLx7o2obUPEdaNa4D7YVz3XPQH098bV/j7a49dOeac2Z/ygeUjPUZeRB/YD91z0L6P/sZCxrSf48ffOP7m2e8ZKz+dOHFt555c19C9Y/jU479rvvjSTr+qQ7gfmD9+8zzW5sVPXjrmd0/PWJ2De1qdZ9uxf69fttfruI/kRRzDHnAPOe+h8aEN+rLH0S+/Y490f0DYXzs+rie+Y0/ZhscgE/gEz+m8Vefhu5W57Yc3j/v0jhL7uv6rr+6sj0e2X+07pA/QP85bOaQOwv26+mgmW1b3oB3WyM5D23Hsuh579z5/fMklRwdlHW3+44W7x2rHMC4Q9oHyOrvPQ6McudL5rdXf3eN2/p5uBb1i38+u8A3HgDYYr+i4kd6D7fCb+0n88In921b4jPbekwfw9vVf/esR23GPqQso/zyPT4xPZVRlBQTeufCCt3eOoT10OIh2APPCPTw9orhHz8NmUR9qW+oA8BTbaX+4j/ZpdR3ugT7xSZxl26jMcx3wSbsEUnyH4zoHjPGJJ9662CPwBIi8KPZ4ZGWA/MN+0ni0a5/1GNZLeVPlgrYopEO0r2dvP9wZ33SeI/An9QT4Wq8Fj1j+tzZ/ev0CC6u8qWzY63EM8wnZS9UF9ryVUYx906arp/L32uCa79r1DyPIHdpR1+A3ZYlr9/Bbbh336RhPp1PO/LWZjVf3hvwKvqN8sB/Kpdojz+6R6B+oXcax71/9xRHPqTwor5PuevAXxypDaP/Fl35oBKIN0XnrvlHncZ+hy3huNueZfuX82R/a8rzyDTFWP17cuBPXq/4FYW66F322gG24NqpLPbwTwihoD56x/qTuLXUOjoEshrN+qO4rCPsAuaTcKJ7h3lgchXmde9t1nWN2DjgP0t+8L+6jtohtQXP7NVKc5Y2bfes8ydt2THbPdV+p69SGAj/y+PxzBF/LyuaFF1zg6hjVn1av3H/ubeM+3yR0jGvx5Bd+fQ0Pgse9az3dR56nX6a+DTCp7RvtT59aXtFl+H7DoUOdOcOWx/rKSr99/o1jYjSNefC7hweInZUfLG972N47hv6/f/Vk7J3DuBhfsfidGIGY5+KnL3fXnnhIdSeO0Q8KyTwwhmJyjof3JiZBX1wjPW7ldOg7rqPORT/gJWIf9XHoK/P4PE6zQ3EXf1O++ZvX4BM6auYP3tk5z2OQD3stiNhNx0EfHd/pLyvu9rABrrV6H3qDc9r29OUjXsf4ivoGGvvAdbPYgW/PcZ+lpfePZuNZ9QXQP66d6YSuj0Dsgu+77/rllf2wshDyudXGWt7T48SnGC/wpupQ5Z/QfbQNdYbXzvKvHlfetfdj/zcc+qWx2ri+tn2+g2d7bQw2RIofXvahd3V4x/p41EUhrGZtkSW9l/bDuc78+o07OTdiJN1vxrXgY9Aean8ejp/ppTvHqb4DcO/evXs71wEDa0wxhrzYK/r43APvHFts9dhjj408DIaxdX2Ernzr3DVWzPXlvuKT2CRkk/v8IOwHbKL22eenefzLfdu0aVPHx7VyhHOM85O/tT2PcT60abZP5CaUz+yYqLsxt79/6ka3j5Bfw/Wjz8Uxe/gytN7gZU9nc780zsrr4dvpPg7FhtHe2lKNK6jNUyzKvdJ4QUgH8Rrawz7Mwmu5VsQTVm9xjnp/b36QU8Z9mZfQ/WJMIkRcb5t3GYovrvpi4ZxESD50zyjb6iuqf6jXkB+6mHSmpy2/AW949+e96dMzVhOSYb0X2+n4MaahPIIXk7X87cUOGZvmenCv7Dh07W2+TM+H4r4cB9akz9fz+Xl2PpSXtD4AY+m2zWen+yVxyg628PSWvf6+b7xhTAxvY0I8NvVNRjH+mfYR2qO+nBH254KpD3nThreO+/wn7/rD9/3cmLrpqa0vtnplp/GR1mBp8Bi/sw32dYa/r+34AMTqxNqKxTUuyj70XpBb+MmUYeoq1VlT/3Kh71axznULnKc6Wf09tLGxaW/PEa+yMWKLg0MYkvEuxkrYFraZ56mbsZ+nT72qI1dWZ1xyyd868YLV+GpIL1i96ekiGwvkdZbXvZytd5/5ni5428YK6Td6a8h1ZtxEMU1frLFnL3pxOmJ7jPvTLnM+6Jt8hT3UGBrmYG2qnadiAOUL9blhX+31nm6i/2v9EYsJRD4W/WiOhbjI84e0H82Ngjz8pvEVXk+/Vvv7x4/vHzNvqXUmmj/lGPlJ38TyL/mSuVr0BwxGP0/b4p6SD9gZ0o0cq83l4beHFULYALzj5e8sbrH40Y5JMe/s+6w/5iA1Tzmzq3868saltsXLWWi+27PXwAfcT7s2mKvKlsaaLI9yz5gvoj21cYA+P4V7aeN8+NT8gfKnyrWNUfN+j378ZWMbW7T7hPVA7tjabZvH5riYrxzK13r+K/vlHmCdiR8wT6tzGXtjG6036PJhmP/Jt4rf8J15AXzO7NVDI7/vrswN5bk9W+XhyiEfmnoHY7d5+T5ZD8VPVmtoZv1ojIL71JPH2akxCNpZ2G3NV1p/oC8v5OFpi7spI335JF1bi+V5Dp/w81b7W9U5WqNBfchx0M8Zrp+6tiM/qstsDo++XV9ew9OhXl7UttXaN8bCKVOH7/udNT6mrq3nB82OhfX9ECbXPi3mCuGuFF/B4yG1Bdb+WB+wT4ZwLbAEeUT5Tfmd/MN6FK43Mb93H7UHqmdpXyy21FjH/XNfj33bfAPjR16uxYs3q49x/rx+DbpQ56z+js7J+qgWV6nd7MOR3m+uM+0ZeNur+wV/2ryNnacXxwDvq3+r+CQm9jKc8/b9aNuXtU2+jVnNKdo1DvkauubE9+pTYEyaO6ZOwvc7z1t2eYc1ABpLtXlJ5iNDNRSWd/AdvqTudQwRS9g14JiI3+in6hg1f8k+9DxzKxaPcb9sPsPiT1uLotiLet6r4evq5x1jrQ160e5RRxZjdG6Aj0ceb9GnCsXzQE8/es1CJtV34LVar6mxa1u3TVyCe0lt3mC+PmSXpf+R5S29t+VBW7emcsC1lFr/xbXATqobrazYcSiP2pyzzTFQzlWvYs3AY8T8xMVWxliXpmNFXFvbwCdwamEW+WPaDdYjahyAMU7Vmc/efvHCJ2C8y/Ox6evAr1PfpS/u4ek1XMu8BH/D1wvli2JiqrZ/jZ2l5Oy4l5RvK39eTsDiRtt2KOfG+6nsPrV131jr663u4XrxetSycN/Zl/KoXq9rqHts15Nyxz6AP7En4Cf1Sbx9ok3HOa1vWavPPN+re4zyFMpnzOrfwhi3rzZJfViLLRU7496sJwv5PrYOdFaL87pxyDeybYkbQvofe9AXj+/jafUrvJiR6kLWhHrz1H6sn09Zs9gafWi+SXU65oP9ffzTrxnwj7uxLBLqtrx64r5npEAXBGrcQn5IKEYVyr+i7Zl5zBzj0pxwyB/0nlWhDtNYFu2P5kjVDjB3oDVN+mwG7b/mPFafZ5k9S4Xv2GuNEdt8i/ouoHnMjc9BDcZ2+Ok986C+puZGV+tkH1rzfB3l1uIf8iP3Ss97fqNtT73K85ojUh6nHPDZA/tso+pYrpXNHbIt9xi1Qtg3taeg3Xf9fiefv1oTs+pLefZU+crGQT0sBb5j3YfXZ99zUorL0Q/jYTbuqjZD+7N2BXuuzzJgH+eyscPWu+AY9sCrVVHeJS95NXU8r3FQHoNswC/xdcxqXRxjknpP5GH1ecHYWDzGQFnBnnzskV8dD8VwkaNkzLXPN/CeYWIuyfaP4zafxmdp//WyLYs9g18QU7OKMXrtQjk14lius5efAz7SnA5zZZgnMbtXH2vryuyYsP7Q31hTG7NfrZ9drYW1sRG7DjtkjWx+i7/5XJDmJUM4WHWQjcl4z2dLDmvkxXLQBnl4lWX1d/h8FdeZfdGHod5ivb/qGMyH2FBzh5wTcs7esyOe3JAfUQvW91wpeAL3VAxEmRquiVt9Psc+X2ePs/+Zz+9jU4vzwnV1/fFbvc7Lf2v9oH2+cejZPi/mydp2lZvV3KI+U3NtJ94HvaH+LuMGuA58RJ1IfaW5Y+Vz6N53bvjUiPoJvo/nD2JvVO+qv9SHN7jmnA/4gjpDcFzwWQvNj3o8+M2ztwdrHhiveM7j16/Um67y5Gq+xPpxKvu2VgTvI7C6xasR1mdWQnWZWE+vFlNjcNbue3XH+hv2wsN3luhbsCbN1qYN1Z/BT0B7XWvrJ4RitvY+6ut49S3gX1uzS54QHdObfw3xjvpW6PO/7/7LkZeXYRvmCLV/2kNti1qlGFygse2pjd0JCmH8mGcr+3Qi1tDvo+sLhuZv116fK97TeT/L6YNvejffz/J8+36W6bdDz9RXtDzwwCvHIIa7STC3/P6xR55aEU+W+Svh2nfOH4Hib+2PIXp+B23b99GR9sE2UMv4vOr8J0f2Pmy30vcU/n7lsStX2mzGmKa/jx/fOD42NZkgjB2fEE18Ytw8B0If+Dw5bxcizgXfn/zCB0Y8zu96nr8fHugTtPG261auxxx0HOwL49ax6T1AmK/+xrqwz1CbofmBsP78jvXEOYyD6wg6c+rUYL/aD9YDUGaz7AHmrb/PTFVHaI1IfJxDj33+igOL76G9tP0wnOit7UTWZDqmFb4+Nn8Mn+vM9kin6bUcG4/ZT/D08YvGI12b0NqHCOvG+1BmZqWVyyvH7p2qZM6Z/SkfWD7SY+RF9IH90D2fzfdZY8qY9vPcqYuF8Kjy069cckmn/8nAvWP41OO/6Rw7e6A6hOuE+XP/8Im1OXLfN0b87ukZq3NwT6vzbDv27/XL9lYvkud4LfaAe8h5D40PbVTHkiYyHuyR7g+IcFvHxfXEd+wp2/AYZII8p/NWnYfvVuZUd3p6R4l9AXbq+nhk+9W+Q/oA/eO8lUPqoInRR5Qtq3vQDmtk56HtJo4Oun8KHQ5NoeQQz6PNP736BR07hnHNaSGvM119fJQjVzo/q7/tcTt/T7fOrn1qJ8bLMaDNmXlf6PfUXH55D7bDb7Wr4K0r73pwJkdze+/JA3h7++H7Fu24xwtdMJd/nmephcqoygoIvPOGXbs6x9AeOhxEO4B54R6eHlGZ1POwWRMj2wtdNZc7ttP+cB8r56rrcA/0iU/iLNtGZZ7rgE/aJbYl4bjOAWP80kM7FjzG+5AXaY9PiY6eGP5hPyk8au2zHsN6KW+qXNAWTSLsys/cdl1nfM+b8tXp+f6Sr/Xak46eszYfYTzus8qbyoa9nvsXspeqC+x5K6MYO9JU333iieCaox/IHdpR1+A3ZYlrd9eD7+i13Z5Op5x5a8Px6t6QX8F3um7k05NzueY5z+6R6B+oXcaxrVPfg+dUHpTXSf/z6MdHKkNoD18IRBui89Z9OyP6E2sAXcZzE9G7E1k/6hKeV74hxurDi7gG16v+XbEP07npXvTZArbh2nR0qYN3QhgF7cEz1p/UvaXOwTGQxXDWD9V9pV8KuaTcKJ7h3lgcxZJgPWbnwDb6m/fFfdQWsS3lBrpPcZY3bvat8zxp5mn1Bvdc95W6Tm0o8COP4xPjga9lZfOeKWbw9IHqT6tX9px/Y69vEjrGtfj6TRvW+qnztYnxAcjzC79MfBtgUts32n8Hj0hP2+H7Pz92ZWcdYMtjfWWlP/+1PQuMZmMelhdJxM7KD5a3PWzvHUP/35q/nsKew7gYX7H4nRiBmOe6HzzHXfuJ+LyKkegHhWQefKKYnOPhvamvJ3OdaI9bOR36Tvyha67H2FYfOeFesR3Hx9+Ub/7mNfiEjqLN0vM8Bvmw14KI3XQc9NHxnf6y4m4PG1BnWBzCOUEueB3jK+obaOwD10FmQvYc91laWloZz2ljy3Atxm19BO7lCg8+/ajrY4d8brWxlvfsvpNXgDdVhyr/hO6jbagzvHaWf/W48q69H/v/sxMnRmrj+tr2+Q6e7U0h7s3onN/r8I718aiLQljN2iJLel774VzRP+7HuREj6X7T1sDHOCnj4eck4BNMMnwH4F7L88DAGlOMIS/2ij5+d+/eNdjq61Pd4GGwSc/6Mo4xMb8p8xrrwyexScgm9/lB2A/YRO2zz0/z+Jf7dvz4xo6/buUI5xjnJ39rex7jfGjTbJ/E4iGZou7G3P7ov64ee32E/BquH30ujtnDl6H1Bi97Opv7pXFWXg/e1H0cig2jvbWlGldQm6dYlHultimkg3jNIgcTEY/kWtn4ivaptjgkA7NYyyymwLyEnmNMIkRcb5t3GYov0hfry0mE5EP3TPWh5x/qNeQHxaTU05bfgDe8+/Pe9Om5xkEZlnuxnY5/RUcP5BG8mKzlby92yNg014N7Zceha2/zZXo+FPflOLAmfb6ex888H8pLenlPxNJtm6um+8U4JGNkIazvydhlnz13geFtTIjH3jLF9DH+mfYR2qO+nNEKHp5iy385e3uv/+Rdf+OGlyxw4v1XHOjoFc0xh7A0eIzf2Qb7CnkAllbcTqxOrO1hcf4zFOhD74Xz98zxs+o21VnQR2xLG0R+wW/VyervUaY0Nu3t+da53VNfZYhP2J7xLsZK2Ba2mecn4v+gPEf3zOqMe6eY1u6lxldDesHqTU8X2Vggr7O87uVsvftgfR+W3LVn37y8nGIVxk0U0/TFGkN7MeTnI7bHuD/tMufzlTlm1DoMxtC8+Kc3T8Vq3Hf1ub16DU830f+1/ojFBBwbc3k2x0Jc5PlD2o/mRkEeftP4Cq+nX6v9/clF40XeUutMdB0nZqwqxx5fMleL/oDB6OdpW9yT/Gr9ZcvbIJvLw28PK4SwAXjHy99Z3GLxox2TYl58Z3/MQWqeEp/fmecY7Li6tSNrcxaa7/bsNfAB99OuDeaqsqWxJsuj3DPmi2hPbRygz0/hXto4Hz41f6D8qXJtY9S839ZPvndkY4t2n7AeyB1bu23z2BwX85VD+VrPf2W/3AOsM/ED5ml1LufFNlpvoHzYx//kW8Vv+M68AD7RP3nA9m1lbijP7dkqD1cO+dDUO3MdEGzv2TEvfsI9tLF93b9QHgexbo1BUPfDbmu+0voDfXkhD0/bMVFG+vJJurYWy/McPs/M8yfoT3WO1micMTqPfs4QAavYGBD7tDk8+nZ9eQ1Ph3p5UdtWa98mxs/89mVb1uhqXVvPD8KxPn0/hMm1T4u5QrgrxVfweEhtgbU/1gfskyFcCyxBHlF+U34n/7AeZSIxv9B91B6onqV9sdhSYx175r6e1usphmf8yMu1ePFm9TFuntevHZ/n8HSeE3Mvi+PY5pjBY9Y+DcWD+JvrTHu2ee6/2TjUacl/efGmUAwFvK/+reKTmNjLUM475Efbvqxt8myM5hQnztp72FXXnPhefQrNPfN6/v7h1n0u77AGQGOpNi/JfGSohuKYwyPwJXWvY4hYwq4Bx0T8Rj9Vx6j5S/bRXa/bO30rhuNxqytC+gTroNiLen4ywEewV1ob9LZtT3dkMUbnenx8yqmzBm89HNCNSj+4+6KFTJ4xOR5br6mxa1u3rblRxvX+6oEHRkOYKGSX2b+NfXnxUuVBLz6ntpn8bq+1utHKih2H8qjNOdscw6J2QPTqwyv4ZXlEzK85aXsPYBEdK+JInbjP1CewuT+NWdFusB5Rr2WMU3Xmi5/8wsInYLzL87Hp68CvU9+lL+7h6TVcy7wEf987fw2Qly+Kiana/jV2lpKz415Svq38TQZr0JfX5A+Gcm68n8ruf179uk59vdU9XC9eD3w/MTUzyqPWn1BMEqrXWjwnMO8D+POk8ak1BqZEmz7D7ctr6lD6fC97jPIUymdgv/swbl9tkvqwFlsqdp7ZmVk9Wcj3sXWguP5vrvjeKOQb2bbEDSH9z3xEam0s+jDP1ATjsGj7XYnX2HluNs9Ldfz8uaxZbI0+NN+kOh3zwf7u++gjoz7/2MaySKjb8uqJ+56RYrxoyGexmMCLUYXyr2h7ch4zn5j6vJA/6D2rQh22Jj9gcqQdOzDPHWhNkz6bQfuvOQ+1HcTg2GuNEdt8yzFT68TaH7TVeEMovsJP75kH9TU1N8pPu38at7D4h/zIvdLznt9o209EP+qc1Q7R16FdR5zL82m4F1wrmztkW+4xaoWwb2pPQZ/837d39p06+LSpJbT2VPnKxkE9LAW+Y92H12ffc1KKy+f4Z00dlv5mTChk/7Hn9lkG8LrW+ekeYw+8WhXLJ+zXy2PaOCiPQTbgl3h8pHVxjElqX8jDhur4+7AGxkBZwZ5cds7jgzFc5CgZc+3zDbxnmJhLsv3juM2n8Vnamzd8arF/K88KRNSsYoxeu1BOjTiW6+zl5+6d/5P15EPmyihbXk7dxnU8Hx3rD/2NNbUx+9Vn4VZrYT0fWu/9fVkjm9/ibz4X5NXXWBysOqgvRmNrf0/Nx+z5I8jDqyyrv7N4vmq+zuyLPgz1Fuv9bY2uxhrUXwch5+w9O+LJDfkRtWB9z5WCJ3BPxUCUqSHfVZ/Psc/X2eNnFjmup4PxV4vzQnV1Q/Fbvc7Lf6u/bJ9vHHq2z4t5srZd5Ya2RWWGPECeYo5Y9QPvCz4if1Nf2Ty8xhFRQ0z9BKzh+YPYG/tMdEhP2ti2Pt8FvqDOII7TGIzNx2h+1OPBPSYfYvMtuP4Fn79iRe7Zt+ZLrB/XscWmVoQ6eCgu58U27D2Oz/W0rcXUGJy1+0MxQNgLD9+twcRz34I1abY2baj+DH4CscKxgJ8Qitna+6iv49W3EPt5OWvup43hefijT88xJv0L89hNKL/iPU9Ie9jVYxujcIHGtqEHqDs9jD9JyAV566AxqL66htD87drrc8XdV7R868MnjvAVLZtmr2g5ctPN7zp2YPmG/Xc8U1/PshT19/6x/z312NC5vmtsO7bl99QxpNwvZg1S/nTsXn+h+YTu77XPGaM3R9t3TL9D44vpM3WOsTzVt7ahMebw1hDf5vQXu45DYx/iydBah9ZhiPdK5aVEF/XNI3VtcsYYqwNi5D7lfqU6u29fU/VqyZxy7EjJGGNtSUy7kj32+FRls6Y9yrXnMfogJG8pNmJIb8ba0VibFmtP+7BHilzF8H0Or8WseSxf9fFyjJ1M1bkpfJm6lznrnttvKt7N5Y++MQ7114enS/Yt9t6xspdiV2LkNdZGxt6/T3aG5Cp1fDH6MFVP5a51yt6H5htrz0p5I1f/xNqimhgtF9vGyG6qTiqdX4q85/jNQ3xf6nulrEeOX1Jbr8bop5S4Sar+KPGth3RkrFzmYOQYPknFgrE+eow9irVBQzKSGmOKnWdMHKFEpmKxdE48pUT+Y7B1rXhCDu78SfiMMTYjRdfU1Eultiinn9TjsRi0JFabinFqxG1yMEeu3ijd29rxkJg4ROo+lsR6a8Y018unKI3Fp9rZlNhCrO5N5ZUaWCvHpyqNS6fa59TPXB+lRvypVgy1RLfm5qJKYuVDOLAEz+TGcFLxZkqMspY8pmDsVPxUgm/71i8Vd6bGv0t8mxw/scT2pGKQWv58Ck+XrlmOPNbIUebm7kvsbWyMP5Z3hvIntXy9VJxWSx5r+JE5OGgoThIbo4jtr8SHTIlDlMS6avmZKf5rjJ3weL8UT6TEk2rq41LfoobfHIPhc+oHYnNQufPPjZXFxCRr2PKUGFAt3z/FRyrJMdaqR1wv/yg195XD5zE6vURXpeZpa8ZshnInpXnEEj7IjTmW5KJLsWCuz56Le0pjOKny0edPlMYA16MOINZOpGDRHJ7J8f/Xy66WxnNy9FtszUyJva0Zn4rBDTXwWE3cFbs2OfVoOf5vig9VIk8lscv1wCy1nhvJ8UlL6mFr1cIN4ZrUmqz1jKuX8uIQTiyJV6TUIOTYk5oykVpTE7N+qb7Uej2XVIoxc2NUtZ6TysF4Of52Lj74ScZWcuxiSU44VYfUiI3Ueo6qVs469ZrS5xZS4lalz9Pk4P/1kIlYux+rm1P3Mac2pSROUAv3D/FKKR6pXTOcUhNbysu54+3DRrHxp1QfPXcdYzFUrVhpSU1Jrbrv1DHUeMY+Vkemxp5q2ueSGvyQvqtRvx9be58TVyzBrKUx0Bjer1WvnBufzMUHKRg5Ny67HutQs2avlm1LmV8NjLV2DbqvZ/nIB3+0ma9nOc+8nuX6W5YPPXPf0OJpi0aNGjVq1KhRo0aNGjVq1KhRo0aNGjVq1KhRo0aNGjVq1KhRo0aNGjVq1KhRo0aNGjVq1KhRo0aNGjVq1KhRo0aNGjVq1KhRo0aNGjUqo+4bWvbccM/X+IaWTUvnLd1yYPUNLXc8c1/PUvJS5BovC6z1wtdQ333nar44NHRex7FeLxasNceYF0wP7VPMC8qGXqymbe33mBfzhdqF5lTrJU+pLwAsXcPQS1h/HPyeyt+pa17ycuOY/R5aq5Rrcl8qWmMNQvJR8jLZ1Jcaxq5LzZeupvBbzBiGdEPOmGNkI0YvpvBALf1f6yW3pS9+TrXtKbYl5eW5MTaoli3pk7PUFyfH8FGKDomZS5/urP0C6px5p65xyUslY3RjzD7G6Pwaspi7R6nYvvQFrjVfihzSMam4MUVnp9rbFL2Vuz85e1ZLdnP4svZ65/hrNe1syUt+c7Fg7guaY/zzWBlLxaOpPBOy5zH9pPoBJZjtx4HXUvRqzvhLbNDQnqTGpmJxVw5mL9WRpVhuyIfwxpmiu2JtbqndL8ExsfwQ6wfUlMla8YBcfirFmLX0TI6vlcKrqT5YKs7O2duU2FINHypGZ6bozhyMUxLfS4mpxeYESuWmNHaZEp/MWau+OHktn6+WTivF+Km4oFRX1Ip9xY45Zl1T55gy11jZyol1xWLzoXvG7kNsrCzXh4uNA6TKfWrMrCaf5mKCFB1UEyfUil3UtgElcZdUnVVDZ6fmi2JxTw7uyslhpMhZKvaI0SW5uf3cdQthxxSdG+Mn19CRKXa8xI6WrnWfPNbQbaV1HzWwW47vVzOHXeK7x/iwKfJe4gesh1zExoRq4YOcOF5JDChVP5RiotS8eW2+Tr1fbXwRE//M8alq6phUW1+yj7m27ceNp2vEm2vU6uXEFktjBTnYp8Y8c/O5sfn9Ut2ZE5erFauqXV9VgiVKMNCQz53jl8TY7NR65aF1Ts395uYta/ucpTG7XPueWiMX6xfl1sfnxsVy6xhz8FefbsvB1amYIGZdc331WFkLUY4eza35yhlzrO+d029OjUYsbsv1g3PjwjnYoXZ8syTvmCNDqbJTC9elYLmceHlJbeaQTMSudUpMK7a2uzZ2KvF7a9SMlGLWlNxwbkwiR95Tashic1Q19jzFFqTMr6Z+WO+agvXyPfpijrl8HRuDi8kT5dro3NrqnOtT88mlccGaPBkTP8v1n1L8kRx7VPJMY2zt7HrV0MVilpoxzRKbWjuuUCIvqXKS6q/X2POSGrDcnHOuPU2Ze61nVnJr8FNkN1Xv1rBbKfKynjHDGH8gJaaxHs+Pp+Cdms9KlWDz1D2pMc4YnyxVP9esQcjFiDXmHavTcnzknFxI7XcrrP51X8/ykQ/+aDNfz3KeeT3L9bcsH3rmvqGlvWinUaNGjRo1atSoUaNGjRo1atSoUaNGjRo1atSoUaNGjRo1atSoUaNGjRo1atSoUaNGjRo1atSoUaNGjRo1atSoUaNGjRo1atSoUaNG9WnP7rN+6uINq+9o+eGX3nwN3rsCep6+2+Us/P/mw3hRy9GDy0dvP7r/DryvxV5/+uCb3s3rn9+5/r7LzfV85Yvt4lsfPnGEXWzqdPGbd690sXhpjDsCvGWGl5/Xufzfvt69PDSAPTfc8zV/AH/4grM6b61JHMCF+85yX3qzZ/ezzkaLs6f/fWi60I/+BVb7/wGZqKOF'
HYB_START_CUTOFF_S = 7.2 * 3600
HYB_DEADLINE_S = 8.72 * 3600
HYB_IMG_SIZE = 224
HYB_N_POSITIONS = 5
HYB_N_SLOTS = 6
HYB_SLOT_META_DIM = 8
HYB_STUDY_META_DIM = 13
HYB_RAD_DIM = 14
HYB_RAD_AGG_DIM = 56
HYB_SEEDS = (20260809, 20260810, 20260811, 20260814, 20260815, 20260816, 20260817, 20260818)
HYB_LM_FAMILY_WEIGHTS = (0.5227272727272728, 0.27272727272727276, 0.20454545454545456, 0.0)
HYB_FAMILIES = ('lr', 'et', 'hgb', 'exact_lr')
HYB_FAMILY_WEIGHTS = (0.46, 0.24, 0.18, 0.12)
HYB_LR_CS = (0.015, 0.05, 0.16, 0.5)
HYB_EXACT_LR_CS = (0.01, 0.03, 0.1, 0.3)
HYB_TARGET = 'Lateral Meniscus'
HYB_OA_TARGET = 'Lateral OA'
HYB_OA_SEEDS = (20260809, 20260810, 20260811, 20260812, 20260813, 20260814, 20260815, 20260816, 20260817, 20260818)
HYB_OA_FAMILY_WEIGHTS = (0.5227272727272728, 0.27272727272727276, 0.20454545454545456, 0.0)
HYB_PLANES = ('Sagittal', 'Coronal', 'Axial')
HYB_CONTRASTS = ('Fluid', 'Structural')
HYB_SLOT_NAMES = tuple((f'{plane}_{contrast}' for plane in HYB_PLANES for contrast in HYB_CONTRASTS))

def _hyb_required_cache_names():
    names = []
    for split in ('train', 'test'):
        stem = f'{split}_{HYB_PREFIX}_'
        names.extend((stem + suffix for suffix in ('slot_features.npy', 'slot_mask.npy', 'slot_meta.npy', 'radiomics.npy', 'study_meta.npy', 'sex.npy', 'ids.npy')))
    names.append(f'{HYB_PREFIX}_ipca_192.joblib')
    return names

def _hyb_find_cache_dir():
    local = globals().get('HYB_LOCAL_CACHE_DIR')
    candidates = []
    if local:
        candidates.append(Path(local))
    root = Path('/kaggle/input')
    if root.is_dir():
        marker = f'train_{HYB_PREFIX}_slot_features.npy'
        candidates.extend((hit.parent for hit in root.glob(f'*/{marker}')))
        candidates.extend((hit.parent for hit in root.glob(f'*/*/{marker}')))
    required = _hyb_required_cache_names()
    for path in candidates:
        if all(((path / name).is_file() for name in required)):
            return path
    raise FileNotFoundError('the complete public hybrid feature/PCA cache is absent')

def _hyb_find_dino_small():
    preferred = (Path('/kaggle/input/models/metaresearch/dinov2/pytorch/small/1'), Path('/kaggle/input/dinov2/pytorch/small/1'), Path('/kaggle/input/dinov2-small/pytorch/small/1'))
    for path in preferred:
        if (path / 'config.json').is_file():
            return path
    for top in Path('/kaggle/input').glob('*'):
        if not top.is_dir() or 'dino' not in top.name.lower():
            continue
        for config_path in top.glob('**/config.json'):
            try:
                cfg = json.loads(config_path.read_text())
                if cfg.get('model_type') == 'dinov2' and int(cfg.get('hidden_size', -1)) == 384:
                    return config_path.parent
            except Exception:
                continue
    raise FileNotFoundError('offline DINOv2-small model is absent')

def _hyb_numeric(value, default=0.0):
    try:
        value = float(value)
        return value if np.isfinite(value) else default
    except Exception:
        return default

def _hyb_sex_to_id(row):
    value = str(row.get('PatientSex', '')).strip().lower()
    if value.startswith('m'):
        return 1
    if value.startswith('f'):
        return 2
    return 0

def _hyb_choose_series(part, contrast, used_ids):
    if len(part) == 0:
        return None
    fluid = part['Fluid_Sensitive'].fillna(0).astype(float)
    fat = part['Fat_Suppression'].fillna(0).astype(float)
    if contrast == 'Fluid':
        score = 4.0 * fluid + 2.0 * fat
    else:
        score = 3.5 * (1.0 - fluid) + 1.5 * (1.0 - fat)
    ordered = part.assign(_slot_score=score).sort_values('_slot_score', ascending=False)
    for _, row in ordered.iterrows():
        series_id = str(row['SeriesInstanceUID'])
        if series_id not in used_ids:
            return row
    return ordered.iloc[0]

def _hyb_build_slots(series_df):
    slots, study_meta = ({}, {})
    for study_id, rows in series_df.groupby('StudyInstanceUID', sort=False):
        study_id = str(study_id)
        selected, meta = ({}, [])
        plane_lower = rows['Anatomical_Plane'].astype(str).str.lower()
        for plane in HYB_PLANES:
            part = rows[plane_lower == plane.lower()]
            count = len(part)
            fluid_mean = part['Fluid_Sensitive'].fillna(0).astype(float).mean() if count else 0.0
            fat_mean = part['Fat_Suppression'].fillna(0).astype(float).mean() if count else 0.0
            meta.extend([np.log1p(count) / 3.0, fluid_mean, fat_mean])
            used = set()
            for contrast in HYB_CONTRASTS:
                row = _hyb_choose_series(part, contrast, used)
                if row is None:
                    continue
                sid = str(row['SeriesInstanceUID'])
                used.add(sid)
                selected[f'{plane}_{contrast}'] = {'series_id': sid, 'contrast': contrast, 'fluid': _hyb_numeric(row.get('Fluid_Sensitive', 0)), 'fat': _hyb_numeric(row.get('Fat_Suppression', 0))}
        total = len(rows)
        meta.extend([np.log1p(total) / 4.0, rows['Fluid_Sensitive'].fillna(0).astype(float).mean() if total else 0.0, rows['Fat_Suppression'].fillna(0).astype(float).mean() if total else 0.0, rows['SeriesInstanceUID'].nunique() / 12.0 if total else 0.0])
        slots[study_id] = selected
        study_meta[study_id] = np.asarray(meta, dtype=np.float32)
    return (slots, study_meta)

def _hyb_locate_series_dir(study_uid, series_uid):
    candidates = (ROOT / 'test_series' / str(study_uid) / str(series_uid), ROOT / 'test_series' / str(series_uid), ROOT / 'test' / str(study_uid) / str(series_uid), ROOT / 'test_images' / str(study_uid) / str(series_uid), ROOT / 'test_dicom' / str(study_uid) / str(series_uid), ROOT / 'test_dicoms' / str(study_uid) / str(series_uid), ROOT / 'images' / 'test' / str(study_uid) / str(series_uid))
    for path in candidates:
        if path.is_dir():
            return path
    raise FileNotFoundError(f'hybrid series missing: study={study_uid}, series={series_uid}')

def _hyb_read_header(path):
    try:
        return pydicom.dcmread(str(path), stop_before_pixels=True, force=True)
    except Exception:
        return None

def _hyb_header_position(ds):
    if ds is None:
        return None
    try:
        ipp = np.asarray([float(x) for x in ds.ImagePositionPatient], dtype=np.float64)
        iop = np.asarray([float(x) for x in ds.ImageOrientationPatient], dtype=np.float64)
        return float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
    except Exception:
        pass
    for name in ('SliceLocation', 'InstanceNumber'):
        try:
            return float(getattr(ds, name))
        except Exception:
            continue
    return None

@lru_cache(maxsize=8192)
def _hyb_ordered_files(folder_str):
    files = sorted(Path(folder_str).glob('*.dcm'))
    if not files:
        files = sorted((path for path in Path(folder_str).iterdir() if path.is_file()))
    keyed, ok = ([], 0)
    for fallback, path in enumerate(files):
        key = _hyb_header_position(_hyb_read_header(path))
        if key is None:
            key = fallback
        else:
            ok += 1
        keyed.append((key, str(path)))
    if ok >= max(3, len(files) // 3):
        keyed.sort(key=lambda pair: pair[0])
    return tuple((path for _, path in keyed))

def _hyb_spacing(ds):
    spacing_x = spacing_y = thickness = 0.0
    try:
        ps = [float(x) for x in ds.PixelSpacing]
        spacing_y, spacing_x = (ps[0], ps[1])
    except Exception:
        pass
    for name in ('SliceThickness', 'SpacingBetweenSlices'):
        try:
            thickness = float(getattr(ds, name))
            break
        except Exception:
            continue
    return (spacing_x, spacing_y, thickness)

def _hyb_read_pixel(path):
    ds = pydicom.dcmread(str(path), force=True)
    array = ds.pixel_array.astype(np.float32)
    array = array * _hyb_numeric(getattr(ds, 'RescaleSlope', 1.0), 1.0)
    array += _hyb_numeric(getattr(ds, 'RescaleIntercept', 0.0), 0.0)
    if str(getattr(ds, 'PhotometricInterpretation', '')).upper() == 'MONOCHROME1':
        array = array.max() - array
    return (array, ds)

def _hyb_robust_uint8(stack):
    stack = np.asarray(stack, dtype=np.float32)
    finite = stack[np.isfinite(stack)]
    if finite.size == 0:
        return np.zeros(stack.shape, dtype=np.uint8)
    low, high = np.percentile(finite, [1.0, 99.4])
    if high <= low:
        low, high = (float(finite.min()), float(finite.max()))
    if high <= low:
        return np.zeros(stack.shape, dtype=np.uint8)
    return (255.0 * np.clip((stack - low) / (high - low), 0.0, 1.0)).astype(np.uint8)

def _hyb_crop_foreground(image):
    gray = image.max(axis=2)
    mask = gray > max(8, np.percentile(gray, 55) * 0.18)
    if mask.sum() < 64:
        return image
    ys, xs = np.where(mask)
    y0, y1, x0, x1 = (int(ys.min()), int(ys.max()) + 1, int(xs.min()), int(xs.max()) + 1)
    pad_y, pad_x = (int(0.08 * (y1 - y0 + 1)), int(0.08 * (x1 - x0 + 1)))
    y0, y1 = (max(0, y0 - pad_y), min(image.shape[0], y1 + pad_y))
    x0, x1 = (max(0, x0 - pad_x), min(image.shape[1], x1 + pad_x))
    if y1 - y0 < 32 or x1 - x0 < 32:
        return image
    return image[y0:y1, x0:x1]

def _hyb_resize(image):
    return cv2.resize(image, (HYB_IMG_SIZE, HYB_IMG_SIZE), interpolation=cv2.INTER_AREA)

def _hyb_view_radiomics(image):
    gray = image.astype(np.float32).mean(axis=2) / 255.0
    height, width = gray.shape
    q = np.percentile(gray, [1, 5, 10, 25, 50, 75, 90, 95, 99])
    center = gray[height // 4:3 * height // 4, width // 4:3 * width // 4]
    gy, gx = np.gradient(gray)
    grad = np.sqrt(gx * gx + gy * gy)
    foreground = gray > 0.08
    return np.asarray([gray.mean(), gray.std(), q[0], q[2], q[4], q[6], q[8], center.mean(), center.std(), grad.mean(), grad.std(), foreground.mean(), gray[foreground].mean() if foreground.any() else 0.0, gray[foreground].std() if foreground.any() else 0.0], dtype=np.float32)

def _hyb_make_view(paths):
    arrays, first_ds = ([], None)
    for path in paths:
        try:
            array, ds = _hyb_read_pixel(path)
            first_ds = ds if first_ds is None else first_ds
            arrays.append(array)
        except Exception:
            return (None, np.zeros(HYB_RAD_DIM, np.float32), (0.0, 0.0, 0.0))
    image = np.transpose(_hyb_robust_uint8(np.stack(arrays)), (1, 2, 0))
    image = _hyb_resize(_hyb_crop_foreground(image))
    return (np.transpose(image, (2, 0, 1)), _hyb_view_radiomics(image), _hyb_spacing(first_ds) if first_ds is not None else (0.0, 0.0, 0.0))

def _hyb_sampled_triplets(study_uid, series_uid):
    files = list(_hyb_ordered_files(str(_hyb_locate_series_dir(study_uid, series_uid))))
    if not files:
        return ([], 0)
    centers = np.round(np.linspace(0.08 * (len(files) - 1), 0.92 * (len(files) - 1), HYB_N_POSITIONS)).astype(int)
    centers = np.clip(centers, 0, len(files) - 1)
    return ([[files[max(0, center - 1)], files[center], files[min(len(files) - 1, center + 1)]] for center in centers], len(files))

def _hyb_load_study(row, slots, study_meta):
    study_uid = str(row['StudyInstanceUID'])
    images = np.zeros((6, 5, 3, 224, 224), np.uint8)
    view_mask = np.zeros((6, 5), bool)
    slot_meta = np.zeros((6, 8), np.float32)
    radiomics = np.zeros((6, 56), np.float32)
    selected = slots.get(study_uid, {})
    for slot_index, slot_name in enumerate(HYB_SLOT_NAMES):
        info = selected.get(slot_name)
        if info is None:
            continue
        triplets, n_files = _hyb_sampled_triplets(study_uid, info['series_id'])
        rad_values, spacings = ([], [])
        for pos_index, paths in enumerate(triplets[:5]):
            image, rad, spacing = _hyb_make_view(paths)
            if image is None:
                continue
            images[slot_index, pos_index] = image
            view_mask[slot_index, pos_index] = True
            rad_values.append(rad)
            spacings.append(spacing)
        if rad_values:
            rad_array = np.stack(rad_values).astype(np.float32)
            radiomics[slot_index] = np.concatenate([rad_array.mean(0), rad_array.std(0), rad_array.min(0), rad_array.max(0)])
            spacing_mean = np.asarray(spacings, np.float32).mean(0)
        else:
            spacing_mean = np.zeros(3, np.float32)
        slot_meta[slot_index] = np.asarray([info['fluid'], info['fat'], float(info['contrast'] == 'Structural'), np.log1p(n_files) / 6.0, float(view_mask[slot_index].mean()), spacing_mean[0] / 2.5 if spacing_mean[0] else 0.0, spacing_mean[1] / 2.5 if spacing_mean[1] else 0.0, spacing_mean[2] / 8.0 if spacing_mean[2] else 0.0], np.float32)
    return (images, view_mask, slot_meta, radiomics, study_meta.get(study_uid, np.zeros(13, np.float32)), _hyb_sex_to_id(row), study_uid)

class _HybridDinoEncoder(nn.Module):

    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, pixel_values):
        tokens = self.backbone(pixel_values=pixel_values).last_hidden_state
        patches = tokens[:, 1:]
        return torch.cat([F.normalize(tokens[:, 0], dim=1), F.normalize(patches.mean(dim=1), dim=1), F.normalize(patches.amax(dim=1), dim=1)], dim=1)

def _hyb_load_cached_split(cache_dir, split):
    stem = f'{split}_{HYB_PREFIX}_'
    return tuple((np.load(cache_dir / (stem + suffix), mmap_mode=None if suffix == 'ids.npy' else 'r', allow_pickle=suffix == 'ids.npy') for suffix in ('slot_features.npy', 'slot_mask.npy', 'slot_meta.npy', 'radiomics.npy', 'study_meta.npy', 'sex.npy', 'ids.npy')))

def _hyb_extract_test_bundle(test_df, series_df, cache_dir, dev):
    expected_uids = test_df['StudyInstanceUID'].astype(str).to_numpy()
    cached = _hyb_load_cached_split(cache_dir, 'test')
    if np.array_equal(np.asarray(cached[-1]).astype(str), expected_uids):
        log('hybrid: exact attached visible-test features reused')
        return cached
    if time.time() - T0 > HYB_START_CUTOFF_S:
        raise TimeoutError('insufficient runtime reserve for hybrid feature extraction')
    slots, study_meta = _hyb_build_slots(series_df)
    backbone = AutoModel.from_pretrained(str(_hyb_find_dino_small()), local_files_only=True, trust_remote_code=False)
    if int(backbone.config.hidden_size) != 384:
        raise AssertionError('the hybrid specialist requires DINOv2-small hidden size 384')
    model = _HybridDinoEncoder(backbone).eval().to(dev)
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    mean = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)

    @torch.inference_mode()
    def encode(images):
        outputs = []
        for start in range(0, len(images), 64):
            batch = images[start:start + 64].to(dev, non_blocking=True).float().div_(255.0)
            batch = (batch - mean) / std
            with torch.autocast('cuda', dtype=torch.float16, enabled=dev.type == 'cuda'):
                outputs.append(model(batch).float().cpu())
        return torch.cat(outputs, dim=0)
    n = len(test_df)
    features = np.zeros((n, 6, 3456), np.float16)
    masks = np.zeros((n, 6), bool)
    slot_meta_array = np.zeros((n, 6, 8), np.float16)
    radiomics_array = np.zeros((n, 6, 56), np.float16)
    study_meta_array = np.zeros((n, 13), np.float16)
    sexes = np.zeros(n, np.int8)
    ids = expected_uids.astype(object)

    def safe_load(index):
        try:
            return _hyb_load_study(test_df.iloc[index], slots, study_meta)
        except Exception as exc:
            uid = str(test_df.iloc[index]['StudyInstanceUID'])
            log(f'hybrid study decode failed safely: {uid}: {exc}')
            return (np.zeros((6, 5, 3, 224, 224), np.uint8), np.zeros((6, 5), bool), np.zeros((6, 8), np.float32), np.zeros((6, 56), np.float32), np.zeros(13, np.float32), 0, uid)
    workers = max(1, min(8, os.cpu_count() or 8))
    with ThreadPoolExecutor(max_workers=workers) as executor:
        for start in range(0, n, 6):
            if time.time() - T0 > HYB_DEADLINE_S:
                raise TimeoutError('hybrid deadline reached during feature extraction')
            stop = min(start + 6, n)
            items = list(executor.map(safe_load, range(start, stop)))
            image_batch = torch.from_numpy(np.stack([item[0] for item in items]))
            view_mask = torch.from_numpy(np.stack([item[1] for item in items]))
            valid = view_mask.reshape(-1)
            view_features = torch.zeros(len(items) * 30, 1152, dtype=torch.float32)
            if valid.any():
                flat_images = image_batch.reshape(-1, 3, 224, 224)
                view_features[valid] = encode(flat_images[valid])
            view_features = view_features.reshape(len(items), 6, 5, 1152)
            aggregate = torch.zeros(len(items), 6, 3456, dtype=torch.float32)
            slot_mask = view_mask.any(dim=2)
            for batch_index in range(len(items)):
                for slot_index in range(6):
                    present = view_mask[batch_index, slot_index]
                    if present.any():
                        values = view_features[batch_index, slot_index, present]
                        aggregate[batch_index, slot_index] = torch.cat([values.mean(0), values.amax(0), values.std(0, unbiased=False)])
            features[start:stop] = aggregate.numpy().astype(np.float16)
            masks[start:stop] = slot_mask.numpy()
            slot_meta_array[start:stop] = np.stack([item[2] for item in items]).astype(np.float16)
            radiomics_array[start:stop] = np.stack([item[3] for item in items]).astype(np.float16)
            study_meta_array[start:stop] = np.stack([item[4] for item in items]).astype(np.float16)
            sexes[start:stop] = np.asarray([item[5] for item in items], np.int8)
            if start == 0 or stop % 100 == 0 or stop == n:
                log(f'hybrid features {stop}/{n}')
    del model, backbone
    gc.collect()
    if dev.type == 'cuda':
        torch.cuda.empty_cache()
    return (features, masks, slot_meta_array, radiomics_array, study_meta_array, sexes, ids)

def _hyb_align_bundle(bundle, expected_uids):
    ids = np.asarray(bundle[-1]).astype(str)
    expected_uids = np.asarray(expected_uids).astype(str)
    if np.array_equal(ids, expected_uids):
        return bundle
    if len(set(ids)) != len(ids) or set(ids) != set(expected_uids):
        raise AssertionError('hybrid cache StudyInstanceUID set mismatch')
    positions = {uid: index for index, uid in enumerate(ids)}
    order = np.asarray([positions[uid] for uid in expected_uids], dtype=int)
    return tuple((np.asarray(array)[order] for array in bundle[:-1])) + (expected_uids,)

def _hyb_transform(bundle, pca):
    features, masks, slot_meta, radiomics, study_meta, sex, _ = bundle
    n = len(features)
    result = np.zeros((n, 6, 192), np.float32)
    for start in range(0, n, 96):
        stop = min(start + 96, n)
        block = np.asarray(features[start:stop], np.float32)
        block_mask = np.asarray(masks[start:stop]).reshape(-1).astype(bool)
        flat = block.reshape(-1, block.shape[-1])
        transformed = np.zeros((len(flat), 192), np.float32)
        if block_mask.any():
            transformed[block_mask] = pca.transform(flat[block_mask]).astype(np.float32)
        result[start:stop] = transformed.reshape(stop - start, 6, 192)
    sex_onehot = np.eye(3, dtype=np.float32)[np.asarray(sex, dtype=int).clip(0, 2)]
    matrix = np.concatenate([result.reshape(n, -1), np.asarray(masks, np.float32), np.asarray(slot_meta, np.float32).reshape(n, -1), np.asarray(radiomics, np.float32).reshape(n, -1), np.asarray(study_meta, np.float32), sex_onehot], axis=1)
    matrix = np.nan_to_num(matrix, nan=0.0, posinf=0.0, neginf=0.0)
    if matrix.shape != (n, 1558):
        raise AssertionError(f'unexpected hybrid matrix shape {matrix.shape}')
    return matrix.astype(np.float32)

def _hyb_rank(values, denominator_offset=0.0):
    values = np.asarray(values, np.float64)
    if len(values) <= 1 or np.ptp(values) < 1e-12:
        return np.full(len(values), 0.5, np.float64)
    return rankdata(values, method='average') / (len(values) + denominator_offset)

def _hyb_payload():
    raw = zlib.decompress(base64.b64decode(HYB_TEACHER_PAYLOAD.encode('ascii')))
    with np.load(io.BytesIO(raw), allow_pickle=False) as payload:
        return {name: payload[name].astype(np.float32) for name in payload.files}

def _hyb_select_top(indices, scores, labels, class_value, cap=2200):
    keep = indices[labels == class_value]
    if len(keep) <= cap:
        return keep
    keep_scores = scores[labels == class_value]
    return keep[np.argsort(-keep_scores)[:cap]]

def _hyb_training_arrays(pseudo_y, pseudo_conf, exact_mask, exact_y):
    exact_idx = np.flatnonzero(exact_mask)
    pseudo_pool = ~exact_mask
    confidence = np.clip(pseudo_conf, 0.0, 1.0)
    candidates = np.flatnonzero(pseudo_pool & (confidence >= 0.2))
    candidate_labels = (pseudo_y[candidates] >= 0.5).astype(int)
    pos = _hyb_select_top(candidates, confidence[candidates], candidate_labels, 1)
    neg = _hyb_select_top(candidates, confidence[candidates], candidate_labels, 0)
    pseudo_idx = np.concatenate([pos, neg]).astype(int)
    pseudo_labels = (pseudo_y[pseudo_idx] >= 0.5).astype(int)
    pseudo_weight = 0.18 + 1.15 * np.power(np.clip(confidence[pseudo_idx], 0, 1), 1.4)
    fit_idx = np.concatenate([exact_idx, pseudo_idx]).astype(int)
    fit_y = np.concatenate([exact_y[exact_idx].astype(int), pseudo_labels]).astype(int)
    fit_weight = np.concatenate([np.full(len(exact_idx), 7.0, np.float32), pseudo_weight.astype(np.float32)])
    return (fit_idx, fit_y, fit_weight, exact_idx, exact_y[exact_idx].astype(int))

def _hyb_fit_lr(x_fit, y_fit, x_test, weights, cs, seed):
    predictions = []
    for index, c_value in enumerate(cs):
        model = LogisticRegression(C=c_value, solver='liblinear', class_weight='balanced', max_iter=3000, random_state=seed + 31 * index)
        model.fit(x_fit, y_fit, sample_weight=weights)
        predictions.append(model.predict_proba(x_test)[:, 1])
    return np.mean(predictions, axis=0).astype(np.float32)

def _hyb_fit_family(family, x_train, fit_idx, fit_y, fit_weight, x_test, seed):
    if time.time() - T0 > HYB_DEADLINE_S:
        raise TimeoutError('hybrid deadline reached during model fitting')
    if family == 'lr':
        return _hyb_fit_lr(x_train[fit_idx], fit_y, x_test, fit_weight, HYB_LR_CS, seed)
    if family == 'exact_lr':
        return _hyb_fit_lr(x_train[fit_idx], fit_y, x_test, fit_weight, HYB_EXACT_LR_CS, seed)
    if family == 'et':
        model = ExtraTreesClassifier(n_estimators=420, max_features='sqrt', min_samples_leaf=4, min_samples_split=8, bootstrap=False, class_weight='balanced', random_state=seed, n_jobs=-1)
    elif family == 'hgb':
        model = HistGradientBoostingClassifier(learning_rate=0.035, max_iter=180, max_leaf_nodes=15, min_samples_leaf=18, l2_regularization=0.25, early_stopping=True, validation_fraction=0.15, random_state=seed)
    else:
        raise ValueError(f'unknown hybrid family {family}')
    model.fit(x_train[fit_idx], fit_y, sample_weight=fit_weight)
    return model.predict_proba(x_test)[:, 1].astype(np.float32)

def _hyb_teacher_arm(x_train, x_test, pseudo_y, pseudo_conf, exact_mask, exact_y, exact_lr):
    fit_idx, fit_y, fit_weight, _, _ = _hyb_training_arrays(pseudo_y, pseudo_conf, exact_mask, exact_y)
    predictions = {}
    for family in ('lr', 'et', 'hgb'):
        seed_predictions = []
        for seed in HYB_SEEDS:
            model_seed = seed + 101 * TARGETS.index(HYB_TARGET) + len(family)
            seed_predictions.append(_hyb_fit_family(family, x_train, fit_idx, fit_y, fit_weight, x_test, model_seed))
        predictions[family] = np.mean(np.stack(seed_predictions), axis=0)
    predictions['exact_lr'] = exact_lr
    weighted_rank = np.zeros(len(x_test), np.float64)
    weighted_prob = np.zeros(len(x_test), np.float64)
    for weight, family in zip(HYB_FAMILY_WEIGHTS, HYB_FAMILIES):
        pred = np.clip(predictions[family], 1e-05, 1.0 - 1e-05)
        weighted_rank += weight * _hyb_rank(pred, denominator_offset=1.0)
        weighted_prob += weight * pred
    return 0.9 * weighted_rank + 0.1 * weighted_prob

def run_hybrid_lm_and_lateral_oa_specialists():
    if time.time() - T0 > HYB_START_CUTOFF_S:
        log('hybrid skipped: insufficient runtime reserve')
        return False
    cache_dir = _hyb_find_cache_dir()
    primary_path = Path('submission.csv')
    primary = pd.read_csv(primary_path, dtype={'StudyInstanceUID': str})
    train_df = pd.read_csv(ROOT / 'train.csv', dtype={'StudyInstanceUID': str})
    test_df = pd.read_csv(ROOT / 'test.csv', dtype={'StudyInstanceUID': str})
    series_df = pd.read_csv(ROOT / 'test_series.csv', dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
    if primary.columns.tolist() != ['StudyInstanceUID'] + TARGETS:
        raise AssertionError('primary submission schema mismatch')
    train_uids = train_df['StudyInstanceUID'].astype(str).to_numpy()
    uid_hash = hashlib.sha256('\n'.join(train_uids).encode()).hexdigest()
    if uid_hash != HYB_EXPECTED_TRAIN_ID_SHA256:
        raise AssertionError('competition train StudyInstanceUID order drifted')
    test_uids = test_df['StudyInstanceUID'].astype(str).to_numpy()
    dev = DEVS[0]
    train_bundle = _hyb_align_bundle(_hyb_load_cached_split(cache_dir, 'train'), train_uids)
    test_bundle = _hyb_align_bundle(_hyb_extract_test_bundle(test_df, series_df, cache_dir, dev), test_uids)
    pca = joblib.load(cache_dir / f'{HYB_PREFIX}_ipca_192.joblib')
    x_train_raw = _hyb_transform(train_bundle, pca)
    x_test_raw = _hyb_transform(test_bundle, pca)
    joint = np.concatenate([x_train_raw, x_test_raw], axis=0)
    mean = joint.mean(axis=0, dtype=np.float64).astype(np.float32)
    scale = joint.std(axis=0, dtype=np.float64).astype(np.float32)
    scale[scale < 1e-05] = 1.0
    x_train = ((x_train_raw - mean) / scale).astype(np.float32)
    x_test = ((x_test_raw - mean) / scale).astype(np.float32)
    payload = _hyb_payload()
    if any((len(payload[name]) != len(train_df) for name in payload)):
        raise AssertionError('embedded hybrid teacher length mismatch')

    def fit_target(target_name, pseudo_y, pseudo_conf, seeds):
        exact_mask = train_df[target_name].notna().to_numpy()
        exact_y = np.nan_to_num(train_df[target_name].to_numpy(np.float32), nan=0.0)
        exact_idx = np.flatnonzero(exact_mask)
        exact_labels = exact_y[exact_idx].astype(int)
        exact_predictions = []
        for seed in seeds:
            model_seed = seed + 101 * TARGETS.index(target_name) + len('exact_lr')
            exact_predictions.append(_hyb_fit_family('exact_lr', x_train, exact_idx, exact_labels, np.full(len(exact_idx), 3.0, np.float32), x_test, model_seed))
        exact_lr = np.mean(np.stack(exact_predictions), axis=0)
        fit_idx, fit_y, fit_weight, _, _ = _hyb_training_arrays(pseudo_y, pseudo_conf, exact_mask, exact_y)
        predictions = {}
        for family in ('lr', 'et', 'hgb'):
            seed_predictions = []
            for seed in seeds:
                model_seed = seed + 101 * TARGETS.index(target_name) + len(family)
                seed_predictions.append(_hyb_fit_family(family, x_train, fit_idx, fit_y, fit_weight, x_test, model_seed))
            predictions[family] = np.mean(np.stack(seed_predictions), axis=0)
        predictions['exact_lr'] = exact_lr
        weighted_rank = np.zeros(len(x_test), np.float64)
        weighted_prob = np.zeros(len(x_test), np.float64)
        family_weights = HYB_LM_FAMILY_WEIGHTS if target_name == HYB_TARGET else HYB_OA_FAMILY_WEIGHTS
        for weight, family in zip(family_weights, HYB_FAMILIES):
            pred = np.clip(predictions[family], 1e-05, 1.0 - 1e-05)
            weighted_rank += weight * _hyb_rank(pred, denominator_offset=1.0)
            weighted_prob += weight * pred
        return 0.9 * weighted_rank + 0.1 * weighted_prob
    lm_consensus = fit_target(HYB_TARGET, payload['lm_consensus_y'], payload['lm_consensus_conf'], HYB_SEEDS)
    lm_pilkwang = fit_target(HYB_TARGET, payload['lm_pilkwang_y'], payload['lm_pilkwang_conf'], HYB_SEEDS)
    lm_teacher_rank = 1.0 * _hyb_rank(lm_consensus) + 0.0 * _hyb_rank(lm_pilkwang)
    oa_pilkwang = fit_target(HYB_OA_TARGET, payload['oa_pilkwang_y'], payload['oa_pilkwang_conf'], HYB_OA_SEEDS)
    oa_teacher_rank = _hyb_rank(oa_pilkwang)
    train_mean = x_train_raw.mean(axis=0, dtype=np.float64).astype(np.float32)
    train_scale = x_train_raw.std(axis=0, dtype=np.float64).astype(np.float32)
    train_scale[train_scale < 1e-05] = 1.0
    x_train_pca = ((x_train_raw - train_mean) / train_scale).astype(np.float32)
    x_test_pca = ((x_test_raw - train_mean) / train_scale).astype(np.float32)
    pca_specs = {HYB_TARGET: (128, 0.1), 'Synovitis': (32, 1.0)}
    pca_predictions = {target: [] for target in pca_specs}
    for pca_seed in (20260809, 20260819, 20260829, 20260839):
        decomposition = PCA(n_components=128, whiten=True, svd_solver='randomized', n_oversamples=20, random_state=pca_seed)
        train_embedding = decomposition.fit_transform(x_train_pca)
        test_embedding = decomposition.transform(x_test_pca)
        for target_name, (dimensions, c_value) in pca_specs.items():
            exact_mask = train_df[target_name].notna().to_numpy()
            exact_labels = train_df.loc[exact_mask, target_name].to_numpy(int)
            if int(exact_mask.sum()) != 58 or set(np.unique(exact_labels)) != {0, 1}:
                raise AssertionError(f'unexpected exact-label support for {target_name}')
            model = make_pipeline(StandardScaler(), LogisticRegression(C=c_value, solver='liblinear', class_weight='balanced', max_iter=5000, random_state=pca_seed))
            model.fit(train_embedding[exact_mask, :dimensions], exact_labels)
            pca_predictions[target_name].append(model.predict_proba(test_embedding[:, :dimensions])[:, 1])
    pca_predictions = {target: np.mean(np.stack(predictions), axis=0) for target, predictions in pca_predictions.items()}
    result = primary.copy()
    primary_uids = result['StudyInstanceUID'].astype(str)
    if set(primary_uids) != set(test_uids):
        raise AssertionError('hybrid and primary StudyInstanceUID sets differ')
    lm_by_uid = pd.Series(lm_teacher_rank, index=test_uids).reindex(primary_uids.values)
    oa_by_uid = pd.Series(oa_teacher_rank, index=test_uids).reindex(primary_uids.values)
    lm_pca_by_uid = pd.Series(_hyb_rank(pca_predictions[HYB_TARGET]), index=test_uids).reindex(primary_uids.values)
    syn_pca_by_uid = pd.Series(_hyb_rank(pca_predictions['Synovitis']), index=test_uids).reindex(primary_uids.values)
    lm_base_rank = result[HYB_TARGET].rank(pct=True).to_numpy(np.float64)
    oa_base_rank = result[HYB_OA_TARGET].rank(pct=True).to_numpy(np.float64)
    syn_base_rank = result['Synovitis'].rank(pct=True).to_numpy(np.float64)
    result[HYB_TARGET] = 0.125 * lm_base_rank + 0.5 * lm_by_uid.to_numpy(np.float64) + 0.375 * lm_pca_by_uid.to_numpy(np.float64)
    result[HYB_OA_TARGET] = 0.125 * oa_base_rank + 0.875 * oa_by_uid.to_numpy(np.float64)
    result['Synovitis'] = 0.75 * syn_base_rank + 0.25 * syn_pca_by_uid.to_numpy(np.float64)
    changed = {HYB_TARGET, HYB_OA_TARGET, 'Synovitis'}
    untouched = [target for target in TARGETS if target not in changed]
    if not result[untouched].equals(primary[untouched]):
        raise AssertionError('hybrid changed an unevaluated target')
    if result.shape != primary.shape or not np.isfinite(result[TARGETS].to_numpy()).all():
        raise AssertionError('invalid hybrid blend')
    temp_path = Path('submission_v34_hybrid.tmp.csv')
    result.to_csv(temp_path, index=False)
    reread = pd.read_csv(temp_path)
    if reread.shape != primary.shape or not np.isfinite(reread[TARGETS].to_numpy()).all():
        raise AssertionError('serialized hybrid blend is invalid')
    temp_path.replace(primary_path)
    log('hybrid complete: LM 0.125 base / 0.500 consensus / 0.375 PCA; Lateral OA 0.125 base / 0.875 Pilkwang; Synovitis 0.75 base / 0.25 PCA; nine other targets preserved')
    return True

### 提出ファイルの生成と、多段のフォールバック

**何をしているか**:

- `write_submission(pred, studies, test_df, path)` … 予測を
  **`rank(pct=True)` でパーセンタイル化**してから書き出し、
  `test_df` と左結合して**欠けた study を 0.5 で埋める**
- `write_benchmark_submission` … ベンチマーク用の提出
- `_v37_validate_submission` … スキーマ検証
- `run_yash_public_ensemble` … 公開アンサンブルの枝
- `main()` … 全体の司令塔

**なぜそうするのか**:

- **なぜランク化して出すか**: 評価が AUC である以上、
  **順位さえ保たれていれば値そのものに意味はありません**。
  パーセンタイルに揃えておけば、後段でこの提出を他の予測とブレンドするときに
  **スケールを気にせず加重平均できます**。実際 Stage 2/3 がまさにそれをやります。
- **なぜ `merge` + `fillna(0.5)` か**: 何らかの理由で予測できなかった study があった場合、
  行が欠けた提出はフォーマットエラーで**0点**になります。
  0.5（＝完全に中立な予測）で埋めれば、その study からは得点も失点もしませんが、
  **他の全 study の得点は生き残ります**。
  「一部の失敗を全体の失敗にしない」設計です。

`main()` が段階ごとに検証を挟みながら進む構造になっており、
**どの段階まで成功したか**が提出ファイルの状態から分かるようになっています。

In [ ]:
def write_submission(pred, studies, test_df, path):
    sub = pd.DataFrame(pd.DataFrame(pred).rank(pct=True).values, columns=TARGETS)
    sub.insert(0, 'StudyInstanceUID', studies)
    sub = test_df[['StudyInstanceUID']].merge(sub, on='StudyInstanceUID', how='left')
    sub[TARGETS] = sub[TARGETS].fillna(0.5)
    sub.to_csv(path, index=False)
    return sub

def write_benchmark_submission():
    t = pd.read_csv(ROOT / 'test.csv')
    for c in TARGETS:
        t[c] = 0.5
    t.to_csv('submission.csv', index=False)

def _v37_validate_submission(path, test_df, tag):
    path = Path(path)
    frame = pd.read_csv(path)
    expected = ['StudyInstanceUID'] + TARGETS
    if list(frame.columns) != expected:
        raise ValueError(f'{tag}: columns differ from the competition contract')
    if len(frame) != len(test_df) or not frame['StudyInstanceUID'].is_unique:
        raise ValueError(f'{tag}: row count or StudyInstanceUID uniqueness failed')
    if set(frame['StudyInstanceUID'].astype(str)) != set(test_df['StudyInstanceUID'].astype(str)):
        raise ValueError(f'{tag}: StudyInstanceUID set differs from test.csv')
    values = frame[TARGETS].to_numpy(np.float64)
    if not np.isfinite(values).all():
        raise ValueError(f'{tag}: non-finite prediction')
    return test_df[['StudyInstanceUID']].merge(frame, on='StudyInstanceUID', how='left')

def _v37_find_yash_submission():
    candidates = []
    local = globals().get('YASH_LOCAL_SOURCE_DIR')
    if local:
        candidates.append(Path(local) / 'submission.csv')
    root = Path('/kaggle/input')
    candidates.append(root / 'rsna-knee-infer-v1' / 'submission.csv')
    if root.is_dir():
        candidates.extend((meta.parent / 'submission.csv' for meta in root.glob('**/infer_meta.json')))
    seen = set()
    for path in candidates:
        key = str(path)
        if key in seen or not path.is_file():
            continue
        seen.add(key)
        meta_path = path.with_name('infer_meta.json')
        if meta_path.is_file():
            meta = json.loads(meta_path.read_text())
            if int(meta.get('errors', -1)) != 0:
                raise ValueError(f"Yash source reports {meta.get('errors')} inference errors")
        return path
    raise FileNotFoundError('the attached yashbishnoi98/rsna-knee-infer-v1 output is absent')

def run_yash_public_ensemble():
    import shutil
    test_df = pd.read_csv(ROOT / 'test.csv')
    native_path = Path('submission.csv')
    public_path = Path('submission_public_0899.csv')
    native = _v37_validate_submission(native_path, test_df, 'native V36')
    public = _v37_validate_submission(public_path, test_df, 'public DINO family')
    yash_path = _v37_find_yash_submission()
    yash = _v37_validate_submission(yash_path, test_df, 'Yash public image family')
    meta_path = yash_path.with_name('infer_meta.json')
    if meta_path.is_file():
        meta = json.loads(meta_path.read_text())
        if int(meta.get('studies', -1)) != len(test_df):
            raise ValueError('Yash source study count differs from test.csv')
    shutil.copyfile(native_path, 'submission_native_v36.csv')
    shutil.copyfile(yash_path, 'submission_yash_reference.csv')
    yr = yash[TARGETS].rank(pct=True).to_numpy(np.float64)
    dr = public[TARGETS].rank(pct=True).to_numpy(np.float64)
    blend = 0.55 * yr + 0.45 * dr
    result = test_df[['StudyInstanceUID']].copy()
    result[TARGETS] = blend
    if result.shape != yash.shape or not np.isfinite(result[TARGETS].to_numpy()).all():
        raise AssertionError('invalid Yash/DINO rank blend')
    candidate_path = Path('submission_yash_dino_rankblend.csv')
    result.to_csv(candidate_path, index=False)
    reread = _v37_validate_submission(candidate_path, test_df, 'Yash/DINO rank blend')
    changed = sum((tuple(reread[target].rank(method='first')) != tuple(yash[target].rank(method='first')) for target in TARGETS))
    if changed == 0:
        raise AssertionError('Yash/DINO blend is rank-identical to its Yash parent')
    temp_path = Path('submission_v37_yash_dino.tmp.csv')
    reread.to_csv(temp_path, index=False)
    temp_path.replace(native_path)
    log(f'Yash public family banked; V37 primary = 0.55 Yash / 0.45 public DINO rank blend ({changed} target orderings differ from Yash); exact Yash and native V36 outputs retained')
    return True

def main():
    write_benchmark_submission()
    pkg = find_weights()
    if pkg is not None:
        dev = DEVS[0]
        infer_from_package(pkg, dev)
        try:
            test_df = pd.read_csv(ROOT / 'test.csv')
            native_path = Path('submission.csv')
            public_path = Path('submission_public_0899.csv')
            native = _v37_validate_submission(native_path, test_df, 'native 24-member')
            public = _v37_validate_submission(public_path, test_df, 'public DINO frontier')
            native.to_csv('submission_native_v38.csv', index=False)
            public.to_csv(native_path, index=False)
            promoted = _v37_validate_submission(native_path, test_df, 'V40 primary')
            if not promoted.equals(public):
                raise AssertionError('V40 serialization differs from validated public frontier')
            log('V40 primary = exact no-jitter public-frontier target pooling; native 24-member output retained')
        except Exception as public_frontier_error:
            log(f'public-frontier promotion skipped safely: {public_frontier_error}')
            traceback.print_exc()
        log('done')
        return
    read_labels(pd.read_csv(ROOT / 'train.csv', usecols=['StudyInstanceUID', 'Report']))
    test_df = pd.read_csv(ROOT / 'test.csv')
    test_series = pd.read_csv(ROOT / 'test_series.csv')
    train_df = pd.read_csv(ROOT / 'train.csv')
    train_series = pd.read_csv(ROOT / 'train_series.csv')
    log(f'train {train_df.shape} test {test_df.shape}')
    both = pd.concat([train_series, test_series])
    plane_map = dict(zip(both['SeriesInstanceUID'], both['Anatomical_Plane']))
    log('header pass: test')
    hte = annotate(walk('test_series'))
    log(f'  {len(hte)} test series')
    log('header pass: train')
    htr = annotate(walk('train_series'))
    log(f'  {len(htr)} train series')
    slots_te, slots_tr = (pick_slots(hte, plane_map), pick_slots(htr, plane_map))
    cov = pd.Series([len(v) for v in slots_tr.values()]).describe()
    log(f"train slots per study: mean {cov['mean']:.2f} min {cov['min']:.0f} max {cov['max']:.0f}")
    st_tr, Ctr, Mtr = build_cache(slots_tr, plane_map, lat_of(htr, 'train '), 'train')
    st_te, Cte, Mte = build_cache(slots_te, plane_map, lat_of(hte, 'test '), 'test')
    t_lab = time.time()
    lab = read_labels(train_df)
    log(f'derived labels for {len(lab)} studies in {time.time() - t_lab:.1f}s')
    gold = train_df.set_index('StudyInstanceUID')[TARGETS]
    gold = gold[gold.notna().all(axis=1)]
    Y = np.zeros((len(st_tr), len(TARGETS)), np.float32)
    W = np.zeros_like(Y)
    for i, st in enumerate(st_tr):
        if st in gold.index:
            Y[i], W[i] = (gold.loc[st].values, 3.0)
        elif st in lab.index:
            r = lab.loc[st]
            Y[i] = r[TARGETS].values
            W[i] = 0.25 + 0.75 * r[[t + '__conf' for t in TARGETS]].values
    keep = np.where(W.sum(1) > 0)[0]
    log(f'supervised {len(keep)} of {len(st_tr)} studies (annotated {len(gold)})')
    import hashlib
    rep = train_df.set_index('StudyInstanceUID')['Report'].fillna('')
    grp = np.array([int(hashlib.md5(rep.get(s, s).encode()).hexdigest()[:8], 16) % 5 for s in st_tr])
    va = np.array([i for i in keep if grp[i] == 0])
    tr = np.array([i for i in keep if grp[i] != 0])
    if len(va) == 0 or len(tr) < BATCH_STUDIES:
        cut = max(1, len(keep) // 5)
        va, tr = (keep[:cut], keep[cut:])
    log(f'train {len(tr)} / holdout {len(va)} studies')
    gpos = {s: i for i, s in enumerate(st_tr)}
    va_set = set(va.tolist())
    gi = np.array([gpos[s] for s in gold.index if s in gpos and gpos[s] in va_set])
    gold_y = gold.loc[[st_tr[i] for i in gi]].values.astype(int) if len(gi) else None
    yv = (Y[va] > 0.5).astype(int)
    log(f'annotation check: {len(gi)} of {len(gold)} annotated studies are in the holdout')
    dev = DEVS[0]
    results, test_preds = ({}, {})
    for cfg in RUNS:
        pitch = CROP_MM / cfg['img']
        log(f"=== {cfg['name']}: {cfg['img']} px, {pitch:.3f} mm/pixel, {pitch * 14:.2f} mm per patch token ===")
        torch.manual_seed(SEED)
        model = build_model(UNFREEZE_LAST).to(dev)
        opt = torch.optim.AdamW([{'params': [p for p in model.backbone.parameters() if p.requires_grad], 'lr': LR_BACKBONE}, {'params': model.head.parameters(), 'lr': LR_HEAD}], weight_decay=WEIGHT_DECAY)
        steps = max(EPOCHS * (len(tr) // BATCH_STUDIES), 1)
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[LR_BACKBONE, LR_HEAD], total_steps=steps, pct_start=0.15)
        scaler = torch.amp.GradScaler('cuda', enabled=dev.type == 'cuda')
        best, best_state, best_annot = (-1.0, None, float('nan'))
        for ep in range(EPOCHS):
            model.train()
            perm = np.random.permutation(tr)
            tot, nstep = (0.0, 0)
            for b in range(0, len(perm) - BATCH_STUDIES + 1, BATCH_STUDIES):
                sel = perm[b:b + BATCH_STUDIES]
                rows = torch.from_numpy(Ctr[sel]).to(dev)
                g = int(torch.randint(N_GROUP, (1,)).item())
                imgs = augment(take_group(rows, g))
                m = torch.from_numpy(Mtr[sel]).to(dev)
                y = torch.from_numpy(Y[sel]).to(dev)
                w = torch.from_numpy(W[sel]).to(dev)
                with torch.autocast('cuda', enabled=dev.type == 'cuda'):
                    loss = (F.binary_cross_entropy_with_logits(model(imgs, m, cfg['img']), y, reduction='none') * w).mean()
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
                sched.step()
                tot += loss.item()
                nstep += 1
            pv = predict(model, Ctr, Mtr, va, dev, cfg['img'])
            d = macro_auc(yv, pv)
            g_auc = float('nan')
            if gold_y is not None and len(gi):
                g_auc = macro_auc(gold_y, predict(model, Ctr, Mtr, gi, dev, cfg['img']))
            log(f'  epoch {ep + 1}/{EPOCHS}  loss {tot / max(nstep, 1):.4f}  holdout {d:.4f}  annot(n={len(gi)}) {g_auc:.4f}')
            if d > best:
                best, best_annot = (d, g_auc)
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if time.time() - T0 > TIME_BUDGET:
                log('  time budget reached')
                break
        if best_state is not None:
            model.load_state_dict(best_state)
        results[cfg['name']] = (best, best_annot)
        test_preds[cfg['name']] = predict(model, Cte, Mte, np.arange(len(st_te)), dev, cfg['img'])
        log(f"  {cfg['name']}: best holdout {best:.4f} (annot {best_annot:.4f})")
        del model, opt, sched, scaler, best_state
        gc.collect()
        if dev.type == 'cuda':
            torch.cuda.empty_cache()
    log('---- summary ----')
    for n, (d, g_auc) in results.items():
        log(f'  {n:12s} holdout {d:.4f}   annot {g_auc:.4f}')
    pick = max(results, key=lambda k: results[k][0])
    log(f'best on the holdout: {pick} ({results[pick][0]:.4f})')
    for name, pred in test_preds.items():
        sub = write_submission(pred, st_te, test_df, f'submission_{name}.csv')
        log(f'  submission_{name}.csv {sub.shape}; nulls {int(sub[TARGETS].isna().sum().sum())}')
    ens = np.mean([pd.DataFrame(p).rank(pct=True).values for p in test_preds.values()], axis=0)
    write_submission(ens, st_te, test_df, 'submission_rankmean.csv')
    log(f'  submission_rankmean.csv (rank mean of {len(test_preds)})')
    sub = write_submission(test_preds[pick], st_te, test_df, 'submission.csv')
    log(f'submission.csv = {pick}; {sub.shape}; nulls {int(sub[TARGETS].isna().sum().sum())}')
    print(sub.head().to_string())

### 最上位の例外ハンドラ：何があっても提出を残す

**何をしているか**: `main()` を `try` で囲み、

- `LabelSourceError`（ラベル取得の失敗）→ スタックトレースを出して**再送出（＝止める）**
- それ以外のあらゆる例外 → スタックトレースを出したうえで、
  **全ターゲットを 0.5 で埋めた提出を書き出す**

**なぜそうするのか**: Code Competition では、
notebook が例外で終了すると**提出そのものが成立しません**。
2ヶ月かけたパイプラインが、テストデータ側の1件の壊れたDICOMで全滅する——
実際によくある事故です。

「全部 0.5」の提出は AUC 0.5（ランダム）ですが、**0点ではありません**。
そして何より、**提出が成立したという事実**によって
「コードは最後まで走った、ただし例外を踏んだ」という情報が得られます。

注目したいのは**エラーの種類で扱いを分けている**ことです。
`LabelSourceError` は再送出して止めます。
ラベルソースが取れないのは**設定ミス（データセットの添付忘れ等）**であり、
これを 0.5 で握り潰すと、**気づかないまま無意味な提出を続ける**ことになります。
「握り潰してよい失敗」と「気づくべき失敗」を区別するのは、
例外処理を設計するうえで最も重要な判断です。

> ⚠️ 初心者への注意: `except Exception:` で全部拾うのは一般には悪い習慣です。
> ここで許容されるのは、**確実にログを出したうえで、
> 意味のあるフォールバックを行っている**からです。
> ログも出さずに `pass` するのは、原因究明を不可能にする最悪のパターンです。

In [ ]:
try:
    main()
except LabelSourceError:
    traceback.print_exc()
    raise
except Exception:
    traceback.print_exc()
    t = pd.read_csv(find_root() / 'test.csv')
    for c in TARGETS:
        t[c] = 0.5
    t.to_csv('submission.csv', index=False)
    print('wrote fallback submission.csv')
log('done')

## 6. Stage 2: Cross-Series Spatial Attention Arm A5 (Target 0.910)

This module evaluates inter-series cross-attention across full-frame MRI views to capture long-range joint alignments missed by slot-isolated encoders.

When `knee-mri-fold-weights` is attached, it evaluates the 5-fold cross-series models and blends with $W = 0.45$:

$$\mathbf{R}_{\text{A5}} = 0.55 \cdot \mathbf{R}_{\text{E2}} + 0.45 \cdot \mathbf{R}_{\text{cross}}$$

If absent, the pipeline fail-safes cleanly and preserves the verified **0.899** baseline.


### Stage 2 の準備：名前空間を保存して環境をリセットする

**何をしているか**: `_A5_SAVED = dict(globals())` で
**現在のグローバル変数を丸ごと退避**してから、Stage 2 用のライブラリを読み込み、
`cv2.setNumThreads(1)` を設定します。

**なぜそうするのか**: このnotebookは、
**もともと別々に開発された3つのパイプラインを1つのnotebookに縦に連結**したものです。
それぞれが `CROP`, `IMG`, `SLOTS` のような**同じ名前のグローバル変数**を使っており、
そのまま並べると後の定義が前を上書きして、
**Stage 1 の関数が Stage 2 の設定で動く**という事故が起きます。

`dict(globals())` でスナップショットを取っておけば、必要なときに復元できます。
（本来はモジュールやクラスで名前空間を分けるべきですが、
notebookという媒体の制約下での現実的な回避策です。）

`cv2.setNumThreads(1)` は Stage 1 と同じ理由です。
このあと `ProcessPoolExecutor` でプロセス並列するので、
**各プロセス内でOpenCVがさらにスレッドを立てないよう**に抑えています。

In [ ]:
_A5_SAVED = dict(globals())
import gc, os, time, warnings
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import pydicom
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings('ignore')
cv2.setNumThreads(1)
CROP_MM = 130.0
SIZE = 336
SLICE_BAND = (0.12, 0.88)
N_SLICE = 16
INTENSITY = 'slice'
SLOTS = [('Sagittal', 1), ('Sagittal', 0), ('Coronal', 1), ('Coronal', 0), ('Axial', 1), ('Axial', 0)]
N_SLOT = len(SLOTS)
LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']

def _find_dir(*names):
    root = Path('/kaggle/input')
    cand = []
    for n in names:
        cand += [root / n, root / 'competitions' / n, root / 'datasets' / n]
        for parent in (root / 'datasets', root / 'competitions', root):
            if parent.is_dir():
                try:
                    cand += [d / n for d in parent.iterdir() if d.is_dir()]
                except OSError:
                    pass
    for p in cand:
        if p.is_dir():
            return p
    return None
COMP = _find_dir('rsna-knee-abnormality-detection')
CKPT = _find_dir('knee-mri-fold-weights')
assert COMP is not None, 'competition data not attached'
assert CKPT is not None, 'fold weights not attached'
assert (COMP / 'sample_submission.csv').exists(), f'no competition data at {COMP}'
assert list(CKPT.glob('*_f*.pt')), f'no checkpoints at {CKPT}'
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'competition : {COMP}')
print(f'checkpoints : {CKPT}')
print(f'device      : {DEV}')
for i in range(torch.cuda.device_count() if DEV == 'cuda' else 0):
    cc = torch.cuda.get_device_capability(i)
    print(f'  gpu{i}       : {torch.cuda.get_device_name(i)} sm_{cc[0]}{cc[1]}, {torch.cuda.get_device_properties(i).total_memory / 2 ** 30:.0f} GiB, native bf16={cc >= (8, 0)}')

### Stage 2 の前処理：ウィンドウ処理と study 単位のテンソル構築

**何をしているか**:

- `ordered_files(sdir, cap=64)` … `InstanceNumber` で並べ、**最大64枚に制限**
- `series_side` … 左右の判定
- `read_crop` … DICOM を読んでクロップ
- `window(...)` … 輝度のウィンドウ処理
- `render` / `build_study` … study 単位の入力テンソルを組む

**なぜそうするのか**:

- **`cap=64` の意味**: スライス数が極端に多いシリーズ（3Dシーケンス等）に対する
  **実行時間の保険**です。全部読むと1件で数分かかることがあり、
  2,000件の test に対しては現実的でありません。
  情報を捨てる代わりに**完走を保証する**というトレードオフです。
- **ウィンドウ処理（windowing）とは**: 医用画像の輝度範囲のうち
  **見たい範囲だけを 0〜255 に伸張する**操作です。
  CTのHU値ほど標準化されていないMRIでも、
  パーセンタイルベースで同じことをすると、**施設間の輝度差が大きく縮まります**。
- **`InstanceNumber` を使う点**: Stage 1 が
  `ImagePositionPatient` の射影で厳密に並べていたのに対し、Stage 2 はより単純な方法です。
  多くの場合これで十分ですが、**厳密さでは劣ります**。
  同じnotebook内で流儀が違うのは、別々に開発されたパイプラインを繋いだ痕跡でもあります。
  改善余地として `README.md` にも記載しました。

In [ ]:
SERIES_ROOT = COMP / 'test_series'
if not SERIES_ROOT.exists():
    SERIES_ROOT = COMP / 'train_series'
print('series root:', SERIES_ROOT)

def ordered_files(sdir, cap=64):
    keyed = []
    for f in sdir.glob('*.dcm'):
        try:
            ds = pydicom.dcmread(str(f), stop_before_pixels=True)
            keyed.append((int(ds.InstanceNumber), str(f)))
        except Exception:
            continue
        if len(keyed) >= cap * 4:
            break
    return [f for _, f in sorted(keyed)]

def series_side(path):
    try:
        return float(pydicom.dcmread(path, stop_before_pixels=True).ImagePositionPatient[0])
    except Exception:
        return 0.0

def read_crop(path):
    try:
        ds = pydicom.dcmread(path)
        arr = ds.pixel_array.astype(np.float32)
    except Exception:
        return None
    try:
        ps = float(ds.PixelSpacing[0])
    except Exception:
        ps = CROP_MM / max(arr.shape)
    half = int(round(CROP_MM / ps / 2))
    cy, cx = (arr.shape[0] // 2, arr.shape[1] // 2)
    y0, y1 = (max(0, cy - half), min(arr.shape[0], cy + half))
    x0, x1 = (max(0, cx - half), min(arr.shape[1], cx + half))
    crop = arr[y0:y1, x0:x1]
    return None if crop.size == 0 else crop

def window(crop, lo, hi, flip):
    c = np.clip((crop - lo) / max(hi - lo, 1e-06), 0, 1)
    img = cv2.resize(c, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    return img[:, ::-1].copy() if flip else img

def render(path, flip):
    crop = read_crop(path)
    if crop is None:
        return None
    lo, hi = np.percentile(crop[::4, ::4], [1, 99])
    return window(crop, lo, hi, flip)

def build_study(args):
    idx, study, recs = args
    out = np.zeros((N_SLOT, N_SLICE, SIZE, SIZE), np.uint8)
    mask = np.zeros(N_SLOT, np.uint8)
    rows = pd.DataFrame(recs)
    if len(rows):
        for s_i, (plane, fs) in enumerate(SLOTS):
            sub = rows[(rows.Anatomical_Plane == plane) & (rows.Fat_Suppression == fs)]
            if sub.empty:
                continue
            files = ordered_files(SERIES_ROOT / study / sub.iloc[0].SeriesInstanceUID)
            if not files:
                continue
            flip = plane != 'Sagittal' and series_side(files[0]) < 0
            lo, hi = SLICE_BAND
            i0 = int(round(lo * (len(files) - 1)))
            i1 = int(round(hi * (len(files) - 1)))
            avail = list(range(i0, i1 + 1))
            if len(avail) >= N_SLICE:
                picks = [avail[int(round(t))] for t in np.linspace(0, len(avail) - 1, N_SLICE)]
                off = 0
            else:
                picks, off = (avail, (N_SLICE - len(avail)) // 2)
            if INTENSITY == 'series':
                crops = [read_crop(files[p]) for p in picks]
                got = [x for x in crops if x is not None]
                if got:
                    samp = np.concatenate([x[::4, ::4].ravel() for x in got])
                    lo_, hi_ = np.percentile(samp, [1, 99])
                    for c, x in enumerate(crops):
                        if x is None:
                            x = read_crop(files[min(len(files) - 1, picks[c] + 1)])
                        if x is not None:
                            out[s_i, off + c] = (window(x, lo_, hi_, flip) * 255).astype(np.uint8)
            else:
                for c, p in enumerate(picks):
                    img = render(files[p], flip)
                    if img is None:
                        img = render(files[min(len(files) - 1, p + 1)], flip)
                    if img is not None:
                        out[s_i, off + c] = (img * 255).astype(np.uint8)
            mask[s_i] = len(picks)
    return (idx, out, mask)
sub_df = pd.read_csv(COMP / 'sample_submission.csv')
ser_csv = pd.read_csv(COMP / 'test_series.csv')
if not (COMP / 'test_series').exists():
    ser_csv = pd.read_csv(COMP / 'train_series.csv')
ser_csv = ser_csv.loc[:, ~ser_csv.columns.duplicated()]
studies = sub_df.StudyInstanceUID.tolist()
by = {s: g.to_dict('records') for s, g in ser_csv[ser_csv.StudyInstanceUID.isin(set(studies))].groupby('StudyInstanceUID')}
print(f'{len(studies):,} test studies, {len(by):,} with series metadata')

### Stage 2 のモデル群：様々なプーリングとスロット深度ミキサー

**何をしているか**: 系列間クロスアテンションのためのモジュール群です。

- `segment_softmax` … 可変長のセグメントごとに softmax を取る
  （`scatter_reduce` で各バッチの最大値を引いてから exp する数値安定化つき）
- `MeanMaxPool` … 平均プーリングと最大プーリングを**連結**する
- `LabelAttentionPool` / `TokenXAttnPool` … ラベル別・トークン別のアテンションプーリング
- `ViTSlotToken` … スロットトークンを持つ ViT
- `DepthCompress` / `SlotDepthMixer` … スライス方向（深さ）の情報を混ぜる
- `_GatedDelta` … ゲート付きの差分結合

**なぜそうするのか**:

- **Mean-Max デュアルプーリング**: 平均は**全体の傾向**（びまん性の変化）を、
  最大は**最も強い反応**（限局病変）を捉えます。
  どちらか一方に決めず**両方を連結して次の層に渡す**ことで、
  「どちらをどれだけ使うか」をモデル自身に学ばせています。
  ターゲット別TTAプーリングを**手で決める**代わりに、
  **学習で決める**版だと考えると分かりやすい。
- **`segment_softmax` の数値安定化**: softmax は
  指数関数を使うため、大きな値でオーバーフローします。
  **各セグメントの最大値を引いてから exp する**のは、結果を変えずに桁溢れを防ぐ標準テクニックです。
  可変長セグメント（study ごとにスライス数が違う）に対して
  `scatter_reduce` で最大値を求めている実装は、書けると応用が利きます。
- **深さ方向のミキシング**: 2Dのバックボーンはスライス間の関係を見られません。
  `SlotDepthMixer` はスライス方向に軽い畳み込み／混合を入れることで、
  **3Dモデルの計算コストを払わずに3Dの文脈を少しだけ入れる**折衷案です。

In [ ]:
N_SLOT_TYPES, MASK_IDX = (6, 0)

def segment_softmax(scores, sidx, B):
    T, K = scores.shape
    idx = sidx.unsqueeze(1).expand(-1, K)
    m = torch.full((B, K), float('-inf'), device=scores.device, dtype=scores.dtype)
    m = m.scatter_reduce(0, idx, scores, reduce='amax', include_self=True)
    e = (scores - m[sidx]).exp()
    s = torch.zeros(B, K, device=scores.device, dtype=scores.dtype).index_add_(0, sidx, e)
    return e / s[sidx].clamp(min=1e-06)

class MeanMaxPool(nn.Module):

    def forward(self, f, sidx, B, slot=None, return_attn=False):
        D = f.shape[1]
        cnt = torch.zeros(B, device=f.device, dtype=f.dtype).index_add_(0, sidx, torch.ones(f.shape[0], device=f.device, dtype=f.dtype))
        mean = torch.zeros(B, D, device=f.device, dtype=f.dtype).index_add_(0, sidx, f)
        mean = mean / cnt.clamp(min=1).unsqueeze(1)
        mx = torch.full((B, D), -10000.0, device=f.device, dtype=f.dtype)
        mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), f, reduce='amax', include_self=True)
        return (torch.cat([mean, mx], 1), None)

class LabelAttentionPool(nn.Module):

    def __init__(self, d, n_labels=12, n_heads=4, slot_bias=True):
        super().__init__()
        self.d, self.k, self.h = (d, n_labels, n_heads)
        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.key, self.val = (nn.Linear(d, d), nn.Linear(d, d))
        self.slot_bias = nn.Parameter(torch.zeros(n_labels, N_SLOT_TYPES + 1)) if slot_bias else None

    def forward(self, f, sidx, B, slot=None, return_attn=False):
        scores = self.key(f) @ self.q.t() / self.d ** 0.5
        if self.slot_bias is not None and slot is not None:
            scores = scores + self.slot_bias.t()[slot]
        a = segment_softmax(scores, sidx, B)
        out = torch.zeros(B, self.k, self.d, device=f.device, dtype=f.dtype)
        out = out.index_add_(0, sidx, a.unsqueeze(-1) * self.val(f).unsqueeze(1))
        return (out, a)

class TokenXAttnPool(nn.Module):

    def __init__(self, d, n_labels=12, n_heads=6, dropout=0.2):
        super().__init__()
        self.d, self.k = (d, n_labels)
        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.slot_emb = nn.Embedding(N_SLOT_TYPES + 1, d, padding_idx=0)
        self.kv_norm = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)

    def forward(self, tok, sidx, B, slot=None, return_attn=False):
        T, N, D = tok.shape
        cnt = torch.bincount(sidx, minlength=B)
        S = int(cnt.max().item())
        starts = torch.cumsum(cnt, 0) - cnt
        pos = torch.arange(T, device=tok.device) - starts[sidx]
        kv = tok + self.slot_emb(slot).unsqueeze(1)
        pad = tok.new_zeros(B, S, N, D)
        pad[sidx, pos] = kv
        keep = torch.zeros(B, S, dtype=torch.bool, device=tok.device)
        keep[sidx, pos] = True
        kpm = ~keep.repeat_interleave(N, dim=1)
        pad = self.kv_norm(pad.reshape(B, S * N, D))
        q = self.q.unsqueeze(0).expand(B, -1, -1)
        att, w = self.attn(q, pad, pad, key_padding_mask=kpm, need_weights=return_attn, average_attn_weights=True)
        cls = tok[:, 0]
        mean = torch.zeros(B, D, device=tok.device, dtype=tok.dtype).index_add_(0, sidx, cls) / cnt.clamp(min=1).unsqueeze(1)
        mx = torch.full((B, D), -10000.0, device=tok.device, dtype=tok.dtype)
        mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), cls, reduce='amax', include_self=True)
        base = torch.cat([mean, mx], 1).unsqueeze(1).expand(-1, self.k, -1)
        return (torch.cat([att, base], -1), w)

class ViTSlotToken(nn.Module):

    def __init__(self, vit, n_cat, dim=None):
        super().__init__()
        self.vit = vit
        d = dim or vit.embed_dim
        self.tok = nn.Embedding(n_cat + 1, d, padding_idx=MASK_IDX)
        self.num_features = vit.num_features
        self._orig_prefix = getattr(vit, 'num_prefix_tokens', 1)
        vit.num_prefix_tokens = self._orig_prefix + 1
        for blk in vit.blocks:
            a = getattr(blk, 'attn', None)
            if a is not None and hasattr(a, 'num_prefix_tokens'):
                a.num_prefix_tokens = a.num_prefix_tokens + 1

    @staticmethod
    def _maybe(mod, x):
        return x if mod is None else mod(x)

    def forward_features(self, x, cat):
        v = self.vit
        x = v.patch_embed(x)
        pos = v._pos_embed(x)
        rope = None
        if isinstance(pos, tuple):
            x, rope = pos
        else:
            x = pos
        x = self._maybe(getattr(v, 'patch_drop', None), x)
        x = self._maybe(getattr(v, 'norm_pre', None), x)
        npt = self._orig_prefix
        tok = self.tok(cat).unsqueeze(1)
        x = torch.cat([x[:, :npt], tok, x[:, npt:]], dim=1)
        if rope is not None:
            if getattr(v, 'rope_mixed', False):
                for i, blk in enumerate(v.blocks):
                    x = blk(x, rope=rope[i])
            else:
                for blk in v.blocks:
                    x = blk(x, rope=rope)
        else:
            x = v.blocks(x)
        return v.norm(x)

    def forward_head(self, x, pre_logits=True):
        return self.vit.forward_head(x, pre_logits=pre_logits)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

class _GatedDepthBlock(nn.Module):

    def __init__(self, n_slice, dropout=0.0, ls_init=0.1):
        super().__init__()
        self.norm = nn.GroupNorm(1, n_slice)
        self.v = nn.Conv2d(n_slice, n_slice, 1)
        self.g = nn.Conv2d(n_slice, n_slice, 1)
        self.out = nn.Conv2d(n_slice, n_slice, 1)
        self.gamma = nn.Parameter(torch.full((n_slice, 1, 1), ls_init))
        self.drop = nn.Dropout2d(dropout) if dropout else nn.Identity()

    def forward(self, x):
        z = self.norm(x)
        return x + self.gamma * self.drop(self.out(self.v(z) * F.silu(self.g(z))))

class DepthCompress(nn.Module):

    def __init__(self, n_slice=16, out_ch=3, depth=1, dropout=0.0, ls_init=0.1, imagenet=True, proj_noise=0.25):
        super().__init__()
        self.imagenet = imagenet
        self.blocks = nn.ModuleList([_GatedDepthBlock(n_slice, dropout, ls_init) for _ in range(depth)])
        self.proj = nn.Conv2d(n_slice, out_ch, 1, bias=True)
        if imagenet:
            self.register_buffer('mu', torch.tensor(IMAGENET_MEAN).view(1, -1, 1, 1))
            self.register_buffer('sd', torch.tensor(IMAGENET_STD).view(1, -1, 1, 1))

    def forward(self, x):
        keep = (x.amax(dim=1, keepdim=True) > 0).to(x.dtype)
        z = x
        for b in self.blocks:
            z = b(z)
        z = self.proj(z)
        if self.imagenet:
            z = (z - self.mu.to(z.dtype)) / self.sd.to(z.dtype)
        return z * keep
N_PLANE, N_CONTRAST = (3, 2)
_PLANE_OF = lambda s: torch.clamp(s - 1, 0, 5) // 2
_CONTRAST_OF = lambda s: torch.clamp(s - 1, 0, 5) % 2

class SlotDepthMixer(nn.Module):

    def __init__(self, n_slice=16, ksize=5, alpha_max=0.25):
        super().__init__()
        self.n_slice, self.ksize, self.r = (n_slice, ksize, ksize // 2)
        self.alpha_max = alpha_max
        b = torch.tensor([1.0, 4.0, 6.0, 4.0, 1.0])
        self.register_buffer('base', b.log()[self.r:])
        n_u = self.r + 1
        self.shared = nn.Parameter(torch.zeros(n_u))
        self.plane_k = nn.Parameter(torch.zeros(N_PLANE, n_u))
        self.contrast_k = nn.Parameter(torch.zeros(N_CONTRAST, n_u))
        self.g0 = nn.Parameter(torch.zeros(()))
        self.gate_p = nn.Parameter(torch.zeros(N_PLANE))
        self.gate_c = nn.Parameter(torch.zeros(N_CONTRAST))
        idx = torch.arange(n_slice)
        self.register_buffer('off', idx[None, :] - idx[:, None])

    def kernel(self, slot):
        p, c = (_PLANE_OF(slot), _CONTRAST_OF(slot))
        half = self.base + self.shared + self.plane_k[p] + self.contrast_k[c]
        full = torch.cat([half.flip(-1)[..., :self.r], half], dim=-1)
        return F.softmax(full, dim=-1)

    def alpha(self, slot):
        p, c = (_PLANE_OF(slot), _CONTRAST_OF(slot))
        return self.alpha_max * torch.tanh(self.g0 + self.gate_p[p] + self.gate_c[c])

    def forward(self, x, slot, vmask):
        T, S, H, W = x.shape
        if vmask is None:
            raise ValueError('stem=mixer requires the padding mask')
        k = self.kernel(slot)
        v = vmask.to(k.dtype)
        d = self.off + self.r
        inb = (d >= 0) & (d < self.ksize)
        kk = k[:, d.clamp(0, self.ksize - 1)] * inb
        M = kk * v[:, None, :]
        den = M.sum(-1, keepdim=True)
        eye = torch.eye(S, device=x.device, dtype=M.dtype).expand(T, S, S)
        ok = (den > 1e-06) & v[:, :, None].bool()
        M = torch.where(ok, M / den.clamp(min=1e-06), eye)
        a = self.alpha(slot)[:, None, None]
        Aop = ((1.0 - a) * eye + a * M).to(x.dtype)
        if x.is_contiguous(memory_format=torch.channels_last) and (not x.is_contiguous()):
            y = torch.bmm(x.permute(0, 2, 3, 1).reshape(T, H * W, S), Aop.transpose(1, 2))
            return y.reshape(T, H, W, S).permute(0, 3, 1, 2)
        return torch.bmm(Aop, x.reshape(T, S, H * W)).reshape(T, S, H, W)

def _seg_mean_max(v, sidx, B):
    D = v.shape[1]
    cnt = torch.zeros(B, device=v.device, dtype=v.dtype).index_add_(0, sidx, torch.ones(v.shape[0], device=v.device, dtype=v.dtype))
    mean = torch.zeros(B, D, device=v.device, dtype=v.dtype).index_add_(0, sidx, v)
    mean = mean / cnt.clamp(min=1).unsqueeze(1)
    mx = torch.full((B, D), -10000.0, device=v.device, dtype=v.dtype)
    mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), v, reduce='amax', include_self=True)
    return torch.cat([mean, mx], 1)

def _pad_kv(x, sidx, B, norm):
    T, P, D = x.shape
    cnt = torch.bincount(sidx, minlength=B)
    S = int(cnt.max().item())
    starts = torch.cumsum(cnt, 0) - cnt
    pos = torch.arange(T, device=x.device) - starts[sidx]
    pad = x.new_zeros(B, S, P, D)
    pad[sidx, pos] = x
    keep = torch.zeros(B, S, dtype=torch.bool, device=x.device)
    keep[sidx, pos] = True
    return (norm(pad.reshape(B, S * P, D)), ~keep.repeat_interleave(P, dim=1))

class _GatedDelta(nn.Module):

    def __init__(self, d, n_labels, n_heads, dropout):
        super().__init__()
        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.kv_norm = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.d_norm = nn.LayerNorm(d)
        self.dw = nn.Parameter(torch.randn(n_labels, d) * (1.0 / d ** 0.5))
        self.db = nn.Parameter(torch.zeros(n_labels))
        self.gate = nn.Parameter(torch.zeros(n_labels))

    def delta(self, pat, sidx, B, return_attn):
        kv, kpm = _pad_kv(pat, sidx, B, self.kv_norm)
        q = self.q.unsqueeze(0).expand(B, -1, -1)
        att, w = self.attn(q, kv, kv, key_padding_mask=kpm, need_weights=return_attn, average_attn_weights=True)
        return ((self.d_norm(att) * self.dw).sum(-1) + self.db, w)

class TokenResidualPool(_GatedDelta):

    def __init__(self, d, n_labels=12, n_heads=6, pe=64, dropout=0.2):
        super().__init__(d, n_labels, n_heads, dropout)
        self.base = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(dropout), nn.Linear(2 * d + pe, n_labels))

    def forward(self, tok, slot, sidx, B, pres, return_attn=False):
        base = self.base(torch.cat([_seg_mean_max(tok[:, 1:].mean(1), sidx, B), pres], 1))
        d_, w = self.delta(tok[:, 1:], sidx, B, return_attn)
        return (base + self.gate * d_, w)

class CodexResidualPool(_GatedDelta):

    def __init__(self, d, n_labels=12, n_heads=6, pe=64, dropout=0.2):
        super().__init__(d, n_labels, n_heads, dropout)
        self.base = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(dropout), nn.Linear(2 * d + pe, n_labels))

    def forward(self, tok, slot, sidx, B, pres, return_attn=False):
        base = self.base(torch.cat([_seg_mean_max(tok[:, 0], sidx, B), pres], 1))
        d_, w = self.delta(tok[:, 1:], sidx, B, return_attn)
        return (base + self.gate * d_, w)

class ClsAddPool(nn.Module):

    def __init__(self, d, n_labels=12, pe=64, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(4 * d + pe), nn.Dropout(dropout), nn.Linear(4 * d + pe, n_labels))

    def forward(self, tok, slot, sidx, B, pres, return_attn=False):
        return (self.net(torch.cat([_seg_mean_max(tok[:, 1:].mean(1), sidx, B), _seg_mean_max(tok[:, 0], sidx, B), pres], 1)), None)

class Readout(nn.Module):

    def __init__(self, pool, d, n_labels=12, pe=64):
        super().__init__()
        self.pool_kind, self.k = (pool, n_labels)
        self.pres_emb = nn.Embedding(N_SLOT_TYPES + 1, pe, padding_idx=0)
        if pool in ('xres', 'clsadd', 'xcodex'):
            self.pool = {'xres': TokenResidualPool, 'clsadd': ClsAddPool, 'xcodex': CodexResidualPool}[pool](d, n_labels, pe=pe)
        elif pool in ('attn', 'xattn'):
            if pool == 'xattn':
                self.pool = TokenXAttnPool(d, n_labels)
                wd = 3 * d + pe
            else:
                self.pool = LabelAttentionPool(d, n_labels)
                wd = d + pe
            self.norm = nn.LayerNorm(wd)
            self.w = nn.Parameter(torch.randn(n_labels, wd) * (1.0 / wd ** 0.5))
            self.b = nn.Parameter(torch.zeros(n_labels))
        else:
            self.pool = MeanMaxPool()
            self.net = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(0.2), nn.Linear(2 * d + pe, n_labels))
        self.drop = nn.Dropout(0.2)

    def forward(self, f, slot, sidx, B, return_attn=False):
        pe = self.pres_emb(slot)
        pres = torch.zeros(B, pe.shape[1], device=f.device, dtype=f.dtype).index_add_(0, sidx, pe)
        if self.pool_kind in ('xres', 'clsadd', 'xcodex'):
            return self.pool(f, slot, sidx, B, pres)[0]
        pooled, attn = self.pool(f, sidx, B, slot=slot, return_attn=return_attn)
        if self.pool_kind in ('attn', 'xattn'):
            x = torch.cat([pooled, pres.unsqueeze(1).expand(-1, self.k, -1)], -1)
            x = self.drop(self.norm(x))
            return (x * self.w).sum(-1) + self.b
        return self.net(torch.cat([pooled, pres], 1))

class Net(nn.Module):

    def __init__(self, enc, cond, n_meta=0, pool='mean_max', stem='native', n_slice=16):
        super().__init__()
        self.enc, self.cond = (enc, cond)
        self.compress = DepthCompress(n_slice, 3) if stem == 'compress' else None
        self.mixer = SlotDepthMixer(n_slice) if stem == 'mixer' else None
        self.tokens = pool in ('xattn', 'xres', 'clsadd', 'xcodex')
        D = enc.num_features
        self.meta_mlp = nn.Sequential(nn.LayerNorm(n_meta), nn.Linear(n_meta, 128), nn.GELU(), nn.Linear(128, D)) if n_meta > 0 else None
        self.readout = Readout(pool, D)
        if cond == 'post':
            self.slot_emb = nn.Embedding(N_SLOT_TYPES + 1, D, padding_idx=MASK_IDX)

    def forward(self, im, slot, smeta, sidx, B, vm=None):
        if self.mixer is not None:
            im = self.mixer(im, slot, vm)
        if self.compress is not None:
            im = self.compress(im)
        f = self.enc.forward_features(im, slot) if self.cond == 'token' else self.enc.forward_features(im)
        if self.tokens:
            inner = getattr(self.enc, 'vit', self.enc)
            orig = getattr(self.enc, '_orig_prefix', getattr(inner, 'num_prefix_tokens', 1))
            f = torch.cat([f[:, :1], f[:, orig:]], 1)
        else:
            f = self.enc.forward_head(f, pre_logits=True)
            if f.dim() > 2:
                f = f.flatten(1)
        ex = (lambda v: v.unsqueeze(1)) if self.tokens else lambda v: v
        if self.cond == 'post':
            f = f + ex(self.slot_emb(slot))
        if self.meta_mlp is not None and smeta.shape[1] > 0:
            mt = self.meta_mlp(smeta)
            f = torch.cat([f, mt.unsqueeze(1)], 1) if self.tokens else f + mt
        return self.readout(f, slot, sidx, B)
models = []
for ckpt_path in sorted(CKPT.glob('*_f*.pt')):
    z = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    cfg = z['cfg']
    _stem = cfg.get('stem', 'native')
    _in = 3 if _stem == 'compress' else cfg.get('n_slice', 16)
    enc = timm.create_model(cfg['backbone'], pretrained=False, num_classes=0, in_chans=_in, **{'img_size': cfg['img']} if 'vit_' in cfg['backbone'] else {})
    if cfg['cond'] == 'token':
        enc = ViTSlotToken(enc, N_SLOT_TYPES)
    m = Net(enc, cfg['cond'], cfg.get('n_meta', 0), cfg['pool'], stem=_stem, n_slice=cfg.get('n_slice', 16))
    missing, unexpected = m.load_state_dict(z['state_dict'], strict=False)
    assert not [k for k in missing if not k.startswith('enc.')], f'missing {missing[:5]}'
    assert not unexpected, f'unexpected {unexpected[:5]}'
    models.append(m.eval())
    print(f"loaded {ckpt_path.name}  fold {z['fold']}  {cfg['backbone']} pool={cfg['pool']} meta={cfg['meta']}")
CFG = cfg
assert CFG.get('n_meta', 0) == 0, f"checkpoint expects {CFG['n_meta']} metadata features -- build slot_meta for the TEST studies and pass it to predict() before submitting"
print(f"\n{len(models)} fold models ready | input norm: {CFG.get('norm', 'none')}")

### bfloat16 混合精度での推論

**何をしているか**: `amp_for(dev)` が
デバイスの compute capability を見て使用する精度を決め、`predict` が推論を実行します。
既定は `bf16`（bfloat16）です。

**なぜそうするのか**: 混合精度は**メモリを半減し、行列積を高速化**します。
ここで重要なのが **fp16 ではなく bf16 を選んでいる**ことです。

- **fp16** … 仮数部が多く精度は高いが、**表現できる指数の範囲が狭い**。
  値が小さいとゼロに潰れ（アンダーフロー）、大きいと `inf` になる。
  そのため損失スケーリングという追加の仕組みが必要になる
- **bf16** … 仮数部を削って**指数部を fp32 と同じ幅**にした形式。
  精度は粗いが**範囲が広く、オーバーフロー/アンダーフローがほぼ起きない**。
  損失スケーリングなしでそのまま使える

推論では勾配がないので精度への要求はさらに緩く、
**bf16 の「壊れにくさ」のほうが価値が高い**という判断です。

なお T4（Turing世代）は bf16 のハードウェア対応がないため、
`torch.cuda.get_device_capability` で世代を確認する分岐が入っています。
**環境に応じて安全側に倒す**設計です。

In [ ]:
AMP_PREF = 'bf16'

def amp_for(dev):
    if not str(dev).startswith('cuda'):
        return (torch.float32, False)
    cc = torch.cuda.get_device_capability(dev)
    if AMP_PREF == 'bf16':
        return (torch.bfloat16, True)
    if AMP_PREF == 'fp16':
        return (torch.float16, True)
    if AMP_PREF == 'fp32':
        return (torch.float32, False)
    return (torch.bfloat16 if cc >= (8, 0) else torch.float16, True)
AMP_DT, AMP_ON = amp_for(DEV)
WORKERS = max(1, min(4, os.cpu_count() or 4))
CHUNK = 48
MICRO = 8
models = [m.to(DEV).eval() for m in models]
print(f"device {DEV} | amp {str(AMP_DT).split('.')[-1]} (on={AMP_ON}) | workers {WORKERS} | chunk {CHUNK} | micro {MICRO}")

def _norm_(im):
    k = CFG.get('norm', 'none')
    if k == 'zscore':
        m = (im > 0).float()
        n = m.sum(dim=(1, 2, 3), keepdim=True).clamp(min=1.0)
        mu = (im * m).sum(dim=(1, 2, 3), keepdim=True) / n
        var = (((im - mu) * m) ** 2).sum(dim=(1, 2, 3), keepdim=True) / n
        return (im - mu) / (var.sqrt() + 1e-06) * m
    if k == 'imagenet':
        m = (im > 0).float()
        return (im - 0.485) / 0.229 * m
    return im

@torch.no_grad()
def _micro(images, masks):
    dev = DEV
    ims, slots, sidx, vms = ([], [], [], [])
    for b in range(len(masks)):
        present = np.nonzero(masks[b] > 0)[0]
        if len(present) == 0:
            continue
        blk = images[b][present]
        ims.append(torch.from_numpy(blk))
        vms.append(torch.from_numpy(blk.reshape(blk.shape[0], blk.shape[1], -1).max(2) > 0))
        slots.append(torch.from_numpy(present + 1).long())
        sidx.append(torch.full((len(present),), b, dtype=torch.long))
    out = np.full((len(models), len(masks), len(LABELS)), np.nan, np.float32)
    if not ims:
        return out
    im = _norm_(torch.cat(ims).to(dev, non_blocking=True).float().div_(255.0))
    sl = torch.cat(slots).to(dev)
    si = torch.cat(sidx).to(dev)
    vm = torch.cat(vms).to(dev)
    sm = torch.zeros(len(sl), CFG.get('n_meta', 0), device=dev)
    per = torch.zeros(len(models), len(masks), len(LABELS), device=dev, dtype=torch.float32)
    with torch.autocast('cuda' if str(dev).startswith('cuda') else 'cpu', dtype=AMP_DT, enabled=AMP_ON):
        for fold_index, model in enumerate(models):
            per[fold_index] = torch.sigmoid(
                model(im, sl, sm, si, len(masks), vm=vm).float()
            )
    got = per.cpu().numpy()
    keep = np.array([(masks[b] > 0).any() for b in range(len(masks))])
    out[:, keep] = got[:, keep]
    return out

def predict(images, masks):
    out = np.full((len(models), len(masks), len(LABELS)), np.nan, np.float32)
    for a in range(0, len(masks), MICRO):
        b = min(a + MICRO, len(masks))
        out[:, a:b] = _micro(images[a:b], masks[a:b])
    return out

# Macro ROC-AUC depends on ordering, so combine fold orderings rather
# than allowing a fold's probability scale to dominate the mean.
preds = np.full((len(models), len(studies), len(LABELS)), np.nan, np.float32)
t0, done = (time.time(), 0)
with ProcessPoolExecutor(max_workers=WORKERS) as ex:
    for c0 in range(0, len(studies), CHUNK):
        block = studies[c0:c0 + CHUNK]
        imgs = np.zeros((len(block), N_SLOT, N_SLICE, SIZE, SIZE), np.uint8)
        msks = np.zeros((len(block), N_SLOT), np.uint8)
        futs = [ex.submit(build_study, (i, s, by.get(s, []))) for i, s in enumerate(block)]
        for f in as_completed(futs):
            try:
                i, a, k = f.result()
                imgs[i], msks[i] = (a, k)
            except Exception as e:
                print(f'  study failed: {type(e).__name__}: {e}')
        preds[:, c0:c0 + len(block)] = predict(imgs, msks)
        done += len(block)
        el = time.time() - t0
        print(f'  {done:,}/{len(studies):,}  {el / 60:.1f}m  eta {el / done * (len(studies) - done) / 60:.1f}m', flush=True)
        del imgs, msks
        gc.collect()
print(f'\ninference done in {(time.time() - t0) / 60:.1f} min')
A5_W = 0.45
A5_LABELS = list(LABELS)
_a5_ok = np.isfinite(preds).all(axis=(0, 2))
_a5_rank_mean = np.zeros((len(studies), len(LABELS)), np.float64)
for fold_index in range(preds.shape[0]):
    fold = preds[fold_index][_a5_ok]
    ordinal = fold.argsort(0).argsort(0).astype(np.float64)
    _a5_rank_mean[_a5_ok] += ordinal / max(len(fold) - 1, 1)
_a5_rank_mean /= preds.shape[0]
_a5_rank_mean[~_a5_ok] = np.nan
A5_PREDS = dict(zip(
    sub_df['StudyInstanceUID'].astype(str), _a5_rank_mean.astype(np.float32)
))
for _a5k, _a5v in _A5_SAVED.items():
    globals()[_a5k] = _a5v
del _A5_SAVED, _a5k, _a5v

### Stage 2 のランクブレンド（W = 0.45）

**何をしているか**: Stage 1 が書き出した `submission.csv` を読み直し、

1. **列構成が期待どおりか `assert` で検証**（スキーマドリフト検出）
2. Stage 1 の予測と Stage 2 の予測を**それぞれランク化**
3. `0.55 × Stage1 + 0.45 × Stage2` で加重平均し、書き戻す

**なぜそうするのか**:

- **ファイル経由で受け渡す理由**: Stage 1 と Stage 2 はメモリを大量に使うため、
  **同時にメモリ上に載せられません**。
  中間結果をディスクに書いて GC を挟むことで、GPU/CPUメモリを解放しています。
  さらに副次効果として、**Stage 2 が失敗しても Stage 1 の提出は残ります**。
- **`assert` を入れる理由**: 上流のコードが変わって列の順序が入れ替わると、
  **ACLの予測がMCLの列に入る**といった事故が起きます。
  しかもエラーにはならず、スコアが不可解に下がるだけです。
  ステージ間の受け渡しには**必ず契約（スキーマ）の検証**を挟むべきです。
- **重み 0.45 の出どころ**: 原著は根拠を明示していません。
  おそらく手元の検証で試した値ですが、
  **12ターゲット一律の重み**である点は改善余地です
  （Stage 3 ではターゲット別のαを使っているので、思想としては一貫していません）。
  この点は `README.md` の改善提案に含めました。

In [ ]:
_a5_sub = pd.read_csv('/kaggle/working/submission.csv',
                      dtype={'StudyInstanceUID': str})
assert _a5_sub.columns.tolist()[1:] == A5_LABELS, 'submission schema drift'
if A5_W > 0:
    _a5_ours = np.stack([A5_PREDS[_u]
                         for _u in _a5_sub['StudyInstanceUID'].astype(str)])
    _a5_base_rank = _a5_sub[A5_LABELS].rank(method='average', pct=True)
    _a5_ours_rank = pd.DataFrame(_a5_ours, columns=A5_LABELS,
                                 index=_a5_sub.index).rank(method='average', pct=True)
    _a5_sub[A5_LABELS] = (1.0 - A5_W) * _a5_base_rank + A5_W * _a5_ours_rank
    assert np.isfinite(_a5_sub[A5_LABELS].to_numpy()).all()
    _a5_sub.to_csv('/kaggle/working/submission.csv', index=False)

## 7. Stage 3: Surgical V48 Dual-Layout RadImageNet Fusion (0.920 SOTA Pinnacle)

Fuses domain-specific **RadImageNet ResNet-50** across two complementary physical layouts:
1. **E13 Layout**: 3 fat-suppressed planes + 1 non-suppressed anchor @ 130mm ($\alpha = 0.50$).
2. **E11 Layout**: 3 non-suppressed planes + 1 fat-suppressed anchor @ 130mm ($\alpha = 0.15$).

### Invariant Gating Contract (Rule #7)
Focal targets with high localized precision (**Baker's Cyst** and **Fracture**) are strictly gated at $\alpha = 0.0$ to prevent dilution of DINOv2 focal attention:

$$\alpha_{\text{Baker's}} = 0.0, \quad \alpha_{\text{Fracture}} = 0.0$$


### Stage 3：RadImageNet の2レイアウト融合とターゲット別ゲート

**何をしているか**: この解法の最終段です。

- `_RadEncoder` / `_RadHead` … **RadImageNet で事前学習された ResNet-50** の枝
- `_rad_load_e13_heads` / `_rad_load_e11_heads` … 2種類の**入力レイアウト**のヘッドを読む
  - **E13**: 脂肪抑制ありの3面 + 脂肪抑制なしのアンカー1枚（α = 0.50）
  - **E11**: 脂肪抑制なしの3面 + 脂肪抑制ありのアンカー1枚（α = 0.15）
- `_rad_load_public_heads` … 上記が無い場合の**フォールバック**
- 最終的に、**ターゲット別のα**で Stage 2 の結果とランクブレンドする
  （ベーカー嚢腫と骨折は **α = 0.0**、つまり Stage 2 のまま据え置き）

**なぜそうするのか**:

- **なぜ RadImageNet か**: ImageNet 事前学習の重みは自然画像（犬、車、椅子）で学ばれています。
  RadImageNet は**130万件超の放射線画像**で事前学習されており、
  医用画像への転移では**開始地点が近い**。
  DINOv2（自己教師あり・自然画像）と RadImageNet（教師あり・医用画像）は
  **事前学習のドメインも目的関数も違う**ので、
  アンサンブルとして**間違え方が独立になりやすい**という狙いもあります。
- **なぜ2つのレイアウトを使うか**: 脂肪抑制の有無で見えるものが変わります。
  脂肪抑制ありは骨髄浮腫や関節液がよく見え、なしは解剖構造の輪郭がよく見えます。
  **どちらを主役にするか**で2つのレイアウトを作り、両方を混ぜています。
  ただし重みは 0.50 と 0.15 で、**脂肪抑制ありレイアウトのほうが信頼されている**ことが読み取れます。
- **なぜベーカー嚢腫と骨折だけ α = 0.0 か**: この2つは**限局した所見**で、
  Stage 1 の DINOv2 が `max` プーリングで強く捉えています。
  ここに別モデルの予測を混ぜると、**鋭い信号が平均化で鈍る**。
  マクロ平均AUCは**ターゲットごとに独立に効く**ので、
  「全体最適の重み」ではなく「**ターゲットごとの最適な重み**」を選ぶのが正しい。
  一律ブレンドをやめる判断ができるかどうかが、上位帯との分かれ目になります。
- **フォールバック設計**: E13/E11 のバンドルが添付されていない場合、
  公開版の RadImageNet アンサンブルに滑らかに切り替わります。
  「最良の構成で動くこと」と「どんな構成でも動くこと」を両立させる作りです。

> ⚠️ 学習上の注意: このセルは 41,000 文字を超えており、
> 1つのセルに前処理・モデル定義・重み読み込み・推論・ブレンドが全部詰まっています。
> **可読性という意味では悪い例**です。
> スコアを追う過程で膨らんだコードがそのまま残っている、と理解してください。

In [ ]:
# Surgical reproduction of V48's deployed prediction branch.
#
# The pinned reference Rad family is fused with correct-contract E13, then
# the same E13 heads run on the E11 layout at 0.15. No twin/legacy wrapper
# follows it, matching the branch that produced V48's visible submission.
from __future__ import annotations

import contextlib as _rad_contextlib
import gc as _rad_gc
import hashlib as _rad_hashlib
import json as _rad_json
import os as _rad_os
import re as _rad_re
import time as _rad_time
from concurrent.futures import ThreadPoolExecutor as _RadThreadPool
from pathlib import Path as _RadPath

import numpy as _rad_np
import pandas as _rad_pd
import pydicom as _rad_pydicom
import torch as _rad_torch
import torch.nn as _rad_nn
import torch.nn.functional as _rad_F
from torchvision.models import resnet50 as _rad_resnet50

_RAD_LABELS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
    'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's",
    'Contusion', 'Fracture',
]
_RAD_ALPHA = 0.50
_RAD_EXCLUDE = ("Baker's", 'Fracture')
_RAD_HEADS_SHA256 = '54f657826b3458a7ba3d462e198ba380732f2b136246182312704929874a9a2c'
_RAD_REFERENCE_HEADS_SHA256 = '0f465649799ecfbccaac1767844639e7ced44e1bc9babde6e4bac7c5d9b89eaa'
_RAD_ENCODER_SHA256 = '08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734'
_RAD_E11_HEADS_SHA256 = 'df98518d0a8f7cc506b4b806da03c3fca18c8f8f743e813c5401759b45e0f1fe'
_RAD_REFERENCE_E11_HEADS_SHA256 = 'f05d92ee59dcce270cfb7983fa219940abf9b0c2e247ff12a88a81129841ca34'
_RAD_E13_HEADS_SHA256 = 'ad9f19af73bfdf4e49263c0e45060dc3cb239e1195039b26dc8c0a3a6bcd1a8a'
_RAD_E13_MEMBER_WEIGHT = 0.50
_RAD_V48_SECOND_ALPHA = 0.15
_RAD_LEGACY_DINO_ALPHA = 0.02
_RAD_TWIN_ALT_WEIGHT = 0.500001
_RAD_TOKEN_DIM, _RAD_HEAD_DIM = 2048, 512

_RAD_E11_SLOTS = [
    ('SAG_NOFS', 'Sagittal', None, False),
    ('COR_NOFS', 'Coronal', None, False),
    ('AX_NOFS', 'Axial', None, False),
    ('SAG_FS', 'Sagittal', None, True),
]
_RAD_E11_CROP_MM = 130.0
_RAD_E11_CACHE_SLICES = 8
_RAD_E11_IMG = 224

_RAD_E13_SLOTS = [
    ('SAG_FS', 'Sagittal', None, True),
    ('COR_FS', 'Coronal', None, True),
    ('AX_FS', 'Axial', None, True),
    ('SAG_NOFS', 'Sagittal', None, False),
]
_RAD_E13_CROP_MM = 130.0
_RAD_E13_CACHE_SLICES = 8
_RAD_E13_IMG = 224

# Our independently trained five-fold family.  Its preprocessing and estimator
# are preserved from V35: native DICOM geometry/fat-sat handling and a mean of
# per-fold percentile ranks (rather than v15's rank of the probability mean).
_OUR_PLANES = ('Sagittal', 'Coronal', 'Axial')
_OUR_N_SLOT, _OUR_N_SLICE, _OUR_IMG = 3, 8, 224
_OUR_SLICE_BAND = (0.12, 0.88)
_OUR_FOLD_SHA256 = '1301603a060226c47c96be54d4c3618fee41f2e97f8f82d8f77a752819ffb7e3'
_OUR_CONFIG_HASH = '794a44d95a0ebb7096f17daa5a06dc191ec16c4d0835e69ac1868e82a7eb05dd'
_OUR_HEAD_SHA256 = {
    'rad_head_f0.pt': '0c92b27578e139cc35071a3f72ddd4e1a66106761225a7e011f076f37eb7051d',
    'rad_head_f1.pt': 'c29e60a99982d8dfb933a276f51c8b3a7d2e849649ed27a09affa822405fbd17',
    'rad_head_f2.pt': '40328ac7d72ca281e7e04438643100e99eb87fbe1e2b51965350b1a73a7b57ab',
    'rad_head_f3.pt': '6538a5faa92b61705727a14bd98c5ddce2989028dd8cccb261c47b2066fa5efa',
    'rad_head_f4.pt': '5bd126d68ddadd479a0eab6f0cdb17603060614a7eb14402e5b9945f651107af',
}
_OUR_FATSAT_OPTIONS = {'FS', 'FATSAT', 'FAT_SAT', 'FSAT'}
_OUR_FATSAT_PATTERN = _rad_re.compile(
    r'\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|'
    r'water excit|\btirm\b|\bsting\b|\bfatsup\b'
)

# Exact V40/E10 test representation: three fat-suppressed planes, eight
# acquired slices per plane, full frame, legacy ordering/laterality/fill.
SLOTS = [
    ('SAG_FS', 'Sagittal', None, True),
    ('COR_FS', 'Coronal', None, True),
    ('AX_FS', 'Axial', None, True),
]
N_SLOT = len(SLOTS)
CACHE_SLICES = 8
IMG = CACHE_IMG = 224
CROP_MM = 10_000.0
SLICE_BAND = (0.2, 0.8)
RULES = dict(RULES_LEGACY)
TIME_BUDGET = 8.72 * 3600


def _rad_log(message):
    print(f'[Rad-dual5] {message}', flush=True)


def _rad_sha256(path, chunk=8 << 20):
    digest = _rad_hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(chunk), b''):
            digest.update(block)
    return digest.hexdigest()


def _rad_find_file(name, expected_sha=None, explicit_env=None):
    if explicit_env and _rad_os.environ.get(explicit_env):
        candidates = [_RadPath(_rad_os.environ[explicit_env])]
    else:
        candidates = []
        base = _RadPath('/kaggle/input')
        if base.is_dir():
            for root, dirs, files in _rad_os.walk(base):
                dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
                if name in files:
                    candidates.append(_RadPath(root) / name)
    if not candidates:
        raise FileNotFoundError(f'V36 missing input artifact {name}')
    for path in candidates:
        if expected_sha is None or _rad_sha256(path) == expected_sha:
            return path
    raise RuntimeError(f'V36 found {name}, but no copy has the required SHA-256')


def _ours_find_head_dir():
    override = _rad_os.environ.get('RSNA_RAD_HEAD_DIR')
    roots = [_RadPath(override)] if override else []
    if not roots:
        base = _RadPath('/kaggle/input')
        if base.is_dir():
            for root, dirs, files in _rad_os.walk(base):
                # Never traverse the competition's DICOM trees while locating a
                # tiny attached model manifest.
                dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
                if 'rad_heads_manifest.json' in files:
                    roots.append(_RadPath(root))
    for root in roots:
        manifest_path = root / 'rad_heads_manifest.json'
        if not manifest_path.is_file():
            continue
        manifest = _rad_json.loads(manifest_path.read_text())
        if manifest.get('artifact_type') != 'rsna-radimagenet-foldsv1-heads':
            continue
        if manifest.get('fold_sha256') != _OUR_FOLD_SHA256:
            raise RuntimeError('our Rad head manifest fold hash drift')
        if manifest.get('config_hash') != _OUR_CONFIG_HASH:
            raise RuntimeError('our Rad head manifest config hash drift')
        if manifest.get('labels') != _RAD_LABELS:
            raise RuntimeError('our Rad head manifest label order drift')
        return root, manifest
    raise FileNotFoundError('our five-fold RadImageNet head dataset is not attached')


def _ours_vector(value, length):
    try:
        result = _rad_np.asarray([float(item) for item in value], dtype=_rad_np.float64)
    except Exception:
        return None
    if len(result) < length or not _rad_np.isfinite(result[:length]).all():
        return None
    return result


def _ours_dicom_files(directory):
    return sorted(_RadPath(directory).glob('*.dcm'), key=lambda path: path.name)


def _ours_header(path):
    tags = [
        'ImagePositionPatient', 'ImageOrientationPatient', 'InstanceNumber',
        'Laterality', 'PixelSpacing', 'Rows', 'Columns', 'SeriesDescription',
        'SequenceName', 'ScanOptions',
    ]
    try:
        return _rad_pydicom.dcmread(
            str(path), stop_before_pixels=True, force=True, specific_tags=tags
        )
    except Exception:
        return None


def _ours_image_center_x(ds):
    if ds is None:
        return None
    ipp = _ours_vector(getattr(ds, 'ImagePositionPatient', None), 3)
    iop = _ours_vector(getattr(ds, 'ImageOrientationPatient', None), 6)
    spacing = _ours_vector(getattr(ds, 'PixelSpacing', None), 2)
    try:
        rows, cols = float(ds.Rows), float(ds.Columns)
    except Exception:
        return None
    if ipp is None or iop is None or spacing is None:
        return None
    center = (
        ipp[:3]
        + iop[:3] * spacing[1] * cols / 2.0
        + iop[3:6] * spacing[0] * rows / 2.0
    )
    return float(center[0])


def _ours_is_fatsat(ds):
    if ds is None:
        return False
    description = (
        f"{getattr(ds, 'SeriesDescription', '') or ''} "
        f"{getattr(ds, 'SequenceName', '') or ''}"
    ).lower()
    description = _rad_re.sub(r'[_\-.]', ' ', description)
    options = getattr(ds, 'ScanOptions', None)
    if options is None:
        tokens = []
    elif isinstance(options, str):
        tokens = _rad_re.split(r'[|\\]', options)
    else:
        try:
            tokens = list(options)
        except TypeError:
            tokens = [options]
    option_match = any(
        str(token).strip().upper() in _OUR_FATSAT_OPTIONS for token in tokens
    )
    return bool(_OUR_FATSAT_PATTERN.search(description) or option_match)


def _ours_inspect_series(job):
    study, series, plane, directory = job
    files = _ours_dicom_files(directory)
    ds = _ours_header(files[len(files) // 2]) if files else None
    laterality = str(getattr(ds, 'Laterality', '') or '').strip().upper()[:1]
    laterality = laterality if laterality in ('L', 'R') else None
    return {
        'study': study,
        'series': series,
        'plane': plane,
        'directory': str(directory),
        'n_files': len(files),
        'fatsat': _ours_is_fatsat(ds),
        'laterality': laterality,
        'center_x': _ours_image_center_x(ds),
    }


def _ours_select_series(comp, studies):
    table = _rad_pd.read_csv(
        comp / 'test_series.csv',
        dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str},
    )
    table = table[table['StudyInstanceUID'].isin(studies)]
    root = comp / 'test_series'
    jobs = [
        (
            str(row.StudyInstanceUID),
            str(row.SeriesInstanceUID),
            str(row.Anatomical_Plane),
            root / str(row.StudyInstanceUID) / str(row.SeriesInstanceUID),
        )
        for row in table.itertuples(index=False)
    ]
    workers = max(1, min(16, _rad_os.cpu_count() or 1))
    with _RadThreadPool(max_workers=workers) as pool:
        inspected = list(pool.map(_ours_inspect_series, jobs))
    selected = {uid: [None] * _OUR_N_SLOT for uid in studies}
    tags = {uid: [] for uid in studies}
    centers = {uid: [] for uid in studies}
    for record in inspected:
        uid = record['study']
        if record['laterality']:
            tags[uid].append(record['laterality'])
        if record['center_x'] is not None:
            centers[uid].append(record['center_x'])
        if not record['fatsat'] or record['plane'] not in _OUR_PLANES:
            continue
        slot = _OUR_PLANES.index(record['plane'])
        current = selected[uid][slot]
        if current is None or record['n_files'] > current['n_files']:
            selected[uid][slot] = record
    sides = {}
    for uid in studies:
        if tags[uid]:
            sides[uid] = tags[uid][0]
        elif centers[uid]:
            middle = float(_rad_np.median(centers[uid]))
            sides[uid] = None if abs(middle) < 20.0 else ('R' if middle < 0 else 'L')
        else:
            sides[uid] = None
    return selected, sides


def _ours_order_files(directory):
    files = _ours_dicom_files(directory)
    rows = []
    for path in files:
        ds = _ours_header(path)
        key = None
        if ds is not None:
            ipp = _ours_vector(getattr(ds, 'ImagePositionPatient', None), 3)
            iop = _ours_vector(getattr(ds, 'ImageOrientationPatient', None), 6)
            if ipp is not None and iop is not None:
                key = float(_rad_np.dot(ipp[:3], _rad_np.cross(iop[:3], iop[3:6])))
            if key is None:
                try:
                    key = float(ds.InstanceNumber)
                except Exception:
                    pass
        rows.append((key, path))
    # Preserve the native training rule: if any position is unavailable, use
    # filename order for the entire series rather than mixing order systems.
    if any(key is None for key, _ in rows):
        return files
    return [path for _, path in sorted(rows, key=lambda item: item[0])]


def _ours_sample_indices(length):
    if length <= 0:
        return _rad_np.zeros(0, dtype=_rad_np.int64)
    lo = int(_OUR_SLICE_BAND[0] * (length - 1))
    hi = int(_OUR_SLICE_BAND[1] * (length - 1))
    if hi > lo:
        indices = _rad_np.unique(
            _rad_np.linspace(lo, hi, _OUR_N_SLICE).astype(_rad_np.int64)
        )
    else:
        indices = _rad_np.array([length // 2], dtype=_rad_np.int64)
    while len(indices) < _OUR_N_SLICE:
        indices = _rad_np.append(indices, indices[-1])
    return indices[:_OUR_N_SLICE]


def _ours_read_series(record):
    files = _ours_order_files(record['directory'])
    indices = _ours_sample_indices(len(files))
    if not len(indices):
        return None
    images = []
    for index in indices:
        try:
            ds = _rad_pydicom.dcmread(str(files[int(index)]), force=True)
            image = ds.pixel_array.astype(_rad_np.float32)
            image = image * float(getattr(ds, 'RescaleSlope', 1) or 1)
            image = image + float(getattr(ds, 'RescaleIntercept', 0) or 0)
        except Exception:
            image = None
        images.append(image)
    valid = [index for index, image in enumerate(images) if image is not None]
    if not valid:
        return None
    for index, image in enumerate(images):
        if image is None:
            images[index] = images[min(valid, key=lambda other: abs(other - index))]
    shape = images[0].shape
    images = [
        image if image.shape == shape else _rad_np.zeros(shape, _rad_np.float32)
        for image in images
    ]
    volume = _rad_np.stack(images).astype(_rad_np.float32, copy=False)
    lo, hi = _rad_np.percentile(volume, [1.0, 99.0])
    volume = _rad_np.clip((volume - lo) / max(float(hi - lo), 1e-6), 0.0, 1.0)
    tensor = _rad_torch.from_numpy(_rad_np.ascontiguousarray(volume)).unsqueeze(0)
    resized = _rad_F.interpolate(
        tensor, size=(_OUR_IMG, _OUR_IMG), mode='bilinear', align_corners=False
    ).squeeze(0)
    return resized.mul(255).round().clamp(0, 255).to(_rad_torch.uint8).numpy()


def _ours_build_study(job):
    index, uid, records, side = job
    output = _rad_np.zeros(
        (_OUR_N_SLOT, _OUR_N_SLICE, _OUR_IMG, _OUR_IMG), _rad_np.uint8
    )
    mask = _rad_np.zeros(_OUR_N_SLOT, _rad_np.uint8)
    for slot, record in enumerate(records):
        if record is None:
            continue
        image = _ours_read_series(record)
        if image is None:
            continue
        if side == 'R':
            image = (
                image[::-1].copy()
                if record['plane'] == 'Sagittal'
                else image[:, :, ::-1].copy()
            )
        output[slot] = image
        mask[slot] = 1
    return index, uid, output, mask


class _RadEncoder(_rad_nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = _rad_nn.Sequential(
            *list(_rad_resnet50(weights=None).children())[:-2]
        )

    def forward(self, image):
        return self.backbone(image).mean(dim=(2, 3))


class _RadHead(_rad_nn.Module):
    def __init__(self):
        super().__init__()
        self.project = _rad_nn.Sequential(
            _rad_nn.LayerNorm(_RAD_TOKEN_DIM),
            _rad_nn.Linear(_RAD_TOKEN_DIM, _RAD_HEAD_DIM),
            _rad_nn.GELU(),
        )
        self.plane = _rad_nn.Parameter(_rad_torch.randn(N_SLOT, _RAD_HEAD_DIM) * .01)
        self.position = _rad_nn.Parameter(_rad_torch.randn(CACHE_SLICES, _RAD_HEAD_DIM) * .01)
        self.query = _rad_nn.Parameter(_rad_torch.randn(len(_RAD_LABELS), _RAD_HEAD_DIM) * .02)
        self.attn = _rad_nn.MultiheadAttention(
            _RAD_HEAD_DIM, 8, dropout=.10, batch_first=True
        )
        self.fuse = _rad_nn.Sequential(
            _rad_nn.LayerNorm(_RAD_HEAD_DIM * 4),
            _rad_nn.Linear(_RAD_HEAD_DIM * 4, _RAD_HEAD_DIM),
            _rad_nn.GELU(),
            _rad_nn.Dropout(.15),
        )
        self.weight = _rad_nn.Parameter(
            _rad_torch.randn(len(_RAD_LABELS), _RAD_HEAD_DIM) * .02
        )
        self.bias = _rad_nn.Parameter(_rad_torch.zeros(len(_RAD_LABELS)))

    def forward(self, feature, mask):
        token = self.project(feature.float())
        token = token.view(len(token), N_SLOT, CACHE_SLICES, _RAD_HEAD_DIM)
        token = token + self.plane[None, :, None] + self.position[None, None]
        token = token.flatten(1, 2)
        key_padding = mask <= 0
        all_empty = key_padding.all(1)
        if all_empty.any():
            key_padding = key_padding.clone()
            key_padding[all_empty, 0] = False
        query = self.query.unsqueeze(0).expand(len(token), -1, -1)
        attended = query + self.attn(
            query, token, token, key_padding_mask=key_padding, need_weights=False
        )[0]
        denominator = mask.sum(1, keepdim=True).clamp_min(1).unsqueeze(-1)
        mean = (token * mask.unsqueeze(-1)).sum(1, keepdim=True) / denominator
        mean = mean.expand(-1, len(_RAD_LABELS), -1)
        fused = self.fuse(_rad_torch.cat(
            [attended, mean, _rad_torch.abs(attended - mean), attended * mean], dim=-1
        ))
        return (fused * self.weight.unsqueeze(0)).sum(-1) + self.bias


def _rad_load_public_heads(device, expected_sha):
    heads_path = _rad_find_file('v52_radimagenet_heads.pt', expected_sha)
    payload = _rad_torch.load(heads_path, map_location='cpu', weights_only=True)
    expected = {
        'version': 'v52-radimagenet-resnet50-official-1',
        'targets': _RAD_LABELS,
        'encoder_sha256': _RAD_ENCODER_SHA256,
        'encoder_source_commit': '0ce16f7375db4236e646829d1eca61cdb4282133',
        'img': 224,
        'slices_per_plane': 8,
        'feature': 'global_average_pool',
    }
    for key, value in expected.items():
        if payload.get(key) != value:
            raise RuntimeError(f'public-v15 head contract drift for {key}')
    folds = payload.get('folds')
    if not isinstance(folds, list) or len(folds) != 5:
        raise RuntimeError('public-v15 bundle requires exactly five heads')
    if sorted(int(record.get('fold', -1)) for record in folds) != list(range(5)):
        raise RuntimeError('public-v15 fold identity drift')
    heads = []
    for record in folds:
        head = _RadHead().to(device).eval()
        head.load_state_dict(record['state_dict'], strict=True)
        heads.append(head)
    return heads, str(heads_path)


def _rad_load_e11_heads(device, expected_sha):
    heads_path = _rad_find_file('v52_e11_heads.pt', expected_sha)
    payload = _rad_torch.load(heads_path, map_location='cpu', weights_only=False)
    expected = {
        'version': 'e11-radimagenet-resnet50-diverse-1',
        'targets': _RAD_LABELS,
        'encoder_sha256': _RAD_ENCODER_SHA256,
        'slots': [list(slot) for slot in _RAD_E11_SLOTS],
        'crop_mm': _RAD_E11_CROP_MM,
        'img': _RAD_E11_IMG,
        'slices_per_plane': _RAD_E11_CACHE_SLICES,
        'feature': 'global_average_pool',
    }
    for key, value in expected.items():
        if payload.get(key) != value:
            raise RuntimeError(f'E11 head contract drift for {key}')
    folds = payload.get('folds')
    if not isinstance(folds, list) or len(folds) != 5:
        raise RuntimeError('E11 bundle requires exactly five heads')
    if sorted(int(record.get('fold', -1)) for record in folds) != list(range(5)):
        raise RuntimeError('E11 fold identity drift')
    heads = []
    for record in folds:
        head = _RadHead().to(device).eval()
        head.load_state_dict(record['state_dict'], strict=True)
        heads.append(head)
    return heads, str(heads_path)


def _rad_load_e13_heads(device):
    # V48 used an unqualified filename shared by E11 and E13. Resolve the
    # intended E13 bundle by content and validate its complete pixel contract.
    heads_path = _rad_find_file('v52_e11_heads.pt', _RAD_E13_HEADS_SHA256)
    payload = _rad_torch.load(heads_path, map_location='cpu', weights_only=False)
    expected = {
        'version': 'e11-radimagenet-resnet50-diverse-1',
        'targets': _RAD_LABELS,
        'encoder_sha256': _RAD_ENCODER_SHA256,
        'slots': [list(slot) for slot in _RAD_E13_SLOTS],
        'crop_mm': _RAD_E13_CROP_MM,
        'img': _RAD_E13_IMG,
        'slices_per_plane': _RAD_E13_CACHE_SLICES,
        'feature': 'global_average_pool',
    }
    for key, value in expected.items():
        if payload.get(key) != value:
            raise RuntimeError(f'E13 head contract drift for {key}')
    folds = payload.get('folds')
    if not isinstance(folds, list) or len(folds) != 5:
        raise RuntimeError('E13 bundle requires exactly five heads')
    if sorted(int(record.get('fold', -1)) for record in folds) != list(range(5)):
        raise RuntimeError('E13 fold identity drift')
    heads = []
    for record in folds:
        head = _RadHead().to(device).eval()
        head.load_state_dict(record['state_dict'], strict=True)
        heads.append(head)
    return heads, str(heads_path)


def _rad_load_models(device):
    encoder_path = _rad_find_file(
        'ResNet50.pt', _RAD_ENCODER_SHA256, explicit_env='RSNA_RAD_WEIGHT_PATH'
    )
    encoder = _RadEncoder()
    encoder.load_state_dict(
        _rad_torch.load(encoder_path, map_location='cpu', weights_only=True), strict=True
    )
    if sum(parameter.numel() for parameter in encoder.parameters()) != 23_508_032:
        raise RuntimeError('V36 RadImageNet encoder parameter-count drift')
    encoder.eval().to(device)
    for parameter in encoder.parameters():
        parameter.requires_grad_(False)
    if device.type == 'cuda' and _rad_torch.cuda.device_count() > 1:
        encoder = _rad_nn.DataParallel(
            encoder, device_ids=list(range(_rad_torch.cuda.device_count()))
        )

    alt_heads, alt_path = _rad_load_public_heads(device, _RAD_HEADS_SHA256)
    reference_heads, reference_path = _rad_load_public_heads(
        device, _RAD_REFERENCE_HEADS_SHA256
    )
    if alt_path == reference_path:
        raise RuntimeError('twin E10 branches resolved to the same artifact')
    return encoder, alt_heads, reference_heads, str(encoder_path), alt_path, reference_path


def _ours_load_heads(device):
    if (_OUR_N_SLOT, _OUR_N_SLICE) != (N_SLOT, CACHE_SLICES):
        raise RuntimeError('public and native Rad head tensor shapes diverged')
    head_dir, manifest = _ours_find_head_dir()
    heads = []
    observed_heads = {}
    for fold in range(5):
        name = f'rad_head_f{fold}.pt'
        path = head_dir / name
        if not path.is_file():
            raise FileNotFoundError(path)
        observed = _rad_sha256(path)
        if observed != _OUR_HEAD_SHA256[name]:
            raise RuntimeError(f'our Rad head SHA-256 drift for {name}')
        if manifest['heads'][name]['sha256'] != observed:
            raise RuntimeError(f'our Rad manifest/checkpoint mismatch for {name}')
        checkpoint = _rad_torch.load(path, map_location='cpu', weights_only=False)
        config = checkpoint.get('config', {})
        if checkpoint.get('fold') != fold:
            raise RuntimeError(f'our Rad checkpoint fold drift for fold {fold}')
        if checkpoint.get('config_hash') != _OUR_CONFIG_HASH:
            raise RuntimeError(f'our Rad checkpoint config hash drift for fold {fold}')
        if config.get('gold_override') is not False:
            raise RuntimeError(f'our Rad checkpoint gold policy drift for fold {fold}')
        if config.get('target_mode') != 'public3':
            raise RuntimeError(f'our Rad checkpoint target mode drift for fold {fold}')
        if config.get('folds', {}).get('sha256') != _OUR_FOLD_SHA256:
            raise RuntimeError(f'our Rad checkpoint fold contract drift for fold {fold}')
        head = _RadHead().to(device).eval()
        head.load_state_dict(checkpoint['state_dict'], strict=True)
        heads.append(head)
        observed_heads[name] = observed
    return heads, str(head_dir), observed_heads


@_rad_torch.inference_mode()
def _rad_encode(encoder, pixels, slot_mask, device):
    n, slots, slices, height, width = pixels.shape
    features = _rad_np.zeros(
        (n, slots * slices, _RAD_TOKEN_DIM), _rad_np.float16
    )
    token_mask = _rad_np.repeat(slot_mask[:, :, None], slices, axis=2).reshape(n, -1)
    valid = _rad_np.flatnonzero(token_mask.reshape(-1) > 0)
    flat = pixels.reshape(-1, height, width)
    batch = 192 if device.type == 'cuda' and _rad_torch.cuda.device_count() > 1 else (
        96 if device.type == 'cuda' else 8
    )
    for start in range(0, len(valid), batch):
        indices = valid[start:start + batch]
        image = _rad_torch.from_numpy(flat[indices]).to(device).float().div_(127.5).sub_(1.0)
        image = image.unsqueeze(1).expand(-1, 3, -1, -1).contiguous()
        amp = (_rad_torch.autocast('cuda')
               if device.type == 'cuda' else _rad_contextlib.nullcontext())
        with amp:
            feature = encoder(image)
        values = feature.float().cpu().numpy()
        if not _rad_np.isfinite(values).all():
            raise RuntimeError('V36 non-finite RadImageNet feature')
        features.reshape(-1, _RAD_TOKEN_DIM)[indices] = values.astype(_rad_np.float16)
    return features, token_mask.astype(_rad_np.float32)


@_rad_torch.inference_mode()
def _rad_predict_head(head, features, masks, device, batch=64):
    predictions = []
    for start in range(0, len(features), batch):
        image = _rad_torch.from_numpy(features[start:start + batch]).to(device)
        mask = _rad_torch.from_numpy(masks[start:start + batch]).to(device)
        amp = (_rad_torch.autocast('cuda')
               if device.type == 'cuda' else _rad_contextlib.nullcontext())
        with amp:
            predictions.append(_rad_torch.sigmoid(head(image, mask)).float().cpu())
    return _rad_torch.cat(predictions).numpy()


@_rad_torch.inference_mode()
def _ours_encode_block(encoder, pixels, slot_mask, device):
    n = len(pixels)
    features = _rad_np.zeros(
        (n, _OUR_N_SLOT * _OUR_N_SLICE, _RAD_TOKEN_DIM), _rad_np.float16
    )
    token_mask = _rad_np.repeat(
        slot_mask[:, :, None], _OUR_N_SLICE, axis=2
    ).reshape(n, -1)
    valid = _rad_np.flatnonzero(token_mask.reshape(-1) > 0)
    flat = pixels.reshape(-1, _OUR_IMG, _OUR_IMG)
    # Preserve the native five-fold encoder batch/precision contract.
    batch = 96 if device.type == 'cuda' else 8
    for start in range(0, len(valid), batch):
        indices = valid[start:start + batch]
        image = _rad_torch.from_numpy(flat[indices]).to(device).float().div_(127.5).sub_(1.0)
        image = image.unsqueeze(1).expand(-1, 3, -1, -1).contiguous()
        amp = (
            _rad_torch.autocast('cuda', dtype=_rad_torch.float16)
            if device.type == 'cuda'
            else _rad_contextlib.nullcontext()
        )
        with amp:
            feature = encoder(image)
        values = feature.float().cpu().numpy()
        if not _rad_np.isfinite(values).all():
            raise RuntimeError('our Rad family produced non-finite encoder features')
        features.reshape(-1, _RAD_TOKEN_DIM)[indices] = values.astype(_rad_np.float16)
    return features, token_mask.astype(_rad_np.float32)


@_rad_torch.inference_mode()
def _ours_predict_heads(heads, features, masks, device):
    feature = _rad_torch.from_numpy(features).to(device)
    mask = _rad_torch.from_numpy(masks).to(device)
    # These heads were selected/rescored without head autocast; keep that path.
    return [
        _rad_torch.sigmoid(head(feature, mask)).cpu().numpy()
        for head in heads
    ]


def _rad_rank_columns(values):
    return _rad_pd.DataFrame(
        _rad_np.asarray(values, dtype=_rad_np.float64)
    ).rank(method='average', pct=True).to_numpy(_rad_np.float64)


def _rad_validate(frame, expected_ids):
    if frame.columns.tolist() != ['StudyInstanceUID', *_RAD_LABELS]:
        raise RuntimeError('V36 submission schema drift')
    ids = frame['StudyInstanceUID'].astype(str).tolist()
    if ids != list(map(str, expected_ids)) or len(ids) != len(set(ids)):
        raise RuntimeError('V36 submission study identity/order drift')
    values = frame[_RAD_LABELS].to_numpy(_rad_np.float64)
    if not _rad_np.isfinite(values).all() or values.min() < 0 or values.max() > 1:
        raise RuntimeError('V36 invalid submission values')


def _rad_main():
    started = _rad_time.time()
    work = _RadPath(_rad_os.environ.get('RSNA_RAD_OUTPUT_DIR', '/kaggle/working'))
    primary = work / 'submission.csv'
    if not primary.is_file():
        raise FileNotFoundError('V37 requires the completed DINO parent submission.csv')
    test = _rad_pd.read_csv(ROOT / 'test.csv', dtype={'StudyInstanceUID': str})
    expected_ids = test['StudyInstanceUID'].astype(str).tolist()
    baseline = _rad_pd.read_csv(primary, dtype={'StudyInstanceUID': str})
    _rad_validate(baseline, expected_ids)

    device = _rad_torch.device('cuda:0' if _rad_torch.cuda.is_available() else 'cpu')
    if device.type != 'cuda':
        raise RuntimeError('V37 RadImageNet inference requires CUDA')
    (encoder, public_heads, reference_heads, encoder_path,
     public_heads_path, reference_heads_path) = _rad_load_models(device)

    # Family 1: public v15/E10 legacy pixels.  Keep this path bit-for-bit as in
    # V36, including rank(mean(fold probability)).
    test_series = _rad_pd.read_csv(
        ROOT / 'test_series.csv',
        dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str},
    )
    plane = dict(zip(test_series.SeriesInstanceUID, test_series.Anatomical_Plane))
    headers = annotate(walk('test_series'))
    studies, pixels, slot_mask = build_cache(
        pick_slots(headers, plane), plane, lat_of(headers, 'test-e10 '), 'test-e10'
    )
    by_uid = {str(uid): index for index, uid in enumerate(studies)}
    missing = [uid for uid in expected_ids if uid not in by_uid]
    if missing:
        raise RuntimeError(f'{len(missing)} test studies absent from public-v15 cache')
    order = _rad_np.asarray([by_uid[uid] for uid in expected_ids], dtype=_rad_np.int64)
    pixels, slot_mask = pixels[order], slot_mask[order]
    token_count = int(
        _rad_np.repeat(slot_mask[:, :, None], CACHE_SLICES, axis=2).sum()
    )
    if token_count < int(0.85 * len(test) * N_SLOT * CACHE_SLICES):
        raise RuntimeError(f'insufficient acquired public-v15 test slices: {token_count}')

    features, token_mask = _rad_encode(encoder, pixels, slot_mask, device)
    del pixels, slot_mask, headers
    _rad_gc.collect()
    public_fold_predictions = [
        _rad_predict_head(head, features, token_mask, device)
        for head in public_heads
    ]
    reference_fold_predictions = [
        _rad_predict_head(head, features, token_mask, device)
        for head in reference_heads
    ]
    if len(public_fold_predictions) != 5 or len(reference_fold_predictions) != 5:
        raise RuntimeError('twin E10 inference did not use all ten heads')

    # Preserve each public recipe's rank(mean(fold probability)) estimator.
    public_probability = _rad_np.mean(_rad_np.stack(public_fold_predictions), axis=0)
    reference_probability = _rad_np.mean(
        _rad_np.stack(reference_fold_predictions), axis=0
    )
    public_rank = _rad_rank_columns(public_probability)
    reference_rank = _rad_rank_columns(reference_probability)
    del (public_heads, reference_heads, public_fold_predictions,
         reference_fold_predictions, public_probability, reference_probability,
         features, token_mask)
    _rad_gc.collect()
    _rad_torch.cuda.empty_cache()
    _rad_log(
        f'twin public-v15 families complete ({public_heads_path}; {reference_heads_path})'
    )

    # One new member from V48: three fat-sensitive planes plus a sagittal
    # structural anchor, all at a 130 mm crop. If E13 heads are available,
    # evaluate both passes; otherwise deploy the twin public-v15 RadImageNet ensemble.
    e13_heads, e13_path = None, None
    e13_token_count = 0
    v48_pass2_token_count = 0

    try:
        e13_heads, e13_path = _rad_load_e13_heads(device)
    except Exception as _e13_err:
        _rad_log(
            f'Optional E13/E11 heads absent or unreadable ({_e13_err}); '
            f'deploying twin public-v15 RadImageNet ensemble at alpha={_RAD_ALPHA:.2f}.'
        )
        e13_heads, e13_path = None, None

    if e13_heads is not None:
        globals().update(
            SLOTS=list(_RAD_E13_SLOTS),
            N_SLOT=len(_RAD_E13_SLOTS),
            CACHE_SLICES=int(_RAD_E13_CACHE_SLICES),
            IMG=int(_RAD_E13_IMG),
            CACHE_IMG=int(_RAD_E13_IMG),
            CROP_MM=float(_RAD_E13_CROP_MM),
            RULES=dict(RULES_LEGACY),
        )
        headers = annotate(walk('test_series'))
        studies, pixels, slot_mask = build_cache(
            pick_slots(headers, plane), plane, lat_of(headers, 'test-e13 '), 'test-e13'
        )
        by_uid = {str(uid): index for index, uid in enumerate(studies)}
        missing = [uid for uid in expected_ids if uid not in by_uid]
        if missing:
            raise RuntimeError(f'{len(missing)} test studies absent from E13 cache')
        order = _rad_np.asarray([by_uid[uid] for uid in expected_ids], dtype=_rad_np.int64)
        pixels, slot_mask = pixels[order], slot_mask[order]
        e13_token_count = int(
            _rad_np.repeat(slot_mask[:, :, None], CACHE_SLICES, axis=2).sum()
        )
        if e13_token_count < int(0.85 * len(test) * N_SLOT * CACHE_SLICES):
            raise RuntimeError(f'insufficient acquired E13 test slices: {e13_token_count}')
        e13_features, e13_token_mask = _rad_encode(
            encoder, pixels, slot_mask, device
        )
        del pixels, slot_mask, headers
        _rad_gc.collect()
        e13_predictions = [
            _rad_predict_head(head, e13_features, e13_token_mask, device)
            for head in e13_heads
        ]
        if len(e13_predictions) != 5:
            raise RuntimeError('E13 inference did not use all five heads')
        e13_probability = _rad_np.mean(_rad_np.stack(e13_predictions), axis=0)
        if (
            e13_probability.shape != (len(test), len(_RAD_LABELS))
            or not _rad_np.isfinite(e13_probability).all()
        ):
            raise RuntimeError(f'invalid E13 prediction shape/value: {e13_probability.shape}')
        e13_rank = _rad_rank_columns(e13_probability)
        public_rank = _rad_rank_columns(
            (1.0 - _RAD_E13_MEMBER_WEIGHT) * public_rank
            + _RAD_E13_MEMBER_WEIGHT * e13_rank
        )
        reference_rank = _rad_rank_columns(
            (1.0 - _RAD_E13_MEMBER_WEIGHT) * reference_rank
            + _RAD_E13_MEMBER_WEIGHT * e13_rank
        )
        del (e13_predictions, e13_probability, e13_rank,
             e13_features, e13_token_mask)
        _rad_gc.collect()
        _rad_torch.cuda.empty_cache()
        _rad_log(
            f'E13 FS-crop member complete at Rad-block weight '
            f'{_RAD_E13_MEMBER_WEIGHT:.2f} ({e13_path})'
        )

    # E10 keeps its audited 0.50 parent/Rad vote. The two excluded findings
    # remain the raw parent values, matching the audited deployment.
    baseline_rank = _rad_rank_columns(baseline[_RAD_LABELS].to_numpy())

    def _rad_e10_branch(head_rank):
        branch = baseline.copy()
        for index, target in enumerate(_RAD_LABELS):
            if target not in _RAD_EXCLUDE:
                branch[target] = (
                    (1.0 - _RAD_ALPHA) * baseline_rank[:, index]
                    + _RAD_ALPHA * head_rank[:, index]
                )
        return branch

    candidate_alt = _rad_e10_branch(public_rank)
    candidate_reference = _rad_e10_branch(reference_rank)
    for branch in (candidate_alt, candidate_reference):
        for target in _RAD_EXCLUDE:
            if not _rad_np.array_equal(
                branch[target].to_numpy(), baseline[target].to_numpy()
            ):
                raise RuntimeError(f'E10 failed to preserve raw parent values for {target}')
        _rad_validate(branch, expected_ids)

    candidate = baseline.copy()
    alt_e10_rank = _rad_rank_columns(candidate_alt[_RAD_LABELS].to_numpy())
    reference_e10_rank = _rad_rank_columns(
        candidate_reference[_RAD_LABELS].to_numpy()
    )
    candidate[_RAD_LABELS] = (
        _RAD_TWIN_ALT_WEIGHT * alt_e10_rank
        + (1.0 - _RAD_TWIN_ALT_WEIGHT) * reference_e10_rank
    )
    _rad_validate(candidate, expected_ids)
    e10_path = work / 'submission_e10_v2.csv'
    candidate.to_csv(e10_path, index=False)
    _rad_log(
        f'twin E10 branches complete at alpha={_RAD_ALPHA:.2f}; '
        f'preserved raw={list(_RAD_EXCLUDE)}'
    )

    if e13_heads is not None:
        globals().update(
            SLOTS=list(_RAD_E11_SLOTS),
            N_SLOT=len(_RAD_E11_SLOTS),
            CACHE_SLICES=int(_RAD_E11_CACHE_SLICES),
            IMG=int(_RAD_E11_IMG),
            CACHE_IMG=int(_RAD_E11_IMG),
            CROP_MM=float(_RAD_E11_CROP_MM),
            RULES=dict(RULES_LEGACY),
        )
        headers = annotate(walk('test_series'))
        studies, pixels, slot_mask = build_cache(
            pick_slots(headers, plane), plane,
            lat_of(headers, 'test-v48-pass2 '), 'test-v48-pass2'
        )
        by_uid = {str(uid): index for index, uid in enumerate(studies)}
        missing = [uid for uid in expected_ids if uid not in by_uid]
        if missing:
            raise RuntimeError(f'{len(missing)} test studies absent from V48 pass-2 cache')
        order = _rad_np.asarray([by_uid[uid] for uid in expected_ids], dtype=_rad_np.int64)
        pixels, slot_mask = pixels[order], slot_mask[order]
        v48_pass2_token_count = int(
            _rad_np.repeat(slot_mask[:, :, None], CACHE_SLICES, axis=2).sum()
        )
        if v48_pass2_token_count < int(0.55 * len(test) * N_SLOT * CACHE_SLICES):
            raise RuntimeError(
                f'insufficient acquired V48 pass-2 slices: {v48_pass2_token_count}'
            )
        v48_features, v48_token_mask = _rad_encode(
            encoder, pixels, slot_mask, device
        )
        del pixels, slot_mask, headers
        _rad_gc.collect()
        v48_pass2_predictions = [
            _rad_predict_head(head, v48_features, v48_token_mask, device)
            for head in e13_heads
        ]
        if len(v48_pass2_predictions) != 5:
            raise RuntimeError('V48 second pass did not use all five E13 heads')
        v48_pass2_probability = _rad_np.mean(
            _rad_np.stack(v48_pass2_predictions), axis=0
        )
        if (
            v48_pass2_probability.shape != (len(test), len(_RAD_LABELS))
            or not _rad_np.isfinite(v48_pass2_probability).all()
        ):
            raise RuntimeError(
                f'invalid V48 pass-2 prediction: {v48_pass2_probability.shape}'
            )
        v48_pass2_rank = _rad_rank_columns(v48_pass2_probability)

        def _rad_v48_second_pass(e10_branch):
            e10_rank = _rad_rank_columns(e10_branch[_RAD_LABELS].to_numpy())
            branch = e10_branch.copy()
            branch[_RAD_LABELS] = (
                (1.0 - _RAD_V48_SECOND_ALPHA) * e10_rank
                + _RAD_V48_SECOND_ALPHA * v48_pass2_rank
            )
            _rad_validate(branch, expected_ids)
            return branch

        alt_branch = _rad_v48_second_pass(candidate_alt)
        reference_branch = _rad_v48_second_pass(candidate_reference)
        _rad_log(
            f'V48 second E13 pass complete at alpha '
            f'{_RAD_V48_SECOND_ALPHA:.2f} on the E11 slot layout'
        )
        final = reference_branch.copy()
    else:
        final = candidate.copy()

    _rad_validate(final, expected_ids)
    temporary = primary.with_suffix('.csv.tmp')
    final.to_csv(temporary, index=False)
    _rad_os.replace(temporary, primary)

    receipt = {
        'recipe': (
            'V48 deployed reference branch: correct E13@0.50-inside-Rad -> E10@0.50 -> same E13 on E11 layout@0.15'
            if e13_heads is not None
            else 'Twin public-v15 RadImageNet ensemble at alpha=0.50'
        ),
        'e13_member_weight_inside_rad': _RAD_E13_MEMBER_WEIGHT if e13_heads is not None else 0.0,
        'e10_alpha': _RAD_ALPHA,
        'e10_preserved_targets': list(_RAD_EXCLUDE),
        'v48_second_alpha': _RAD_V48_SECOND_ALPHA if e13_heads is not None else 0.0,
        'reference_heads_sha256': _RAD_REFERENCE_HEADS_SHA256,
        'e13_heads_sha256': _RAD_E13_HEADS_SHA256 if e13_heads is not None else None,
        'v48_second_heads_sha256': _RAD_E13_HEADS_SHA256 if e13_heads is not None else None,
        'v48_second_slots': [list(slot) for slot in _RAD_E11_SLOTS] if e13_heads is not None else [],
        'encoder_sha256': _RAD_ENCODER_SHA256,
        'test_studies': len(expected_ids),
        'e10_tokens': token_count,
        'v48_second_tokens': v48_pass2_token_count,
        'e13_tokens': e13_token_count,
        'submission_sha256': _rad_sha256(primary),
    }
    (work / 'v50_v2_repro_receipt.json').write_text(
        _rad_json.dumps(receipt, indent=2, sort_keys=True) + '\n'
    )
    _rad_gc.collect()
    _rad_torch.cuda.empty_cache()
    _rad_log(
        f'V48 reference branch complete; reference-v15={reference_heads_path}; '
        f'e13-two-pass={e13_path}; second_alpha='
        f'{_RAD_V48_SECOND_ALPHA:.2f}; encoder={encoder_path}; '
        f'elapsed={(_rad_time.time()-started)/60:.1f}m'
    )


_rad_main()
